<a href="https://colab.research.google.com/github/tommbk50-hub/Cheminformatics/blob/main/AutoGCMSAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenMS, py_hplc and Gemini AI Agents: Automated HPLC-GCMS Chromatrogram Analysis of Complex Crude Reaction Mixtures.

OpenMS is an open-source, C++ framework hosted on Github <https://github.com/OpenMS/OpenMS> for analyzing large volumes of mass spectrometry data. It has been specially designed for analyzing high performance LC-MS data but over recent times, has been extended to analyze data generated by other techniques.

https://pyopenms.readthedocs.io/en/latest/user_guide/background.html


This Google Colab notebook is an end-to-end automated analytical chemistry pipeline that combines OpenMS (pyopenms), hplc-py, RDKit, and Gemini AI agents to process, deconvolute, assign, visualize, and visually quality-control LC-MS chromatograms of complex crude organic reaction mixtures. It operates primarily in a synthetic crude reaction mode (with a secondary reference mode for metabolomics nucleosides) to help medicinal and process chemists track reaction conversion, identify unreacted starting materials, and diagnose synthetic byproducts or impurities.

In [ ]:
#@title 1. Install required Python libraries with pinned versions
!uv pip install --system -q \
    "ipyketcher" \
    "rdkit>=2023.9.1" \
    "mols2grid==2.0.0" \
    "pyteomics==4.6.3" \
    "pyopenms>=3.1.0" \
    "psims==1.3.1" \
    "lxml>=5.1.0" \
    "hdf5plugin==4.3.0" \
    "pandas>=2.0.0,<3.0.0" \
    "pubchempy==1.0.4" \
    "mendeleev" \
    "marimo==0.2.5" \
    "wigglystuff==0.1.3" \
    "matchms" \
    "onnxruntime>=1.17.1" \
    "bottleneck==1.3.8" \
    "plotly>=5.18.0"

!pip -q install kaleido==0.1.0

import os
import glob
import pandas as pd
import matplotlib.colors as mcolors

# Helper to convert matplotlib colors to Plotly-compatible RGB strings
def to_plotly_color(color_val):
    """Converts any matplotlib-compatible color (including RGB tuples) to an rgb(r,g,b) string for Plotly."""
    rgba = mcolors.to_rgba(color_val)
    return f"rgb({int(rgba[0]*255)}, {int(rgba[1]*255)}, {int(rgba[2]*255)})"

# 1. Clone hplc-py for chromatographic deconvolution
if not os.path.exists('hplc-py'):
    !git clone -q https://github.com/cremerlab/hplc-py.git

# 2. Crude Reaction Mode

In [ ]:
#@title 2B. (Optional) Generate Realistic Crude Suzuki-Miyaura Reaction .mzML for Testing
import numpy as np
import pyopenms as oms
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

ELECTRON_MASS = 0.00054858

def generate_crude_suzuki_mzml_wide(filename="crude_suzuki_cell2b.mzML"):
    """Generates a 6-component crude Suzuki-Miyaura .mzML spanning 30–400 s (accurate [M+H]+ / [M+Na]+ m/z)."""
    components = [
        {"name": "4-Methoxyphenol (Boronic Oxidation Impurity)", "smiles": "COc1ccc(O)cc1", "rt": 95.0,  "width": 6.0, "intensity": 1.8e5},
        {"name": "4-Methoxybiphenyl carboxylic acid (Ester Hydrolysis Byproduct)", "smiles": "COc1ccc(-c2ccc(C(=O)O)cc2)cc1", "rt": 155.0, "width": 7.0, "intensity": 3.2e5},
        {"name": "Anisole (Protodeboronation Byproduct)", "smiles": "COc1ccccc1", "rt": 185.0, "width": 6.5, "intensity": 1.2e5},
        {"name": "Methyl 4-bromobenzoate (Unreacted Aryl Bromide SM)", "smiles": "COC(=O)c1ccc(Br)cc1", "rt": 230.0, "width": 7.5, "intensity": 4.5e5},
        {"name": "Target Biaryl Product", "smiles": "COc1ccc(-c2ccc(C(=O)OC)cc2)cc1", "rt": 285.0, "width": 8.0, "intensity": 2.4e6},
        {"name": "Dimethyl [1,1'-biphenyl]-4,4'-dicarboxylate (Aryl Halide Homocoupling Dimer)", "smiles": "COC(=O)c1ccc(-c2ccc(C(=O)OC)cc2)cc1", "rt": 340.0, "width": 8.5, "intensity": 2.9e5},
    ]

    exp = oms.MSExperiment()
    rt_grid = np.linspace(30.0, 400.0, 186)  # 1 scan every 2 seconds
    iso_gen = oms.CoarseIsotopePatternGenerator(5)
    rng = np.random.default_rng(42)

    for scan_idx, rt in enumerate(rt_grid):
        spec = oms.MSSpectrum()
        spec.setMSLevel(1)
        spec.setRT(float(rt))
        spec.setNativeID(f"scan={scan_idx+1}")

        mz_list, int_list = [], []
        for bg_mz in [149.0233, 279.0933, 391.2843]:
            mz_list.append(bg_mz + rng.normal(0, 0.0005))
            int_list.append(float(rng.uniform(4e3, 9e3)))

        for comp in components:
            elution_profile = np.exp(-0.5 * ((rt - comp["rt"]) / comp["width"]) ** 2)
            if elution_profile < 1e-3:
                continue
            mol = Chem.MolFromSmiles(comp["smiles"])
            formula_str = rdMolDescriptors.CalcMolFormula(mol)
            # Keep charge=0 in OpenMS after adding H/Na so OpenMS does NOT add a second proton; subtract 1 electron mass
            for adduct_formula, rel_adduct in [("H", 1.0), ("Na", 0.22)]:
                ef = oms.EmpiricalFormula(formula_str) + oms.EmpiricalFormula(adduct_formula)
                iso_dist = ef.getIsotopeDistribution(iso_gen)
                for iso_peak in iso_dist.getContainer():
                    mz_list.append(float(iso_peak.getMZ()) - ELECTRON_MASS + rng.normal(0, 0.0004))
                    int_list.append(float(comp["intensity"] * elution_profile * rel_adduct * iso_peak.getIntensity()))

        order = np.argsort(mz_list)
        spec.set_peaks((np.array(mz_list, dtype=np.float64)[order], np.array(int_list, dtype=np.float32)[order]))
        exp.addSpectrum(spec)

    oms.MzMLFile().store(filename, exp)
    print(f"✅ Saved Cell 2B crude reaction LC-MS dataset to '{filename}' ({len(rt_grid)} MS1 scans, 30–400 s).")
    print("   Tip: In Cell 2 below, select WORKFLOW_MODE = 'Use_Cell2B_or_Custom_mzML' to analyze this file without overwriting it.")

CELL_2B_MZML = "crude_suzuki_cell2b.mzML"
generate_crude_suzuki_mzml_wide(CELL_2B_MZML)

In [ ]:
#@title 2. Plot HPLC Chromatogram, Convert the HPLC Trace into a Pandas dataframe
import urllib.request
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyopenms as oms
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

WORKFLOW_MODE = "Synthetic_Suzuki_Reaction"  #@param ["Synthetic_Suzuki_Reaction", "Use_Cell2B_or_Custom_mzML", "Metabolomics_Nucleosides"]
CUSTOM_MZML_PATH = "crude_suzuki_cell2b.mzML"  #@param {type:"string"}

ELECTRON_MASS = 0.00054858

def generate_crude_suzuki_mzml(filename="crude_suzuki_reaction.mzML"):
    """Generates a realistic 7-component crude Suzuki-Miyaura reaction LC-MS .mzML file."""
    components = [
        {"name": "4-Methoxyphenol (Boronic Oxidation Impurity)", "smiles": "COc1ccc(O)cc1", "rt": 205.0, "width": 2.8, "intensity": 8.5e5},
        {"name": "(4-Methoxyphenyl)boronic acid (Unreacted SM2)", "smiles": "COc1ccc(B(O)O)cc1", "rt": 220.0, "width": 5.5, "intensity": 4.2e6},
        {"name": "4'-Methoxybiphenyl-4-carboxylic acid (Ester Hydrolysis Byproduct)", "smiles": "COc1ccc(-c2ccc(C(=O)O)cc2)cc1", "rt": 236.0, "width": 6.0, "intensity": 1.8e6},
        {"name": "Anisole (Protodeboronation Byproduct)", "smiles": "COc1ccccc1", "rt": 250.0, "width": 2.5, "intensity": 5.0e5},
        {"name": "Methyl 4-bromobenzoate (Unreacted Aryl Bromide SM1)", "smiles": "COC(=O)c1ccc(Br)cc1", "rt": 266.0, "width": 3.2, "intensity": 6.5e6},
        {"name": "Methyl 4'-methoxy-[1,1'-biphenyl]-4-carboxylate (Target Biaryl Product)", "smiles": "COc1ccc(-c2ccc(C(=O)OC)cc2)cc1", "rt": 282.0, "width": 4.5, "intensity": 2.8e7},
        {"name": "Dimethyl [1,1'-biphenyl]-4,4'-dicarboxylate (Homocoupling Dimer Byproduct)", "smiles": "COC(=O)c1ccc(-c2ccc(C(=O)OC)cc2)cc1", "rt": 295.0, "width": 3.8, "intensity": 1.2e6},
    ]

    exp = oms.MSExperiment()
    rt_grid = np.linspace(190.0, 310.0, 241)
    iso_gen = oms.CoarseIsotopePatternGenerator(5)
    rng = np.random.default_rng(42)

    for scan_idx, rt in enumerate(rt_grid):
        spec = oms.MSSpectrum()
        spec.setMSLevel(1)
        spec.setRT(float(rt))
        spec.setNativeID(f"scan={scan_idx+1}")

        mz_list, int_list = [], []
        for bg_mz in [149.0233, 279.0933]:
            mz_list.append(bg_mz + rng.normal(0, 0.0004))
            int_list.append(float(rng.uniform(1e4, 2.5e4)))

        for comp in components:
            elution_profile = np.exp(-0.5 * ((rt - comp["rt"]) / comp["width"]) ** 2)
            if elution_profile < 1e-3:
                continue
            mol = Chem.MolFromSmiles(comp["smiles"])
            formula_str = rdMolDescriptors.CalcMolFormula(mol)
            # Keep charge=0 in OpenMS so it does not double-protonate; subtract 1 electron mass for +1 ion
            for adduct_formula, rel_adduct in [("H", 1.0), ("Na", 0.18)]:
                ef = oms.EmpiricalFormula(formula_str) + oms.EmpiricalFormula(adduct_formula)
                iso_dist = ef.getIsotopeDistribution(iso_gen)
                for iso_peak in iso_dist.getContainer():
                    mz_list.append(float(iso_peak.getMZ()) - ELECTRON_MASS + rng.normal(0, 0.0003))
                    int_list.append(float(comp["intensity"] * elution_profile * rel_adduct * iso_peak.getIntensity()))

        order = np.argsort(mz_list)
        spec.set_peaks((np.array(mz_list, dtype=np.float64)[order], np.array(int_list, dtype=np.float32)[order]))
        exp.addSpectrum(spec)

    oms.MzMLFile().store(filename, exp)
    print(f"✅ Generated synthetic crude reaction dataset: {filename} ({len(rt_grid)} MS1 scans)")

if WORKFLOW_MODE == "Synthetic_Suzuki_Reaction":
    mzml_filename = "crude_suzuki_reaction.mzML"
    generate_crude_suzuki_mzml(mzml_filename)
elif WORKFLOW_MODE == "Use_Cell2B_or_Custom_mzML":
    mzml_filename = CUSTOM_MZML_PATH if os.path.exists(CUSTOM_MZML_PATH) else globals().get("CELL_2B_MZML", "crude_suzuki_cell2b.mzML")
    if not os.path.exists(mzml_filename):
        raise FileNotFoundError(f"'{mzml_filename}' not found. Run Cell 2B first or set CUSTOM_MZML_PATH to an uploaded .mzML file.")
else:
    url_new = "https://raw.githubusercontent.com/OpenMS/OpenMS/develop/src/tests/topp/FeatureFinderMetaboIdent_1_input.mzML"
    mzml_filename = "FeatureFinderMetaboIdent_1_input.mzML"
    if not os.path.exists(mzml_filename):
        print(f"Downloading {mzml_filename}...")
        urllib.request.urlretrieve(url_new, mzml_filename)

od = oms.MSExperiment()
oms.FileHandler().loadExperiment(mzml_filename, od)

rt_new, tic_new = [], []
for spec in od:
    if spec.getMSLevel() == 1:
        rt_new.append(float(spec.getRT()))
        _, ints = spec.get_peaks()
        tic_new.append(float(np.sum(ints)))

order_rt = np.argsort(rt_new)
hplc_df = pd.DataFrame({
    'Signal': np.array(tic_new, dtype=float)[order_rt],
    'Time': np.array(rt_new, dtype=float)[order_rt]
})
print(f"Active mzML file: {mzml_filename} | Scans: {len(hplc_df)} | RT span: {hplc_df['Time'].min():.1f}–{hplc_df['Time'].max():.1f} s")

plt.figure(figsize=(10, 5))
plt.plot(hplc_df['Time'], hplc_df['Signal'], color='green', linewidth=1.5)
plt.title(f'HPLC MS1 TIC Trace: {mzml_filename} ({WORKFLOW_MODE})')
plt.xlabel('Retention Time (seconds)')
plt.ylabel('Total Ion Current (Signal)')
plt.grid(True)
plt.tight_layout()
globals().setdefault('_notebook_figure_registry', {})['raw_hplc_chromatogram'] = plt.gcf()
plt.show()

# How Cell 3 Predicts Products & Byproducts

Cell 3 uses RDKit reaction templates (SMARTS rules) to automatically turn your starting materials into a searchable LC-MS target list.   

**Load Inputs (Reactant & Reagent)**

* You provide starting materials by picking one of the 20 MedChem reaction presets, typing a SMILES or chemical name (resolved via PubChem), or sketching structures in the Ketcher editor.   

* Only molecules tagged as Reactant are reacted; Reagent entries (like TPPO or DIPEA) are kept unchanged in the table so they can still be matched in the MS1 data.  

**Predict the Primary Product**

* (Product)RDKit (rdChemReactions) applies the selected reaction's forward SMARTS rule to your reactants—or scans the other 19 MedChem templates as a fallback if you drew custom molecules.   Valid, unique outputs are sanitized and added to the table as Product.   

**Enumerate Side Products**

* (Byproduct)RDKit runs both reaction-specific side reactions (e.g., protodeboronation, oxidation, or homocoupling dimers for Suzuki-Miyaura) and universal side reactions (ester hydrolysis, MeOH transesterification, and Boc loss) across both your reactants and predicted products.   Unique structures are deduplicated by canonical SMILES and tagged as Byproduct.   

**Calculate Masses & 2D Cards**

* For every molecule in the table, RDKit computes its molecular formula, monoisotopic exact mass ($M$), expected ESI+ ions ($[\text{M}+\text{H}]^+ = M + 1.007276$ and $[\text{M}+\text{Na}]^+ = M + 22.989218$), and renders a color-coded 2D SVG structure card for the gallery.

In [ ]:
#@title 3. Reaction Predictor & Byproduct Enumerator (Auto-Populated Ketcher + On-Demand RDKit Prediction & 2D Gallery)
import os
import sys
import html
import pathlib
import traitlets
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors, rdChemReactions
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Geometry import Point3D
import pubchempy as pcp

# 1. Dynamic patch for ipyketcher on Python 3.13
if not hasattr(traitlets.List, '__class_getitem__'):
    traitlets.List.__class_getitem__ = classmethod(lambda cls, item: cls)

ipk_init = f"/usr/local/lib/python{sys.version_info.major}.{sys.version_info.minor}/dist-packages/ipyketcher/__init__.py"
if os.path.exists(ipk_init):
    with open(ipk_init, 'r') as f:
        content = f.read()
    if "traitlets.List[str]()" in content:
        with open(ipk_init, 'w') as f:
            f.write(content.replace("traitlets.List[str]()", "traitlets.List()"))

if 'ipyketcher' in sys.modules:
    del sys.modules['ipyketcher']
import ipyketcher

# --- TOP 20 MEDICINAL CHEMISTRY REACTIONS (Reactants, Reagents & Catalysts ONLY) ---
# Primary Products & Byproducts are predicted ONLY when the Predict button is clicked.
MEDCHEM_TOP_20 = {
    "1. Suzuki-Miyaura Cross-Coupling": {
        "forward_smarts": "[c,C:1][Br,I,Cl].[c,C:2][B](O)O>>[c,C:1]-[c,C:2]",
        "inputs": [
            ("Methyl 4-bromobenzoate", "Reactant", "COC(=O)c1ccc(Br)cc1"),
            ("(4-Methoxyphenyl)boronic acid", "Reactant", "COc1ccc(B(O)O)cc1"),
            ("Potassium carbonate (Base)", "Reagent", "[K+].[K+].[O-]C(=O)[O-]"),
            ("Triphenylphosphine oxide (TPPO)", "Reagent", "O=P(c1ccccc1)(c1ccccc1)c1ccccc1"),
            ("Palladium(II) acetate (Pd(OAc)2)", "Catalyst", "CC(=O)O[Pd]OC(=O)C"),
        ],
        "class_byproducts": [
            ("Protodeboronation", "[c,C:1][B](O)O>>[c,C:1][H]", 1),
            ("Boronic Acid Oxidation (Phenol)", "[c:1][B](O)O>>[c:1]O", 1),
            ("Aryl Halide Protodehalogenation", "[c:1][Br,I,Cl]>>[c:1][H]", 1),
            ("Aryl Halide Homocoupling Dimer", "[c:1][Br,I,Cl].[c:2][Br,I,Cl]>>[c:1]-[c:2]", 2),
            ("Boronic Acid Homocoupling Dimer", "[c:1][B](O)O.[c:2][B](O)O>>[c:1]-[c:2]", 2),
        ],
    },
    "2. Amide Coupling (HATU / EDC)": {
        "forward_smarts": "[C:1](=[O:2])[OH].[N;H1,H2;+0:3]>>[C:1](=[O:2])[N:3]",
        "inputs": [
            ("4-Fluorobenzoic acid", "Reactant", "O=C(O)c1ccc(F)cc1"),
            ("4-Methoxybenzylamine", "Reactant", "COc1ccc(CN)cc1"),
            ("DIPEA (Hunig's Base)", "Reagent", "CCN(C(C)C)C(C)C"),
        ],
        "class_byproducts": [
            ("Symmetric Anhydride Intermediate", "[C:1](=[O:2])[OH].[C:3](=[O:4])[OH]>>[C:1](=[O:2])O[C:3](=[O:4])", 2),
            ("Methyl Ester (MeOH Solvolysis)", "[C:1](=[O:2])[OH]>>[C:1](=[O:2])OC", 1),
            ("Amine N-Formylation (+CO)", "[N;H1,H2;+0:1]>>[N:1]C=O", 1),
            ("Amine N-Acetylation Impurity", "[N;H1,H2;+0:1]>>[N:1]C(=O)C", 1),
        ],
    },
    "3. Buchwald-Hartwig Amination": {
        "forward_smarts": "[c:1][Br,I,Cl].[N;H1,H2;+0:2]>>[c:1]-[N:2]",
        "inputs": [
            ("2-Bromopyridine", "Reactant", "Brc1ccccn1"),
            ("Morpholine", "Reactant", "C1COCCN1"),
            ("XPhos Ligand", "Reagent", "CC(C)c1cc(C(C)C)c(-c2ccccc2P(C2CCCCC2)C2CCCCC2)c(C(C)C)c1"),
            ("Palladium(II) acetate (Pd(OAc)2)", "Catalyst", "CC(=O)O[Pd]OC(=O)C"),
        ],
        "class_byproducts": [
            ("Aryl Halide Protodehalogenation", "[c:1][Br,I,Cl]>>[c:1][H]", 1),
            ("Aryl Halide Hydroxylation (Phenol)", "[c:1][Br,I,Cl]>>[c:1]O", 1),
            ("Biaryl Homocoupling Dimer", "[c:1][Br,I,Cl].[c:2][Br,I,Cl]>>[c:1]-[c:2]", 2),
        ],
    },
    "4. Reductive Amination": {
        "forward_smarts": "[C,c:1][CH,C:2]=[O].[N;H1,H2;+0:3]>>[C,c:1][CH,C:2][N:3]",
        "inputs": [
            ("4-Chlorobenzaldehyde", "Reactant", "O=Cc1ccc(Cl)cc1"),
            ("Piperidine", "Reactant", "C1CCNCC1"),
        ],
        "class_byproducts": [
            ("Unreduced Imine Intermediate", "[C,c:1][CH:2]=[O].[N;H2;+0:3]>>[C,c:1][CH:2]=[N:3]", 2),
            ("Aldehyde Over-Reduction (Benzyl Alcohol)", "[C,c:1][CH:2]=[O]>>[C,c:1][CH2:2]O", 1),
            ("Cannizzaro Oxidation (Carboxylic Acid)", "[C,c:1][CH:2]=[O]>>[C,c:1]C(=O)O", 1),
            ("Over-Alkylation (Tertiary Bis-Benzyl Amine)", "[C,c:1][CH:2]=[O].[N;H2;+0:3]>>[N:3]([CH2:2][C,c:1])[CH2:2][C,c:1]", 2),
        ],
    },
    "5. SNAr (Nucleophilic Aromatic Substitution)": {
        "forward_smarts": "[c:1][F,Cl].[N,O,S;H1,H2;+0:2]>>[c:1]-[N,O,S:2]",
        "inputs": [
            ("1-Fluoro-4-nitrobenzene", "Reactant", "O=[N+]([O-])c1ccc(F)cc1"),
            ("1-Methylpiperazine", "Reactant", "CN1CCNCC1"),
        ],
        "class_byproducts": [
            ("Aryl Halide Hydrolysis (Phenol)", "[c:1][F,Cl]>>[c:1]O", 1),
            ("MeOH Solvolysis (Aryl Methyl Ether)", "[c:1][F,Cl]>>[c:1]OC", 1),
            ("Dimethylamino Substitution (DMF Impurity)", "[c:1][F,Cl]>>[c:1]N(C)C", 1),
        ],
    },
    "6. Bimolecular SN2 N-Alkylation": {
        "forward_smarts": "[N,O,S;H1,H2;+0:1].[C;!$(C=O):2][Br,Cl,I]>>[N,O,S:1]-[C:2]",
        "inputs": [
            ("4-Phenylpiperidine", "Reactant", "c1ccc(C2CCNCC2)cc1"),
            ("Benzyl bromide", "Reactant", "BrCc1ccccc1"),
            ("Triethylamine", "Reagent", "CCN(CC)CC"),
        ],
        "class_byproducts": [
            ("Over-Alkylation (Quaternary/Bis-Alkylated)", "[N;H2;+0:1].[C;!$(C=O):2][Br,Cl,I]>>[N:1]([C:2])[C:2]", 2),
            ("Alkyl Halide Hydrolysis (Alcohol)", "[C;!$(C=O):1][Br,Cl,I]>>[C:1]O", 1),
            ("Alkyl Halide MeOH Solvolysis (Methyl Ether)", "[C;!$(C=O):1][Br,Cl,I]>>[C:1]OC", 1),
        ],
    },
    "7. Boc Deprotection (TFA / HCl)": {
        "forward_smarts": "[N,O:1]C(=O)OC(C)(C)C>>[N,O:1][H]",
        "inputs": [
            ("tert-Butyl 4-(4-fluorophenyl)piperazine-1-carboxylate", "Reactant", "CC(C)(C)OC(=O)N1CCN(c2ccc(F)cc2)CC1"),
        ],
        "class_byproducts": [
            ("Carbamic Acid Intermediate (-Isobutene -56.06 Da)", "[N:1]C(=O)OC(C)(C)C>>[N:1]C(=O)O", 1),
            ("N-tert-Butylation Friedel-Crafts Recombination", "[N;H1:1]>>[N:1]C(C)(C)C", 1),
            ("TFA Trifluoroacetamide Adduct (+95.99 Da)", "[N;H1,H2:1]>>[N:1]C(=O)C(F)(F)F", 1),
        ],
    },
    "8. Ester Saponification (LiOH / NaOH)": {
        "forward_smarts": "[C:1](=[O:2])[O:3][C;!$(C=O):4]>>[C:1](=[O:2])[OH]",
        "inputs": [
            ("Ethyl 2-(4-chlorophenyl)acetate", "Reactant", "CCOC(=O)Cc1ccc(Cl)cc1"),
        ],
        "class_byproducts": [
            ("MeOH Transesterification (Methyl Ester)", "[C:1](=[O:2])OCC>>[C:1](=[O:2])OC", 1),
            ("Decarboxylation (-CO2)", "[c,C:1]C(=O)O>>[c,C:1][H]", 1),
        ],
    },
    "9. Sulfonamide Formation": {
        "forward_smarts": "[S:1](=[O:2])(=[O:3])[Cl].[N;H1,H2;+0:4]>>[S:1](=[O:2])(=[O:3])[N:4]",
        "inputs": [
            ("4-Methylbenzenesulfonyl chloride (TsCl)", "Reactant", "Cc1ccc(S(=O)(=O)Cl)cc1"),
            ("Aniline", "Reactant", "Nc1ccccc1"),
            ("DMAP", "Catalyst", "CN(C)c1ccncc1"),
        ],
        "class_byproducts": [
            ("Sulfonyl Chloride Hydrolysis (Sulfonic Acid)", "[S:1](=[O:2])(=[O:3])[Cl]>>[S:1](=[O:2])(=[O:3])O", 1),
            ("Bis-Sulfonylation (Sulfonimide Side Product)", "[S:1](=O)(=O)Cl.[N;H2:2]>>[N:2]([S:1](=O)(=O))[S:1](=O)(=O)", 2),
            ("Methyl Sulfonate Ester (MeOH Solvolysis)", "[S:1](=[O:2])(=[O:3])[Cl]>>[S:1](=[O:2])(=[O:3])OC", 1),
        ],
    },
    "10. Sonogashira Cross-Coupling": {
        "forward_smarts": "[c:1][Br,I].[C;H1:2]#[C:3]>>[c:1]-[C:2]#[C:3]",
        "inputs": [
            ("1-Iodo-4-methoxybenzene", "Reactant", "COc1ccc(I)cc1"),
            ("Phenylacetylene", "Reactant", "C#Cc1ccccc1"),
            ("Copper(I) iodide (CuI)", "Catalyst", "[Cu]I"),
        ],
        "class_byproducts": [
            ("Glaser Alkyne Homocoupling (1,3-Diyne Dimer)", "[C;H1:1]#[C:2].[C;H1:3]#[C:4]>>[C:2]#[C:1]-[C:3]#[C:4]", 2),
            ("Aryl Halide Protodehalogenation", "[c:1][Br,I]>>[c:1][H]", 1),
            ("Alkyne Hydration (Ketone Byproduct)", "[c:1]C#[C:2]>>[c:1]C(=O)[CH2:2]", 1),
        ],
    },
    "11. CuAAC Click Cycloaddition (1,4-Triazole)": {
        "forward_smarts": "[C,c:1][N-]=[N+]=[N].[C;H1:2]#[C:3][C,c:4]>>[C,c:1]n1cc([C,c:4])nn1",
        "inputs": [
            ("Benzyl azide", "Reactant", "[N-]=[N+]=NCc1ccccc1"),
            ("Phenylacetylene", "Reactant", "C#Cc1ccccc1"),
        ],
        "class_byproducts": [
            ("1,5-Disubstituted Triazole Regioisomer", "[C,c:1][N-]=[N+]=[N].[C;H1:2]#[C:3][C,c:4]>>[C,c:1]n1c([C,c:4])cnn1", 2),
            ("Azide Reduction to Primary Amine", "[C,c:1][N-]=[N+]=[N]>>[C,c:1]N", 1),
            ("Glaser Diacetylene Dimer", "[C;H1:1]#[C:2].[C;H1:3]#[C:4]>>[C:2]#[C:1]-[C:3]#[C:4]", 2),
        ],
    },
    "12. Urea Formation (Isocyanate + Amine)": {
        "forward_smarts": "[C,c:1][N:2]=[C:3]=[O:4].[N;H1,H2;+0:5]>>[C,c:1][NH:2][C:3](=[O:4])[N:5]",
        "inputs": [
            ("1-Chloro-4-isocyanatobenzene", "Reactant", "O=C=Nc1ccc(Cl)cc1"),
            ("Pyrrolidine", "Reactant", "C1CCNC1"),
        ],
        "class_byproducts": [
            ("Symmetrical Diaryl Urea (Isocyanate Hydrolysis + Coupling)", "[C,c:1]N=C=O.[C,c:2]N=C=O>>[C,c:1]NC(=O)N[C,c:2]", 2),
            ("Isocyanate Hydrolysis to Aniline", "[C,c:1]N=C=O>>[C,c:1]N", 1),
            ("Methyl Carbamate (MeOH Quench Product)", "[C,c:1]N=C=O>>[C,c:1]NC(=O)OC", 1),
        ],
    },
    "13. Carbamate Formation (Chloroformate + Amine)": {
        "forward_smarts": "[C,c:1][O:2][C:3](=[O:4])Cl.[N;H1,H2;+0:5]>>[C,c:1][O:2][C:3](=[O:4])[N:5]",
        "inputs": [
            ("Benzyl chloroformate (Cbz-Cl)", "Reactant", "O=C(Cl)OCc1ccccc1"),
            ("4-Methoxyaniline", "Reactant", "COc1ccc(N)cc1"),
        ],
        "class_byproducts": [
            ("Chloroformate Decarboxylation / Benzyl Chloride", "[C:1]OC(=O)Cl>>[C:1]Cl", 1),
            ("N-Benzylation Side Product", "[C:1]OC(=O)Cl.[N;H1,H2:2]>>[C:1][N:2]", 2),
            ("Symmetrical Urea Side Product", "[N;H2:1].[N;H2:2]>>[N:1]C(=O)[N:2]", 2),
        ],
    },
    "14. Mitsunobu Ether Synthesis": {
        "forward_smarts": "[c:1][OH].[C;!$(C=O);H2,H1:2][OH]>>[c:1]O[C:2]",
        "inputs": [
            ("4-Cyanophenol", "Reactant", "N#Cc1ccc(O)cc1"),
            ("(Tetrahydrofuran-2-yl)methanol", "Reactant", "OCC1CCCO1"),
            ("Triphenylphosphine oxide (TPPO)", "Reagent", "O=P(c1ccccc1)(c1ccccc1)c1ccccc1"),
            ("Diisopropyl hydrazinedicarboxylate (DIAD-H2)", "Reagent", "CC(C)OC(=O)NNC(=O)OC(C)C"),
        ],
        "class_byproducts": [
            ("C-Alkylation Ortho-Phenol Regioisomer", "[c:1]([OH])[cH:2].[C;!$(C=O):3][OH]>>[c:1]([OH])[c:2][C:3]", 2),
            ("N-Alkylated Hydrazine Adduct (DIAD Byproduct)", "[C;!$(C=O):1][OH]>>CC(C)OC(=O)N([C:1])NC(=O)OC(C)C", 1),
        ],
    },
    "15. Williamson Ether Synthesis": {
        "forward_smarts": "[c,C:1][OH].[C;!$(C=O):2][Br,I,Cl]>>[c,C:1]O[C:2]",
        "inputs": [
            ("4-Hydroxybenzaldehyde", "Reactant", "O=Cc1ccc(O)cc1"),
            ("1-Bromopropane", "Reactant", "CCCBr"),
        ],
        "class_byproducts": [
            ("Aldehyde Oxidation to Carboxylic Acid", "[c:1][CH]=O>>[c:1]C(=O)O", 1),
            ("Alkyl Bromide Hydrolysis (1-Propanol)", "[C:1][Br,I,Cl]>>[C:1]O", 1),
        ],
    },
    "16. Mizoroki-Heck Reaction": {
        "forward_smarts": "[c:1][Br,I].[CH2:2]=[CH:3][C,c:4]>>[c:1][CH:2]=[CH:3][C,c:4]",
        "inputs": [
            ("1-Bromo-4-fluorobenzene", "Reactant", "Fc1ccc(Br)cc1"),
            ("Methyl acrylate", "Reactant", "COC(=O)C=C"),
            ("Palladium(II) acetate (Pd(OAc)2)", "Catalyst", "CC(=O)O[Pd]OC(=O)C"),
        ],
        "class_byproducts": [
            ("1,1-Disubstituted Alpha-Regioisomer", "[c:1][Br,I].[CH2:2]=[CH:3][C,c:4]>>[CH2:2]=[C:3]([c:1])[C,c:4]", 2),
            ("Over-Reduction (Saturated Propanoate)", "[c:1][CH:2]=[CH:3][C:4]>>[c:1][CH2:2][CH2:3][C:4]", 1),
            ("Aryl Halide Protodehalogenation", "[c:1][Br,I]>>[c:1][H]", 1),
        ],
    },
    "17. Alkene / Nitro Hydrogenation (Pd/C)": {
        "forward_smarts": "[c,C:1][N+](=O)[O-]>>[c,C:1]N",
        "inputs": [
            ("Methyl 4-chloro-3-nitrobenzoate", "Reactant", "COC(=O)c1ccc(Cl)c([N+](=O)[O-])c1"),
            ("Palladium catalyst (Pd)", "Catalyst", "[Pd]"),
        ],
        "class_byproducts": [
            ("Hydrodechlorination (Des-Chloro Aniline)", "[c:1]([Cl])[c:2]N>>[cH:1][c:2]N", 1),
            ("Nitroso Intermediate", "[c:1][N+](=O)[O-]>>[c:1]N=O", 1),
            ("Hydroxylamine Intermediate", "[c:1][N+](=O)[O-]>>[c:1]NO", 1),
            ("Azo Dimer Byproduct", "[c:1][N+](=O)[O-].[c:2][N+](=O)[O-]>>[c:1]N=N[c:2]", 2),
        ],
    },
    "18. Wittig / HWE Olefination": {
        "forward_smarts": "[C,c:1][CH:2]=O.[C:3](=O)([O:4][C:5])CP(=O)(OC)OC>>[C,c:1][CH:2]=[CH][C:3](=O)[O:4][C:5]",
        "inputs": [
            ("4-Methoxybenzaldehyde", "Reactant", "COc1ccc(C=O)cc1"),
            ("Trimethyl phosphonoacetate", "Reactant", "COC(=O)CP(=O)(OC)OC"),
        ],
        "class_byproducts": [
            ("Beta-Hydroxy Ester (Uneliminated Aldol Intermediate)", "[C,c:1][CH:2]=O>>[C,c:1][CH:2](O)CC(=O)OC", 1),
            ("Aldehyde Oxidation to Benzoic Acid", "[C,c:1][CH]=O>>[C,c:1]C(=O)O", 1),
        ],
    },
    "19. Friedel-Crafts / Electrophilic Halogenation (NBS)": {
        "forward_smarts": "[c:1]([O,N:2])[cH:3]>>[c:1]([O,N:2])[c:3]Br",
        "inputs": [
            ("1-Methoxynaphthalene", "Reactant", "COc1cccc2ccccc12"),
            ("Succinimide (from NBS)", "Reagent", "O=C1CCC(=O)N1"),
        ],
        "class_byproducts": [
            ("Over-Bromination (Dibromo Byproduct)", "[c:1](Br)[cH:2]>>[c:1](Br)[c:2]Br", 1),
        ],
    },
    "20. Thioether Oxidation to Sulfone (mCPBA)": {
        "forward_smarts": "[C,c:1][S;X2:2][C,c:3]>>[C,c:1][S:2](=O)(=O)[C,c:3]",
        "inputs": [
            ("4-(Methylthio)benzonitrile", "Reactant", "CSc1ccc(C#N)cc1"),
            ("3-Chlorobenzoic acid (from mCPBA)", "Reagent", "O=C(O)c1cccc(Cl)c1"),
        ],
        "class_byproducts": [
            ("Incomplete Oxidation (Sulfoxide Intermediate)", "[C,c:1][S;X2:2][C,c:3]>>[C,c:1][S:2](=O)[C,c:3]", 1),
            ("Nitrile Hydrolysis to Primary Amide", "[c,C:1]C#N>>[c,C:1]C(=O)N", 1),
        ],
    },
    "21. Metabolomics: Nucleoside Mixture (Original Reference)": {
        "forward_smarts": "",
        "inputs": [
            ("adenosine", "Product", "Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O"),
            ("n-deoxyadenosine", "Product", "Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)C1"),
            ("n-inosine", "Product", "O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O"),
            ("n2-o-methylcytidine", "Product", "CO[C@@H]1[C@H](O[C@H](CO)[C@@H]1O)n2ccc(N)nc2=O"),
            ("n5-methylcytidine", "Product", "Cc1cn([C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)c(=O)nc1N"),
            ("n5-formylcytidine", "Product", "Nc1nc(=O)n([C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)cc1C=O"),
        ],
        "class_byproducts": [],
    },
}

UNIVERSAL_SIDE_REACTIONS = [
    ("Ester Hydrolysis (Carboxylic Acid Impurity)", "[C:1](=[O:2])[O:3][CH3,CH2:4]>>[C:1](=[O:2])[OH]", 1),
    ("MeOH Mobile-Phase Transesterification", "[C:1](=[O:2])[O:3][CH2:4]>>[C:1](=[O:2])OC", 1),
    ("Boc Loss (-100.05 Da)", "[N,O:1]C(=O)OC(C)(C)C>>[N,O:1][H]", 1),
]

ROLE_OPTIONS = ['Reactant', 'Reagent', 'Catalyst', 'Product', 'Byproduct']

# --- 2. Ketcher 2D Layout & Trait-Adaptive Population Helpers ---
def build_spaced_2d_molblock(smiles_list) -> str:
    """Combines multiple SMILES into a single non-overlapping 2D V2000 MolBlock for Ketcher."""
    placed_mols = []
    x_cursor = 0.0
    gap = 3.6

    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        m = Chem.Mol(mol)
        AllChem.Compute2DCoords(m)
        conf = m.GetConformer()
        xs = [conf.GetAtomPosition(i).x for i in range(m.GetNumAtoms())]
        ys = [conf.GetAtomPosition(i).y for i in range(m.GetNumAtoms())]
        if not xs:
            continue
        min_x, max_x = min(xs), max(xs)
        mid_y = 0.5 * (min(ys) + max(ys))
        shift_x = x_cursor - min_x
        for i in range(m.GetNumAtoms()):
            pos = conf.GetAtomPosition(i)
            conf.SetAtomPosition(i, Point3D(pos.x + shift_x, pos.y - mid_y, 0.0))
        x_cursor += (max_x - min_x) + gap
        placed_mols.append(m)

    if not placed_mols:
        return ""
    combined = placed_mols[0]
    for m in placed_mols[1:]:
        combined = Chem.CombineMols(combined, m)
    return Chem.MolToMolBlock(combined)

def _detect_ketcher_input_trait():
    """Inspects ipyketcher.KetcherEditor traits and ESM source to pick the exact synced structure trait."""
    cls_traits = set(ipyketcher.KetcherEditor.class_traits().keys())
    esm_text = ""
    raw_esm = getattr(ipyketcher.KetcherEditor, "_esm", "")
    if isinstance(raw_esm, pathlib.Path) and raw_esm.exists():
        esm_text = raw_esm.read_text(errors="ignore")
    elif isinstance(raw_esm, str):
        esm_text = raw_esm

    # Prefer a trait explicitly watched by the frontend JS
    for t_name, fmt in [("molfile", "molblock"), ("molecule", "molblock"), ("molblock", "molblock"), ("structure", "molblock"), ("smiles", "smiles"), ("value", "molblock")]:
        if t_name in cls_traits and (f"change:{t_name}" in esm_text or f'"{t_name}"' in esm_text or f"'{t_name}'" in esm_text):
            return t_name, fmt
    for t_name, fmt in [("molfile", "molblock"), ("molecule", "molblock"), ("smiles", "smiles"), ("value", "smiles")]:
        if t_name in cls_traits:
            return t_name, fmt
    return None, "smiles"

KETCHER_TRAIT, KETCHER_FMT = _detect_ketcher_input_trait()

def _format_entries_for_ketcher(entries) -> str:
    smiles_list = [smi.strip() for _, _, smi in entries if smi and Chem.MolFromSmiles(smi.strip())]
    if not smiles_list:
        return ""
    if KETCHER_FMT == "molblock":
        return build_spaced_2d_molblock(smiles_list)
    return ".".join(smiles_list)

def populate_ketcher_canvas(editor_obj, entries):
    """Pushes the 2D structures of reactants, reagents, and catalyst onto the Ketcher notepad."""
    payload = _format_entries_for_ketcher(entries)
    if KETCHER_TRAIT and hasattr(editor_obj, KETCHER_TRAIT):
        try:
            setattr(editor_obj, KETCHER_TRAIT, payload)
        except Exception:
            pass
    # Fallback: if trait expects SMILES only, ensure smiles is also synced when distinct
    if KETCHER_TRAIT != "smiles" and hasattr(editor_obj, "smiles") and not payload:
        try:
            editor_obj.smiles = ""
        except Exception:
            pass

def read_drawn_smiles_from_ketcher(editor_obj) -> str:
    """Reads the current drawing from Ketcher via smiles or molfile/molecule fallback."""
    smi_val = getattr(editor_obj, "smiles", None)
    if isinstance(smi_val, str) and smi_val.strip():
        return smi_val.strip()
    for t_name in ("molfile", "molecule", "molblock", "structure"):
        raw = getattr(editor_obj, t_name, None)
        if isinstance(raw, str) and "M  END" in raw:
            m = Chem.MolFromMolBlock(raw)
            if m is not None:
                return Chem.MolToSmiles(m, isomericSmiles=True)
    return ""

# --- 3. Initialize Ketcher Pre-Populated with Suzuki-Miyaura Reactants, Reagents & Catalyst ---
DEFAULT_PRESET = "1. Suzuki-Miyaura Cross-Coupling"
initial_suzuki_inputs = list(MEDCHEM_TOP_20[DEFAULT_PRESET]["inputs"])
init_payload = _format_entries_for_ketcher(initial_suzuki_inputs)

init_kwargs = {KETCHER_TRAIT: init_payload} if (KETCHER_TRAIT and init_payload) else {}
try:
    editor = ipyketcher.KetcherEditor(**init_kwargs)
except Exception:
    editor = ipyketcher.KetcherEditor()
    populate_ketcher_canvas(editor, initial_suzuki_inputs)

print("🎨 Ketcher notepad auto-populated with Suzuki-Miyaura reactants, reagents & catalyst. Edit or select a reaction, then click '⚡ Predict Primary Product & Side Products':")
display(editor)

# --- 4. RDKit Mass, Name Lookup & 2D Card Helpers ---
def resolve_name_or_smiles(text_val: str):
    s = text_val.strip()
    if not s:
        return None, None
    mol = Chem.MolFromSmiles(s)
    if mol is not None:
        return Chem.MolToSmiles(mol, isomericSmiles=True), s
    try:
        hits = pcp.get_compounds(s, 'name')
        if hits:
            smi = getattr(hits[0], 'isomeric_smiles', None) or getattr(hits[0], 'canonical_smiles', None)
            if smi and Chem.MolFromSmiles(smi):
                return Chem.MolToSmiles(Chem.MolFromSmiles(smi), isomericSmiles=True), s
    except Exception:
        pass
    return None, None

def calculate_exact_mass(smiles_str):
    mol = Chem.MolFromSmiles(smiles_str)
    if mol:
        return f"{Descriptors.ExactMolWt(mol):.4f}"
    return "Invalid"

def mol_to_svg_card(name: str, role: str, smiles: str) -> str:
    """Renders an RDKit 2D SVG card with role badge (Reactant/Reagent/Catalyst/Product/Byproduct) and exact masses."""
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        return ""
    mol_copy = Chem.Mol(mol)
    AllChem.Compute2DCoords(mol_copy)
    drawer = rdMolDraw2D.MolDraw2DSVG(220, 155)
    drawer.drawOptions().addStereoAnnotation = True
    drawer.drawOptions().padding = 0.10
    drawer.DrawMolecule(mol_copy)
    drawer.FinishDrawing()
    svg = drawer.GetDrawingText().replace("<?xml version='1.0' encoding='iso-8859-1'?>", "")

    em = Descriptors.ExactMolWt(mol)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    mh = em + 1.007276
    mna = em + 22.989218

    badge_colors = {
        "Reactant":  ("#eff6ff", "#1d4ed8", "#bfdbfe"),
        "Reagent":   ("#fefce8", "#a16207", "#fde047"),
        "Catalyst":  ("#f5f3ff", "#6d28d9", "#ddd6fe"),
        "Product":   ("#f0fdf4", "#15803d", "#86efac"),
        "Byproduct": ("#fff1f2", "#be123c", "#fecdd3"),
    }
    bg, fg, border = badge_colors.get(role, ("#f3f4f6", "#374151", "#d1d5db"))

    return (
        f"<div style='width:235px;border:1.5px solid {border};border-radius:10px;background:#ffffff;"
        f"padding:8px 10px;box-shadow:0 2px 6px rgba(0,0,0,0.12);display:flex;flex-direction:column;justify-content:space-between;'>"
        f"<div>"
        f"<span style='background:{bg};color:{fg};border:1px solid {border};border-radius:5px;padding:2px 7px;"
        f"font-size:11px;font-weight:700;'>{html.escape(role)}</span>"
        f"<div style='font-weight:700;font-size:12.5px;color:#111827;margin-top:5px;line-height:1.25;'>{html.escape(name)}</div>"
        f"</div>"
        f"<div style='text-align:center;margin:4px 0;'>{svg}</div>"
        f"<div style='font-size:11px;color:#374151;background:#f9fafb;padding:5px 7px;border-radius:6px;border:1px solid #e5e7eb;'>"
        f"<b>Formula:</b> {formula}<br>"
        f"<b>Exact Mass:</b> {em:.4f} Da<br>"
        f"<b>[M+H]+:</b> {mh:.4f} &nbsp;|&nbsp; <b>[M+Na]+:</b> {mna:.4f}"
        f"</div></div>"
    )

def create_table_row(comp_no, name, classification, smiles):
    role_clean = classification if classification in ROLE_OPTIONS else 'Reactant'
    w_no = widgets.HTML(value=f"{comp_no}", layout=widgets.Layout(width='38px', padding='0 4px'))
    w_name = widgets.Text(value=name, placeholder='Name', layout=widgets.Layout(width='250px', padding='0 4px'))
    w_class = widgets.Dropdown(options=ROLE_OPTIONS, value=role_clean, layout=widgets.Layout(width='115px', padding='0 4px'))
    w_smiles = widgets.Text(value=smiles, placeholder='SMILES', layout=widgets.Layout(width='310px', padding='0 4px'))
    w_mass = widgets.HTML(value=calculate_exact_mass(smiles), layout=widgets.Layout(width='85px', padding='0 4px'))

    def _on_smi_change(change):
        w_mass.value = calculate_exact_mass(change['new'])
    w_smiles.observe(_on_smi_change, names='value')
    return widgets.HBox([w_no, w_name, w_class, w_smiles, w_mass])

header_row = widgets.HBox([
    widgets.HTML(value="<b>No.</b>", layout=widgets.Layout(width='38px', padding='0 4px')),
    widgets.HTML(value="<b>Name</b>", layout=widgets.Layout(width='250px', padding='0 4px')),
    widgets.HTML(value="<b>Role</b>", layout=widgets.Layout(width='115px', padding='0 4px')),
    widgets.HTML(value="<b>SMILES String</b>", layout=widgets.Layout(width='310px', padding='0 4px')),
    widgets.HTML(value="<b>Exact Mass</b>", layout=widgets.Layout(width='85px', padding='0 4px')),
])

# --- 5. Interactive Controls ---
preset_dropdown = widgets.Dropdown(
    options=list(MEDCHEM_TOP_20.keys()),
    value=DEFAULT_PRESET,
    description="MedChem Reaction:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)
btn_predict = widgets.Button(
    description="⚡ Predict Primary Product & Side Products (rdChemReactions)",
    button_style="danger",
    layout=widgets.Layout(width='420px', height='34px')
)
btn_sync_ketcher = widgets.Button(
    description="Draw Table in Ketcher",
    button_style="info",
    tooltip="Re-populate the Ketcher canvas with the current Reactants, Reagents & Catalyst",
    layout=widgets.Layout(width='170px', height='34px')
)

quick_input_text = widgets.Text(
    value="",
    placeholder="Type chemical name (e.g. '4-bromoanisole') or SMILES...",
    description="Add Input:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='360px')
)
quick_input_role = widgets.Dropdown(options=['Reactant', 'Reagent', 'Catalyst'], value='Reactant', layout=widgets.Layout(width='110px'))
btn_add_typed = widgets.Button(description="Add by Name/SMILES", button_style="primary", layout=widgets.Layout(width='155px'))
btn_add_drawn = widgets.Button(description="Add Drawn from Ketcher", button_style="success", layout=widgets.Layout(width='175px'))
btn_clear_inputs = widgets.Button(description="Clear Table", button_style="warning", layout=widgets.Layout(width='115px'))

table_container = widgets.VBox([header_row])
gallery_output = widgets.Output()
status_output = widgets.Output()

def clear_structure_gallery():
    with gallery_output:
        clear_output(wait=True)

def render_structure_gallery():
    """Renders 2D SVG cards for Reactants, Reagents, Catalyst, Predicted Primary Product & Byproducts."""
    with gallery_output:
        clear_output(wait=True)
        cards = []
        for row in table_container.children[1:]:
            name = row.children[1].value.strip()
            role = row.children[2].value
            smi = row.children[3].value.strip()
            if smi:
                card_html = mol_to_svg_card(name, role, smi)
                if card_html:
                    cards.append(card_html)
        if cards:
            display(HTML(
                "<div style='margin-top:12px;margin-bottom:6px;font-weight:700;font-size:14px;color:#38bdf8;'>"
                "🧬 RDKit 2D Structure Gallery (Reactants, Reagents, Catalyst, Predicted Primary Product & Byproducts):</div>"
                "<div style='display:flex;flex-wrap:wrap;gap:12px;align-items:stretch;'>" + "".join(cards) + "</div>"
            ))

# --- 6. RDKit Forward Reaction & Side-Product Prediction Engine ---
def predict_products_and_byproducts(base_rows, reaction_key):
    """
    Takes ONLY Reactants, Reagents & Catalysts; runs forward SMARTS to predict Primary Target Product(s)
    and runs reaction-class + universal side-reaction SMARTS to enumerate plausible Byproducts.
    """
    scheme = MEDCHEM_TOP_20.get(reaction_key, {})
    seen_smiles = set()
    reactants = []
    all_inputs = []

    for name, role, smi in base_rows:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            can = Chem.MolToSmiles(mol, isomericSmiles=True)
            seen_smiles.add(can)
            all_inputs.append((name, role, mol, can))
            if role == "Reactant":
                reactants.append((name, mol, can))

    predicted_products = []
    predicted_byproducts = []

    candidate_forward_templates = []
    if scheme.get("forward_smarts"):
        candidate_forward_templates.append((reaction_key.split(". ", 1)[-1], scheme["forward_smarts"]))
    for k, v in MEDCHEM_TOP_20.items():
        if v.get("forward_smarts") and k != reaction_key:
            candidate_forward_templates.append((k.split(". ", 1)[-1], v["forward_smarts"]))

    for rxn_label, fwd_smarts in candidate_forward_templates:
        rxn = rdChemReactions.ReactionFromSmarts(fwd_smarts)
        n_react = rxn.GetNumReactantTemplates()
        hits_for_template = []
        if n_react == 1:
            for r_name, r_mol, _ in reactants:
                for prod_tuple in rxn.RunReactants((r_mol,)):
                    try:
                        p_mol = prod_tuple[0]
                        Chem.SanitizeMol(p_mol)
                        p_smi = Chem.MolToSmiles(p_mol, isomericSmiles=True)
                        if p_smi not in seen_smiles:
                            seen_smiles.add(p_smi)
                            hits_for_template.append((f"Predicted Product ({rxn_label})", "Product", p_mol, p_smi))
                    except Exception:
                        continue
        elif n_react == 2 and len(reactants) >= 2:
            for i in range(len(reactants)):
                for j in range(len(reactants)):
                    if i == j:
                        continue
                    for prod_tuple in rxn.RunReactants((reactants[i][1], reactants[j][1])):
                        try:
                            p_mol = prod_tuple[0]
                            Chem.SanitizeMol(p_mol)
                            p_smi = Chem.MolToSmiles(p_mol, isomericSmiles=True)
                            if p_smi not in seen_smiles:
                                seen_smiles.add(p_smi)
                                hits_for_template.append((f"Predicted Product ({rxn_label})", "Product", p_mol, p_smi))
                        except Exception:
                            continue
        if hits_for_template:
            predicted_products.extend(hits_for_template)
            break

    pool_for_side_rxns = [(n, m) for n, m, _ in reactants] + [(n, m) for n, _, m, _ in predicted_products]
    side_templates = list(scheme.get("class_byproducts", [])) + UNIVERSAL_SIDE_REACTIONS

    for bp_label, smarts, arity in side_templates:
        rxn = rdChemReactions.ReactionFromSmarts(smarts)
        if arity == 1:
            for src_name, src_mol in pool_for_side_rxns:
                for prod_tuple in rxn.RunReactants((src_mol,)):
                    try:
                        b_mol = prod_tuple[0]
                        Chem.SanitizeMol(b_mol)
                        b_smi = Chem.MolToSmiles(b_mol, isomericSmiles=True)
                        if b_smi not in seen_smiles:
                            seen_smiles.add(b_smi)
                            predicted_byproducts.append((f"{bp_label} ({src_name})", "Byproduct", b_smi))
                    except Exception:
                        continue
        elif arity == 2:
            for i, (n1, m1) in enumerate(pool_for_side_rxns):
                for (n2, m2) in pool_for_side_rxns[i:]:
                    for pair in [(m1, m2), (m2, m1)]:
                        for prod_tuple in rxn.RunReactants(pair):
                            try:
                                b_mol = prod_tuple[0]
                                Chem.SanitizeMol(b_mol)
                                b_smi = Chem.MolToSmiles(b_mol, isomericSmiles=True)
                                if b_smi not in seen_smiles:
                                    seen_smiles.add(b_smi)
                                    predicted_byproducts.append((f"{bp_label}", "Byproduct", b_smi))
                            except Exception:
                                continue

    final_rows = (
        [(n, r, s) for n, r, _, s in all_inputs]
        + [(n, r, s) for n, r, _, s in predicted_products]
        + predicted_byproducts
    )
    return final_rows, len(predicted_products), len(predicted_byproducts)

# --- 7. Event Handlers (NO auto-prediction & NO 2D cards until Predict button is clicked) ---
def load_preset_inputs_only(preset_key, update_ketcher=True):
    """Loads ONLY Reactants, Reagents & Catalyst into the table and Ketcher. Does NOT run prediction or draw 2D cards."""
    base_inputs = list(MEDCHEM_TOP_20[preset_key]["inputs"])
    table_container.children = [header_row] + [
        create_table_row(i + 1, n, r, s) for i, (n, r, s) in enumerate(base_inputs)
    ]
    clear_structure_gallery()
    if update_ketcher:
        populate_ketcher_canvas(editor, base_inputs)

    with status_output:
        clear_output(wait=True)
        print(
            f"✅ Loaded '{preset_key}' ({len(base_inputs)} input component(s) populated in Ketcher & table). "
            "Click '⚡ Predict Primary Product & Side Products (rdChemReactions)' to run RDKit prediction and generate 2D structure cards."
        )

def on_preset_change(change):
    load_preset_inputs_only(change["new"], update_ketcher=True)

def on_predict_clicked(b):
    with status_output:
        clear_output(wait=True)
        # Preset 21 (Nucleosides) has pre-defined reference analytes rather than forward SMARTS
        if preset_dropdown.value.startswith("21."):
            render_structure_gallery()
            print(f"✅ Rendered 2D structure cards for {len(table_container.children) - 1} reference nucleoside(s).")
            return

        # Preserve only Reactant, Reagent, and Catalyst rows so predictions stay clean on re-click
        user_inputs = [
            (r.children[1].value.strip(), r.children[2].value, r.children[3].value.strip())
            for r in table_container.children[1:]
            if r.children[2].value in ("Reactant", "Reagent", "Catalyst") and r.children[3].value.strip()
        ]
        if not user_inputs:
            print("⚠️ Please add at least one Reactant before running the predictor.")
            return

        final_rows, n_prod, n_bp = predict_products_and_byproducts(user_inputs, preset_dropdown.value)
        table_container.children = [header_row] + [
            create_table_row(i + 1, n, r, s) for i, (n, r, s) in enumerate(final_rows)
        ]
        render_structure_gallery()
        print(
            f"⚡ RDKit rdChemReactions predicted {n_prod} Primary Product(s) and {n_bp} Byproduct(s) "
            f"from {len(user_inputs)} input(s) — generated {len(final_rows)} 2D structure card(s) below!"
        )

def on_sync_ketcher_clicked(b):
    current_inputs = [
        (r.children[1].value.strip(), r.children[2].value, r.children[3].value.strip())
        for r in table_container.children[1:]
        if r.children[2].value in ("Reactant", "Reagent", "Catalyst") and r.children[3].value.strip()
    ]
    populate_ketcher_canvas(editor, current_inputs)
    with status_output:
        clear_output(wait=True)
        print(f"🎨 Synced {len(current_inputs)} table component(s) into Ketcher notepad.")

def on_add_typed_clicked(b):
    with status_output:
        clear_output(wait=True)
        raw = quick_input_text.value.strip()
        smi, resolved_name = resolve_name_or_smiles(raw)
        if not smi:
            print(f"❌ Could not parse '{raw}' as a SMILES or resolve via PubChem.")
            return
        idx = len(table_container.children)
        new_row = create_table_row(idx, resolved_name, quick_input_role.value, smi)
        table_container.children = list(table_container.children) + [new_row]
        quick_input_text.value = ""
        clear_structure_gallery()
        # Keep Ketcher canvas updated with all current inputs
        on_sync_ketcher_clicked(None)
        print(f"✅ Added {quick_input_role.value}: {resolved_name} ({smi}). Click Predict to enumerate & render 2D cards.")

def on_add_drawn_clicked(b):
    with status_output:
        clear_output(wait=True)
        drawn = read_drawn_smiles_from_ketcher(editor)
        if not drawn:
            print("⚠️ No structure detected in Ketcher. Draw a molecule or reaction scheme first.")
            return
        rows = list(table_container.children)
        if ">" in drawn:
            parts = drawn.split(">")
            new_entries = []
            for s in parts[0].split("."):
                if s.strip():
                    new_entries.append(("Drawn Reactant", "Reactant", s.strip()))
            if len(parts) > 1:
                for s in parts[1].split("."):
                    if s.strip():
                        new_entries.append(("Drawn Reagent", "Reagent", s.strip()))
            if len(parts) > 2:
                for s in parts[2].split("."):
                    if s.strip():
                        new_entries.append(("Drawn Product", "Product", s.strip()))
            for name, role, smi in new_entries:
                rows.append(create_table_row(len(rows), name, role, smi))
            table_container.children = rows
            clear_structure_gallery()
            print(f"✅ Imported {len(new_entries)} component(s) from Ketcher. Click Predict to enumerate & render 2D cards.")
        else:
            frags = [f.strip() for f in drawn.split(".") if f.strip()]
            for f_smi in frags:
                rows.append(create_table_row(len(rows), f"Drawn {quick_input_role.value}", quick_input_role.value, f_smi))
            table_container.children = rows
            clear_structure_gallery()
            print(f"✅ Added {len(frags)} drawn structure(s) from Ketcher. Click Predict to enumerate & render 2D cards.")

def on_clear_clicked(b):
    table_container.children = [header_row]
    clear_structure_gallery()
    populate_ketcher_canvas(editor, [])
    with status_output:
        clear_output(wait=True)
        print("🧹 Cleared table, 2D gallery, and Ketcher canvas.")

preset_dropdown.observe(on_preset_change, names="value")
btn_predict.on_click(on_predict_clicked)
btn_sync_ketcher.on_click(on_sync_ketcher_clicked)
btn_add_typed.on_click(on_add_typed_clicked)
btn_add_drawn.on_click(on_add_drawn_clicked)
btn_clear_inputs.on_click(on_clear_clicked)

display(widgets.HBox([preset_dropdown, btn_predict, btn_sync_ketcher]))
display(widgets.HBox([quick_input_text, quick_input_role, btn_add_typed, btn_add_drawn, btn_clear_inputs]))
display(status_output)
display(table_container)
display(gallery_output)

# Load ONLY the Suzuki-Miyaura reactants, reagents & catalyst into the table (already pushed to Ketcher)
load_preset_inputs_only(DEFAULT_PRESET, update_ketcher=False)

In [ ]:
#@title 4. Unified Crude Reaction Peak Detection, Halogen Isotope & Confidence Scoring (dataframe = detected_df)
import subprocess
import sys
import os
import re
import warnings

try:
    import scipy.signal
    from scipy.optimize import least_squares
except ImportError:
    print("🔧 Installing missing dependencies (scipy, numpy) via uv...")
    subprocess.run(["uv", "pip", "install", "--system", "--upgrade", "numpy", "scipy"], capture_output=True)
    import scipy.signal
    from scipy.optimize import least_squares
    print("✅ Repair complete.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import display, clear_output, Markdown
import ipywidgets as widgets
import pyopenms as po
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

warnings.filterwarnings('ignore')

if 'hplc-py' not in sys.path:
    sys.path.append(os.path.abspath('hplc-py'))
from hplc.quant import Chromatogram

# 1. Always sync `od` with the active mzML file from Cell 2
mzml_filename = globals().get("mzml_filename", "crude_suzuki_reaction.mzML")
if 'od' not in globals() or od.size() == 0:
    if os.path.exists(mzml_filename):
        od = po.MSExperiment()
        po.FileHandler().loadExperiment(mzml_filename, od)

time_full = hplc_df['Time'].to_numpy(dtype=float)
signal_full = hplc_df['Signal'].to_numpy(dtype=float)
t_min_full, t_max_full = float(time_full.min()), float(time_full.max())
t_span_full = max(t_max_full - t_min_full, 1e-6)
dt_median = float(np.median(np.diff(time_full))) if len(time_full) > 1 else 1.0

# --- 2. Adaptive Peak-Width, Baseline Drift & Noise-Floor Estimation ---
def estimate_chromatogram_characteristics(t_arr, y_arr):
    """Automatically characterizes dt, noise floor, baseline drift, and peak widths for ANY chromatogram."""
    dt = float(np.median(np.diff(t_arr))) if len(t_arr) > 1 else 1.0
    span = max(float(t_arr[-1] - t_arr[0]), dt)
    y_max = float(np.max(y_arr)) if len(y_arr) else 1.0
    y_min = float(np.min(y_arr)) if len(y_arr) else 0.0
    y_range = max(y_max - y_min, 1e-12)
    y_norm = (y_arr - y_min) / y_range

    # Robust high-frequency noise estimate from 2nd differences (immune to broad peaks)
    if len(y_norm) >= 5:
        d2 = np.diff(y_norm, n=2)
        sigma_noise = float(1.4826 * np.median(np.abs(d2 - np.median(d2))) / np.sqrt(6.0))
    else:
        sigma_noise = 0.001

    # Check whether meaningful baseline offset/drift exists at chromatogram edges/troughs
    n_edge = max(3, int(len(y_arr) * 0.05))
    edge_start = float(np.median(y_arr[:n_edge]))
    edge_end = float(np.median(y_arr[-n_edge:]))
    p5 = float(np.percentile(y_arr, 5))
    baseline_drift_frac = max(edge_start, edge_end, p5) / (y_max + 1e-12)
    needs_snip = baseline_drift_frac > 0.035

    # Estimate characteristic peak FWHM in native time units
    rough_prom = max(0.015, 6.0 * sigma_noise)
    rough_idx, _ = scipy.signal.find_peaks(y_norm, prominence=rough_prom, distance=2)
    if len(rough_idx) > 0:
        widths_pts = scipy.signal.peak_widths(y_norm, rough_idx, rel_height=0.5)[0]
        fwhms_t = widths_pts * dt
        median_fwhm = float(np.median(fwhms_t))
        max_fwhm = float(np.max(fwhms_t))
    else:
        median_fwhm = max(6.0 * dt, 0.04 * span)
        max_fwhm = median_fwhm

    # hplc-py requires approx_peak_width >= 10 * dt; scale safely to ~3x the broadest peak FWHM
    adaptive_snip_width = float(np.clip(max(3.0 * max_fwhm, 12.0 * dt), 12.0 * dt, 0.45 * span))
    return {
        "dt": dt,
        "span": span,
        "sigma_noise": sigma_noise,
        "baseline_drift_frac": baseline_drift_frac,
        "needs_snip": needs_snip,
        "median_fwhm": median_fwhm,
        "max_fwhm": max_fwhm,
        "adaptive_snip_width": adaptive_snip_width,
    }

chrom_stats = estimate_chromatogram_characteristics(time_full, signal_full)

def prepare_working_df(df_in, t_start, t_end, baseline_mode="Auto"):
    """Slices df_in to [t_start, t_end] and applies minimal floor subtraction when SNIP is not needed."""
    sub = df_in[(df_in['Time'] >= t_start) & (df_in['Time'] <= t_end)].copy().reset_index(drop=True)
    stats = estimate_chromatogram_characteristics(sub['Time'].to_numpy(dtype=float), sub['Signal'].to_numpy(dtype=float))
    use_snip = stats["needs_snip"] if baseline_mode == "Auto" else (baseline_mode == "SNIP (Adaptive Width)")
    sub['Raw_TIC'] = sub['Signal'].astype(float)
    if not use_snip:
        floor_val = float(np.percentile(sub['Signal'].to_numpy(dtype=float), 2))
        if floor_val > 0 and floor_val < 0.05 * float(sub['Signal'].max()):
            sub['Signal'] = np.maximum(0.0, sub['Signal'].to_numpy(dtype=float) - floor_val)
    return sub, stats, use_snip

# --- 3. Candidate Fetching & Synthetic Adduct Definitions ---
def get_candidates_from_table():
    """Fetches Reactants, Reagents, Predicted Products, and Byproducts from Cell 3's table_container."""
    if 'table_container' in globals():
        candidates = []
        for row in table_container.children[1:]:
            name = row.children[1].value.strip()
            classification = row.children[2].value
            smi = row.children[3].value.strip()
            if smi:
                candidates.append((name, classification, smi))
        if candidates:
            return candidates
    return [
        ("Methyl 4-bromobenzoate", "Reactant", "COC(=O)c1ccc(Br)cc1"),
        ("(4-Methoxyphenyl)boronic acid", "Reactant", "COc1ccc(B(O)O)cc1"),
        ("Triphenylphosphine oxide (TPPO)", "Reagent", "O=P(c1ccccc1)(c1ccccc1)c1ccccc1"),
        ("Methyl 4'-methoxy-[1,1'-biphenyl]-4-carboxylate", "Product", "COc1ccc(-c2ccc(C(=O)OC)cc2)cc1"),
        ("Anisole (Protodeboronation Byproduct)", "Byproduct", "COc1ccccc1"),
        ("4-Methoxyphenol (Boronic Oxidation Impurity)", "Byproduct", "COc1ccc(O)cc1"),
        ("4'-Methoxybiphenyl-4-carboxylic acid (Ester Hydrolysis)", "Byproduct", "COc1ccc(-c2ccc(C(=O)O)cc2)cc1"),
        ("Dimethyl [1,1'-biphenyl]-4,4'-dicarboxylate (Homocoupling Dimer)", "Byproduct", "COC(=O)c1ccc(-c2ccc(C(=O)OC)cc2)cc1"),
    ]

esi_pos_adducts = [
    ("[M+H]+",       1, 1.007276,  1),
    ("[M+NH4]+",     1, 18.033823, 1),
    ("[M+Na]+",      1, 22.989218, 1),
    ("[M+CH3OH+H]+", 1, 33.033489, 1),
    ("[M+K]+",       1, 38.963158, 1),
    ("[M+ACN+H]+",   1, 42.033823, 1),
    ("[2M+H]+",      2, 1.007276,  1),
    ("[2M+Na]+",     2, 22.989218, 1),
]

insource_neutral_losses = [
    ("[M+H-CH3OH]+ (Acylium)", -32.026215),
    ("[M+H-H2O]+",             -18.010565),
    ("[M+H-CO]+",              -27.994915),
    ("[M+H-CO2]+",             -43.989829),
    ("[M+H-C4H8]+ (-tBu)",     -56.062600),
    ("[M+H-Boc]+",             -100.052430),
    ("[M+H-HCl]+",             -35.976678),
    ("[M+H-HBr]+",             -79.926160),
]

def parse_dominant_mz_string(dom_str):
    if not dom_str or dom_str == "None":
        return []
    parsed = []
    for item in str(dom_str).split(","):
        m = re.search(r"([0-9]+\.[0-9]+)\s*\(([0-9.]+)%\)", item.strip())
        if m:
            parsed.append((float(m.group(1)), float(m.group(2))))
    return parsed

def compute_confidence_from_dominant_mz(best_group, dom_mz_list, full_window_spectrum):
    adducts = best_group["adducts"]
    primary = adducts[0]
    mol = best_group["mols"][0]
    exact_mass = best_group["exact_mass"]

    mean_abs_ppm = np.mean([abs(a["ppm"]) for a in adducts])
    ppm_score = max(0.0, 35.0 * (1.0 - (mean_abs_ppm / 15.0)))

    matched_dom_rel_int = 0.0
    for a in adducts:
        for dom_mz, dom_rel in dom_mz_list:
            if abs(dom_mz - a["calc_mz"]) <= 0.015:
                matched_dom_rel_int = max(matched_dom_rel_int, dom_rel)
    if matched_dom_rel_int >= 95.0:
        dom_rank_score = 35.0
    elif matched_dom_rel_int > 0:
        dom_rank_score = 12.0 + 23.0 * (matched_dom_rel_int / 100.0)
    else:
        dom_rank_score = 5.0

    num_C  = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 6)
    num_N  = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 7)
    num_S  = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 16)
    num_Cl = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 17)
    num_Br = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 35)
    num_B  = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 5)

    theo_m1_rel = (num_C * 1.07) + (num_N * 0.37) + (num_S * 0.76) or 10.0
    m0_mz = primary["calc_mz"]
    m1_target = m0_mz + 1.003355
    m2_target = m0_mz + (1.997953 if num_Br > 0 else (1.997050 if num_Cl > 0 else 2.006710))
    m_minus_1_target = m0_mz - 0.996368  # 10B relative to 11B

    iso_score = 0.0
    iso_evidence = []

    m1_obs_rel = None
    for dom_mz, dom_rel in dom_mz_list:
        if abs(dom_mz - m1_target) <= 0.018 and matched_dom_rel_int > 0:
            m1_obs_rel = (dom_rel / matched_dom_rel_int) * 100.0
            break
    if m1_obs_rel is None and full_window_spectrum and matched_dom_rel_int > 0:
        m0_int = max([inten for mz_k, inten in full_window_spectrum.items() if abs(mz_k - m0_mz) <= 0.018], default=0)
        m1_int = max([inten for mz_k, inten in full_window_spectrum.items() if abs(mz_k - m1_target) <= 0.018], default=0)
        if m0_int > 0 and m1_int > 0:
            m1_obs_rel = (m1_int / m0_int) * 100.0

    if m1_obs_rel is not None:
        rel_diff = abs(m1_obs_rel - theo_m1_rel)
        iso_score += max(5.0, 12.0 * (1.0 - min(rel_diff / 15.0, 1.0)))
        iso_evidence.append(f"[M+H+1]+ ({m1_obs_rel:.0f}% vs {theo_m1_rel:.0f}% theo)")

    m2_obs_rel = None
    for dom_mz, dom_rel in dom_mz_list:
        if abs(dom_mz - m2_target) <= 0.020 and matched_dom_rel_int > 0:
            m2_obs_rel = (dom_rel / matched_dom_rel_int) * 100.0
            break
    if m2_obs_rel is not None:
        if num_Br > 0 and abs(m2_obs_rel - 97.3 * num_Br) <= 25.0:
            iso_score += 8.0
            iso_evidence.append(f"79Br/81Br [M+2]+ ({m2_obs_rel:.0f}%)")
        elif num_Cl > 0 and abs(m2_obs_rel - 32.0 * num_Cl) <= 18.0:
            iso_score += 8.0
            iso_evidence.append(f"35Cl/37Cl [M+2]+ ({m2_obs_rel:.0f}%)")
        else:
            iso_score += 4.0
            iso_evidence.append("[M+H+2]+ iso")

    if num_B > 0:
        for dom_mz, dom_rel in dom_mz_list:
            if abs(dom_mz - m_minus_1_target) <= 0.020:
                iso_score += 5.0
                iso_evidence.append(f"10B/11B [M-1]+ ({dom_rel:.0f}%)")
                break

    iso_score = min(20.0, iso_score)

    corrob_score = 0.0
    if len(adducts) > 1:
        corrob_score += min(7.0, (len(adducts) - 1) * 5.0)
        iso_evidence.append("+".join([a["adduct"] for a in adducts[:2]]))

    mh_mz = exact_mass + 1.007276
    for loss_name, shift in insource_neutral_losses:
        frag_mz = mh_mz + shift
        if frag_mz > 65.0:
            for dom_mz, dom_rel in dom_mz_list:
                if abs(dom_mz - frag_mz) <= 0.018:
                    corrob_score += 5.0
                    iso_evidence.append(f"{loss_name} ({dom_mz:.2f})")
                    break

    corrob_score = min(10.0, corrob_score)
    total_conf = round(min(99.9, ppm_score + dom_rank_score + iso_score + corrob_score), 1)
    evidence_summary = f"Dom.Rank: {matched_dom_rel_int:.0f}%"
    if iso_evidence:
        evidence_summary += " | " + ", ".join(iso_evidence[:2])
    return total_conf, evidence_summary

# --- 4. Noise-Floor-Aware Adaptive Prominence Selection ---
print(f"Running Adaptive Chromatogram Characterization on '{mzml_filename}' ({t_min_full:.1f}–{t_max_full:.1f} s)...")
work_df_init, stats_init, use_snip_init = prepare_working_df(hplc_df, t_min_full, t_max_full, baseline_mode="Auto")

prominence_range = np.logspace(-2.7, -0.8, 10)  # 0.002 to 0.158 so minor 1–5% byproducts are captured
results = []
sig_init = work_df_init['Signal'].to_numpy(dtype=float)
sig_init_norm = (sig_init - sig_init.min()) / (sig_init.max() - sig_init.min() + 1e-12)
min_dist_pts = max(2, int(round(0.5 * stats_init["median_fwhm"] / max(stats_init["dt"], 1e-6))))

for p in prominence_range:
    idxs, _ = scipy.signal.find_peaks(sig_init_norm, prominence=p, distance=min_dist_pts)
    results.append((float(p), int(len(idxs))))

proms_all = np.array([x[0] for x in results])
counts_all = np.array([x[1] for x in results])

# Choose lowest prominence cleanly above empirical noise floor (avoiding the right-hand 2-peak collapse!)
noise_floor_prom = max(6.0 * stats_init["sigma_noise"], 0.002)
valid_mask = (proms_all >= noise_floor_prom) & (counts_all > 0) & (counts_all <= 25)
if np.any(valid_mask):
    cand_proms = proms_all[valid_mask]
    cand_counts = counts_all[valid_mask]
    target_count = int(cand_counts[0])
    stable_idxs = np.where(cand_counts >= max(1, target_count - 1))[0]
    best_rel = int(stable_idxs[min(len(stable_idxs) - 1, 1)])
    optimal_prominence = float(cand_proms[best_rel])
    optimal_count = int(cand_counts[best_rel])
else:
    optimal_prominence = 0.005
    optimal_count = int(counts_all[np.argmin(np.abs(proms_all - 0.005))])

default_prominence = float(optimal_prominence)

display(Markdown("---"))
display(Markdown("### ✅ Adaptive Chromatogram Characterization Complete"))
baseline_desc = (
    f"Adaptive SNIP (width={stats_init['adaptive_snip_width']:.1f} s)"
    if use_snip_init
    else "Minimal Floor Offset (flat baseline preserved)"
)
display(Markdown(
    f"- **Active Time Range:** `{t_min_full:.1f}–{t_max_full:.1f} s` (`dt = {stats_init['dt']:.2f} s`, median FWHM ≈ `{stats_init['median_fwhm']:.1f} s`)\n"
    f"- **Baseline Mode (`Auto`):** `{baseline_desc}` "
    f"(edge drift = `{stats_init['baseline_drift_frac']*100:.2f}%`)\n"
    f"- **Optimal Prominence Threshold:** `{optimal_prominence:.4f}` | **TIC Local Maxima Resolved:** `{optimal_count}` (plus EIC shoulder seeding)"
))

fig_opt, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={'width_ratios': [2.5, 1]})
ax1.plot(proms_all, counts_all, marker='o', linestyle='-', color='#1e293b', linewidth=2, label='Detected TIC Peaks')
ax1.axvline(optimal_prominence, color='#e11d48', linestyle='--', linewidth=2, label=f'Selected Threshold ({optimal_prominence:.4f})')
ax1.scatter([optimal_prominence], [optimal_count], color='#e11d48', s=140, zorder=5)
ax1.set_xscale('log')
ax1.set_xticks(prominence_range)
ax1.set_xticklabels([f"{x:.4f}" for x in prominence_range], rotation=45)
ax1.set_title('Adaptive Peak Sensitivity Curve (Noise-Floor Aware)', fontsize=12.5, pad=10)
ax1.set_xlabel('Prominence Threshold (Log Scale)', fontsize=11)
ax1.set_ylabel('TIC Local Maxima Count', fontsize=11)
ax1.legend(fontsize=9.5)
ax1.grid(True, which="both", ls="--", alpha=0.4)

ax2.axis('tight')
ax2.axis('off')
table_data = [[f"{p:.4f}", int(c)] for p, c in zip(proms_all, counts_all)]
table = ax2.table(cellText=table_data, colLabels=['Prominence', 'TIC Peaks'], loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(9.5)
table.scale(1.1, 1.45)
opt_row = int(np.argmin(np.abs(proms_all - optimal_prominence))) + 1
for (r_i, c_i), cell in table.get_celld().items():
    if r_i == opt_row:
        cell.set_edgecolor('#e11d48')
        cell.set_linewidth(2)
ax2.set_title('Prominence Sweep', fontsize=11.5, pad=10)
plt.tight_layout()
plt.show()

# --- 5. Interactive Controls & Adaptive Unified Execution Pipeline ---
prominence_input = widgets.FloatText(value=round(default_prominence, 4), description='Prominence:', step=0.001, style={'description_width': 'initial'}, layout=widgets.Layout(width='185px'))
rt_start_input = widgets.FloatText(value=round(t_min_full, 1), description='Start RT:', step=5.0, style={'description_width': 'initial'}, layout=widgets.Layout(width='165px'))
rt_end_input = widgets.FloatText(value=round(t_max_full, 1), description='End RT:', step=5.0, style={'description_width': 'initial'}, layout=widgets.Layout(width='165px'))
baseline_dropdown = widgets.Dropdown(options=['Auto', 'Minimal Floor Offset', 'SNIP (Adaptive Width)'], value='Auto', description='Baseline:', style={'description_width': 'initial'}, layout=widgets.Layout(width='230px'))
eic_shoulder_toggle = widgets.Checkbox(value=True, description='Detect Co-Eluting Shoulders (EIC)', indent=False, layout=widgets.Layout(width='230px'))
rerun_button = widgets.Button(description='Rerun Peak Fitting', button_style='primary', icon='refresh', layout=widgets.Layout(width='170px'))
output_area = widgets.Output()

peak_ms1_spectra = {}
peak_assignments = {}
chrom = None
current_prominence_val = default_prominence

def find_candidate_eic_shoulders(od_exp, t_arr, sig_arr, tic_rts, candidates, stats, ppm_tol=15.0):
    """Detects buried/co-eluting shoulder peaks (e.g. dimers on a major product tail) via candidate [M+H]+ EICs."""
    if od_exp is None or not candidates or len(t_arr) < 5:
        return []
    t_lo, t_hi = float(t_arr[0]), float(t_arr[-1])
    scans = [(float(s.getRT()), *s.get_peaks()) for s in od_exp if s.getMSLevel() == 1 and t_lo <= s.getRT() <= t_hi]
    if len(scans) < 5:
        return []
    scans.sort(key=lambda x: x[0])
    scan_rts = np.array([s[0] for s in scans])
    tic_max = float(np.max(sig_arr)) if len(sig_arr) else 1.0
    min_sep = max(0.75 * stats["median_fwhm"], 3.0 * stats["dt"])

    shoulder_rts = []
    all_known = list(tic_rts)
    for _, _, smi in candidates:
        mol = Chem.MolFromSmiles(smi)
        if not mol:
            continue
        target_mz = Descriptors.ExactMolWt(mol) + 1.007276
        tol_da = max(0.015, target_mz * ppm_tol * 1e-6)
        eic = np.array([
            float(np.sum(ints[(mzs >= target_mz - tol_da) & (mzs <= target_mz + tol_da)]))
            for _, mzs, ints in scans
        ])
        eic_max = float(eic.max()) if len(eic) else 0.0
        if eic_max < max(1e4, 0.008 * tic_max):
            continue
        eic_norm = eic / (eic_max + 1e-12)
        eic_peaks, _ = scipy.signal.find_peaks(eic_norm, prominence=0.30, distance=3)
        for idx in eic_peaks:
            rt_cand = float(scan_rts[idx])
            tic_at_rt = float(np.interp(rt_cand, t_arr, sig_arr))
            if tic_at_rt < 0.008 * tic_max:
                continue
            if all(abs(rt_cand - existing) >= min_sep for existing in all_known):
                shoulder_rts.append(rt_cand)
                all_known.append(rt_cand)
    return sorted(shoulder_rts)

def _fast_bounded_fallback(chrom_obj, t_vec, y_vec, centers, stats_dict):
    """Sub-second bounded multi-Gaussian deconvolution fallback compatible with Cell 5 & 8C."""
    if not centers:
        chrom_obj.unmixed_chromatograms = np.zeros((len(t_vec), 0))
        return pd.DataFrame(columns=["retention_time", "scale", "skew", "amplitude", "area", "signal_maximum", "peak_id"])

    n_p = len(centers)
    rt_tol = max(0.35 * stats_dict["median_fwhm"], 2.0 * stats_dict["dt"])
    sigma_init = max(stats_dict["median_fwhm"] / 2.355, 1.5 * stats_dict["dt"])
    sigma_hi = max(1.5 * stats_dict["max_fwhm"], 3.0 * stats_dict["dt"])

    p0, lb, ub = [], [], []
    for rt0 in centers:
        amp0 = max(float(np.interp(rt0, t_vec, y_vec)), 1.0)
        p0.extend([amp0, rt0, sigma_init])
        lb.extend([0.0, max(float(t_vec[0]), rt0 - rt_tol), stats_dict["dt"]])
        ub.extend([amp0 * 2.5 + 1.0, min(float(t_vec[-1]), rt0 + rt_tol), sigma_hi])

    def _res(params):
        model = np.zeros_like(t_vec)
        for k in range(n_p):
            a, mu, sig = params[3 * k], params[3 * k + 1], params[3 * k + 2]
            model += a * np.exp(-0.5 * ((t_vec - mu) / sig) ** 2)
        return model - y_vec

    opt = least_squares(_res, p0, bounds=(lb, ub), max_nfev=400, ftol=1e-4, xtol=1e-4)
    comps, rows = [], []
    for k in range(n_p):
        a, mu, sig = float(opt.x[3 * k]), float(opt.x[3 * k + 1]), float(opt.x[3 * k + 2])
        comp_curve = a * np.exp(-0.5 * ((t_vec - mu) / sig) ** 2)
        comps.append(comp_curve)
        area_val = float(np.trapezoid(comp_curve, t_vec) if hasattr(np, "trapezoid") else np.trapz(comp_curve, t_vec))
        rows.append({
            "retention_time": mu,
            "scale": sig,
            "skew": 0.0,
            "amplitude": a,
            "area": area_val,
            "signal_maximum": float(comp_curve.max()),
            "peak_id": k + 1,
        })
    chrom_obj.unmixed_chromatograms = np.column_stack(comps)
    return pd.DataFrame(rows)

def run_unified_pipeline(prom_val, t_start=None, t_end=None, baseline_mode="Auto", detect_shoulders=True):
    global detected_df, peak_assignments, peak_ms1_spectra, chrom, current_prominence_val
    current_prominence_val = float(prom_val)
    t_s = t_min_full if t_start is None else float(t_start)
    t_e = t_max_full if t_end is None else float(t_end)
    if t_e <= t_s:
        t_s, t_e = t_min_full, t_max_full

    with output_area:
        clear_output(wait=True)
        work_df, stats, use_snip = prepare_working_df(hplc_df, t_s, t_e, baseline_mode=baseline_mode)
        t_arr = work_df['Time'].to_numpy(dtype=float)
        sig_arr = work_df['Signal'].to_numpy(dtype=float)
        sig_norm = (sig_arr - sig_arr.min()) / (sig_arr.max() - sig_arr.min() + 1e-12)

        min_dist = max(2, int(round(0.5 * stats["median_fwhm"] / max(stats["dt"], 1e-6))))
        tic_indices, _ = scipy.signal.find_peaks(sig_norm, prominence=prom_val, distance=min_dist)
        tic_rts = [float(t_arr[i]) for i in tic_indices]

        candidates = get_candidates_from_table()
        shoulder_rts = (
            find_candidate_eic_shoulders(globals().get('od'), t_arr, sig_arr, tic_rts, candidates, stats)
            if detect_shoulders else []
        )
        seed_rts = sorted(tic_rts + shoulder_rts)

        baseline_label = (
            f"Adaptive SNIP ({stats['adaptive_snip_width']:.1f} s)"
            if use_snip
            else "Minimal Floor Offset"
        )
        print(
            f"Active window: {t_s:.1f}–{t_e:.1f} s | Prominence: {prom_val:.4f} | "
            f"Baseline: {baseline_label}"
        )
        shoulder_msg = (
            f" + {len(shoulder_rts)} EIC co-eluting shoulder(s) at {[round(r, 1) for r in shoulder_rts]} s."
            if shoulder_rts
            else "."
        )
        print(f"Detected {len(tic_rts)} TIC maxima{shoulder_msg}")

        # Fit peaks with bounded hplc-py settings, window splitting, and fast least-squares fallback
        chrom = Chromatogram(work_df, cols={'time': 'Time', 'signal': 'Signal'})
        fit_kwargs = dict(
            buffer=0,
            rel_height=0.95,
            correct_baseline=use_snip,
            approx_peak_width=stats["adaptive_snip_width"],
            param_bounds={
                "skew": [-4.0, 4.0],
                "scale": [stats["dt"], max(2.0 * stats["max_fwhm"], 4.0 * stats["dt"])],
            },
            max_iter=2500,
            optimizer_kwargs={"ftol": 1e-4, "xtol": 1e-4},
            verbose=True,
        )

        use_fallback = False
        try:
            if seed_rts:
                peaks = chrom.fit_peaks(
                    known_peaks=seed_rts,
                    tolerance=max(0.5, 2.0 * stats["dt"]),
                    **fit_kwargs
                ).reset_index(drop=True)
            else:
                peaks = chrom.fit_peaks(prominence=prom_val, **fit_kwargs).reset_index(drop=True)
            if len(peaks) == 0 and seed_rts:
                use_fallback = True
                fallback_reason = "hplc-py returned 0 peaks for seeded centers"
        except Exception as e:
            use_fallback = True
            fallback_reason = str(e)

        if use_fallback:
            print(f"⚡ hplc-py capped/fallback ({fallback_reason}) — running fast bounded deconvolution...")
            chrom = Chromatogram(work_df, cols={'time': 'Time', 'signal': 'Signal'})
            fallback_centers = seed_rts if seed_rts else tic_rts
            peaks = _fast_bounded_fallback(chrom, t_arr, sig_arr, fallback_centers, stats)

        # Preserve true raw TIC trace inside chrom.df so Cell 5 & Cell 8C always match Cell 2
        chrom.df['Raw_TIC'] = np.interp(chrom.df[chrom.time_col].to_numpy(dtype=float), work_df['Time'], work_df['Raw_TIC'])

        # Sort peaks chronologically and align peak_id 1..N with unmixed_chromatograms columns
        if len(peaks) > 0:
            peaks = peaks.sort_values('retention_time').reset_index(drop=True)
            orig_ids = peaks['peak_id'].astype(int).tolist()
            if getattr(chrom, 'unmixed_chromatograms', None) is not None and chrom.unmixed_chromatograms.shape[1] == len(orig_ids):
                reordered_unmixed = np.column_stack([chrom.unmixed_chromatograms[:, pid - 1] for pid in orig_ids])
                chrom.unmixed_chromatograms = reordered_unmixed
            peaks['peak_id'] = np.arange(1, len(peaks) + 1)
            chrom.peaks = peaks.copy()

        # Derive tight, per-peak elution windows & core apex intervals directly from deconvoluted curves
        c_time = chrom.df[chrom.time_col].to_numpy(dtype=float)
        unmixed = chrom.unmixed_chromatograms
        total_unmixed = np.sum(unmixed, axis=1) + 1e-12 if (unmixed is not None and unmixed.shape[1] > 0) else None

        mapped_starts, mapped_ends, core_windows = [], [], []
        for idx_row, row in peaks.iterrows():
            rt_apex = float(row['retention_time'])
            scale_est = float(row['scale']) if ('scale' in row and pd.notna(row['scale']) and row['scale'] > 0) else max(stats["median_fwhm"] / 2.355, 2.0 * stats["dt"])
            comp = unmixed[:, idx_row] if (unmixed is not None and idx_row < unmixed.shape[1]) else None

            if comp is not None and comp.max() > 0:
                above_10 = np.where(comp >= 0.10 * comp.max())[0]
                p_start = float(c_time[above_10[0]]) if len(above_10) else max(t_s, rt_apex - 2.0 * scale_est)
                p_end = float(c_time[above_10[-1]]) if len(above_10) else min(t_e, rt_apex + 2.0 * scale_est)

                # Core window for clean MS1 extraction (where this component is >=50% of its height AND dominant)
                dom_mask = (comp >= 0.50 * comp.max()) & (comp >= 0.45 * total_unmixed)
                core_idxs = np.where(dom_mask)[0]
                if len(core_idxs) == 0:
                    core_idxs = np.where(comp >= 0.65 * comp.max())[0]
                c_start = float(c_time[core_idxs[0]]) if len(core_idxs) else max(p_start, rt_apex - 0.75 * scale_est)
                c_end = float(c_time[core_idxs[-1]]) if len(core_idxs) else min(p_end, rt_apex + 0.75 * scale_est)
            else:
                p_start = max(t_s, rt_apex - 2.0 * scale_est)
                p_end = min(t_e, rt_apex + 2.0 * scale_est)
                c_start = max(p_start, rt_apex - 0.75 * scale_est)
                c_end = min(p_end, rt_apex + 0.75 * scale_est)

            # Ensure at least 2 scans fall inside core window
            if c_end - c_start < 1.5 * stats["dt"]:
                c_start = max(t_s, rt_apex - stats["dt"])
                c_end = min(t_e, rt_apex + stats["dt"])

            mapped_starts.append(round(p_start, 2))
            mapped_ends.append(round(p_end, 2))
            core_windows.append((c_start, c_end, max(1.5 * scale_est, 0.5 * (p_end - p_start), 2.0 * stats["dt"])))

        peaks['peak start time'] = mapped_starts
        peaks['peak end time'] = mapped_ends

        if len(peaks) > 0 and peaks['area'].sum() > 0:
            peaks['normalised area'] = (peaks['area'] / peaks['area'].sum() * 100).map('{:.2f}%'.format)
        else:
            peaks['normalised area'] = '0.00%'

        # Extract dominant MS1 ions strictly from each peak's core apex window (prevents cross-peak contamination)
        peak_spectra = {i: {} for i in range(len(peaks))}
        if 'od' in globals():
            for spec in od:
                if spec.getMSLevel() == 1:
                    rt = float(spec.getRT())
                    mz_array, int_array = spec.get_peaks()
                    if len(mz_array) == 0:
                        continue
                    mzs_rounded = np.round(mz_array, 3)
                    for i, (c_start, c_end, _) in enumerate(core_windows):
                        if c_start <= rt <= c_end:
                            for m, intensity in zip(mzs_rounded, int_array):
                                peak_spectra[i][m] = peak_spectra[i].get(m, 0.0) + float(intensity)

        TOP_N_PEAKS = 10
        MIN_REL_INTENSITY = 2.0
        dominant_mz_col = []
        for i in range(len(peaks)):
            spectra = peak_spectra[i]
            if not spectra:
                dominant_mz_col.append("None")
                continue
            sorted_mzs = sorted(spectra.items(), key=lambda x: x[1], reverse=True)
            max_int = sorted_mzs[0][1]
            top_strs = []
            for m, inten in sorted_mzs:
                rel_int = (inten / max_int) * 100.0
                if rel_int >= MIN_REL_INTENSITY:
                    top_strs.append(f"{m:.3f} ({rel_int:.0f}%)")
                if len(top_strs) >= TOP_N_PEAKS:
                    break
            dominant_mz_col.append(", ".join(top_strs) if top_strs else "None")

        peaks['dominant m/z'] = dominant_mz_col

        # Build theoretical ESI+ ion library from Cell 3 table
        theoretical_ions = []
        for name, classification, smi in candidates:
            mol = Chem.MolFromSmiles(smi)
            if mol:
                exact_mass = Descriptors.ExactMolWt(mol)
                formula = rdMolDescriptors.CalcMolFormula(mol)
                for adduct_name, mult, shift, z in esi_pos_adducts:
                    calc_mz = (exact_mass * mult + shift) / z
                    theoretical_ions.append({
                        "name": f"{name} ({classification})",
                        "raw_name": name,
                        "classification": classification,
                        "smiles": smi,
                        "mol": mol,
                        "formula": formula,
                        "exact_mass": round(exact_mass, 4),
                        "adduct": adduct_name,
                        "calc_mz": calc_mz
                    })

        PPM_TOLERANCE = 15.0
        MIN_PEAK_INTENSITY = 1e4
        peak_assignments = {}

        if 'od' in globals():
            for idx_row, peak_row in peaks.iterrows():
                p_id = int(peak_row['peak_id'])
                c_start, c_end, rt_tol_adaptive = core_windows[idx_row]
                rt_apex = float(peak_row['retention_time'])
                dom_mz_parsed = parse_dominant_mz_string(peak_row['dominant m/z'])
                window_scans = [spec for spec in od if spec.getMSLevel() == 1 and c_start <= spec.getRT() <= c_end]
                if not window_scans:
                    continue

                window_max_ion = max((float(np.max(s.get_peaks()[1])) for s in window_scans if len(s.get_peaks()[1])), default=0.0)
                min_rel_abs = max(MIN_PEAK_INTENSITY, 0.02 * window_max_ion)

                mass_group_hits = {}
                for ion in theoretical_ions:
                    target_mz = ion["calc_mz"]
                    tol_da = max(0.015, target_mz * (PPM_TOLERANCE * 1e-6))
                    max_int = 0.0
                    obs_mz_at_max = target_mz
                    rt_at_max = rt_apex
                    for spec in window_scans:
                        mzs, ints = spec.get_peaks()
                        mask = (mzs >= target_mz - tol_da) & (mzs <= target_mz + tol_da)
                        if np.any(mask):
                            best_idx = int(np.argmax(ints[mask]))
                            peak_int = float(ints[mask][best_idx])
                            if peak_int > max_int:
                                max_int = peak_int
                                obs_mz_at_max = float(mzs[mask][best_idx])
                                rt_at_max = float(spec.getRT())

                    if max_int >= min_rel_abs and abs(rt_at_max - rt_apex) <= rt_tol_adaptive:
                        em = ion["exact_mass"]
                        if em not in mass_group_hits:
                            mass_group_hits[em] = {
                                "exact_mass": em,
                                "formula": ion["formula"],
                                "names": [],
                                "raw_names": [],
                                "classifications": [],
                                "mols": [],
                                "total_intensity": 0.0,
                                "adducts": {}
                            }
                        if ion["name"] not in mass_group_hits[em]["names"]:
                            mass_group_hits[em]["names"].append(ion["name"])
                            mass_group_hits[em]["raw_names"].append(ion["raw_name"])
                            mass_group_hits[em]["classifications"].append(ion["classification"])
                            mass_group_hits[em]["mols"].append(ion["mol"])
                        if ion["adduct"] not in mass_group_hits[em]["adducts"]:
                            ppm_err = ((obs_mz_at_max - target_mz) / target_mz) * 1e6
                            adduct_weight = 1.25 if ion["adduct"] == "[M+H]+" else 1.0
                            mass_group_hits[em]["adducts"][ion["adduct"]] = {
                                "adduct": ion["adduct"],
                                "obs_mz": obs_mz_at_max,
                                "calc_mz": target_mz,
                                "ppm": ppm_err,
                                "intensity": max_int
                            }
                            mass_group_hits[em]["total_intensity"] += max_int * adduct_weight

                if mass_group_hits:
                    best_group = max(mass_group_hits.values(), key=lambda x: x["total_intensity"])
                    sorted_adducts = sorted(best_group["adducts"].values(), key=lambda x: x["intensity"], reverse=True)
                    best_group["adducts"] = sorted_adducts
                    is_isobaric = len(best_group["raw_names"]) > 1
                    iso_tag = f" [Isobaric Isomers, {best_group['formula']}]" if is_isobaric else ""
                    conf_score, evidence_str = compute_confidence_from_dominant_mz(
                        best_group, dom_mz_parsed, peak_spectra.get(idx_row, {})
                    )
                    assigned_role = best_group["classifications"][0] if best_group["classifications"] else "Byproduct"
                    peak_assignments[p_id] = {
                        "display_name": " / ".join(best_group["raw_names"]),
                        "full_assignment": " / ".join(best_group["names"]) + iso_tag,
                        "role": assigned_role,
                        "classification": assigned_role,  # keeps Cell 5, 7, 8C & 8D in sync
                        "is_isobaric": is_isobaric,
                        "formula": best_group["formula"],
                        "mols": best_group["mols"],
                        "adducts": sorted_adducts,
                        "confidence": conf_score,
                        "evidence": evidence_str
                    }

        role_col, assignment_col, confidence_col = [], [], []
        for _, row in peaks.iterrows():
            p_id = int(row['peak_id'])
            if p_id in peak_assignments:
                assign_info = peak_assignments[p_id]
                top_adduct = assign_info["adducts"][0]
                role_col.append(assign_info["role"])
                assignment_col.append(
                    f"{assign_info['full_assignment']} [{top_adduct['adduct']}: {top_adduct['obs_mz']:.4f} m/z ({top_adduct['ppm']:+.1f} ppm)]"
                )
                confidence_col.append(f"{assign_info['confidence']:.1f}% ({assign_info['evidence']})")
            else:
                dom_list = parse_dominant_mz_string(row['dominant m/z'])
                base_str = f"Base m/z: {dom_list[0][0]:.3f}" if dom_list else "No MS1 signal"
                role_col.append("Unassigned")
                assignment_col.append(f"Unassigned ({base_str})")
                confidence_col.append("N/A")

        peaks['Role'] = role_col
        peaks['Assignment'] = assignment_col
        peaks['Confidence (%)'] = confidence_col
        detected_df = peaks.copy()
        peak_ms1_spectra = {int(row['peak_id']): peak_spectra.get(idx_row, {}) for idx_row, row in peaks.iterrows()}

        total_area = float(detected_df['area'].sum()) if len(detected_df) else 1.0
        role_areas = detected_df.groupby('Role')['area'].sum().to_dict() if total_area > 0 else {}
        prod_pct  = (role_areas.get('Product', 0.0) / total_area) * 100.0
        sm_pct    = (role_areas.get('Reactant', 0.0) / total_area) * 100.0
        bp_pct    = (role_areas.get('Byproduct', 0.0) / total_area) * 100.0
        reag_pct  = (role_areas.get('Reagent', 0.0) / total_area) * 100.0
        unass_pct = (role_areas.get('Unassigned', 0.0) / total_area) * 100.0
        rxn_denom = prod_pct + sm_pct + bp_pct
        rxn_conv  = ((prod_pct + bp_pct) / rxn_denom * 100.0) if rxn_denom > 0 else 0.0

        display(Markdown(
            f"#### 📊 Crude Reaction LCAP% Summary\n"
            f"- **Target Product LCAP:** `{prod_pct:.1f}%` | **Est. SM Conversion:** `{rxn_conv:.1f}%`\n"
            f"- **Unreacted SM:** `{sm_pct:.1f}%` | **Byproducts:** `{bp_pct:.1f}%` | "
            f"**Reagents:** `{reag_pct:.1f}%` | **Unassigned:** `{unass_pct:.1f}%`"
        ))

        def highlight_peaks(row):
            colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
            c = mcolors.to_rgba(colors[(int(row['peak_id']) - 1) % len(colors)], alpha=0.35)
            return [f'background-color: rgba({int(c[0]*255)}, {int(c[1]*255)}, {int(c[2]*255)}, {c[3]})' for _ in row]

        display(detected_df.style.apply(highlight_peaks, axis=1))
        print(f"\nFinal deconvoluted crude reaction peaks: {len(detected_df)} | Assigned species: {len(peak_assignments)}")
        print("✅ Data ready! Run Cell 5 below to render the interactive crude reaction chromatogram & MS1 comparison widgets.")

def on_rerun_clicked(b):
    run_unified_pipeline(
        prominence_input.value,
        t_start=rt_start_input.value,
        t_end=rt_end_input.value,
        baseline_mode=baseline_dropdown.value,
        detect_shoulders=eic_shoulder_toggle.value
    )

rerun_button.on_click(on_rerun_clicked)
display(widgets.HBox([prominence_input, rt_start_input, rt_end_input, baseline_dropdown]))
display(widgets.HBox([eic_shoulder_toggle, rerun_button]))
display(output_area)
run_unified_pipeline(default_prominence, t_min_full, t_max_full, "Auto", True)

In [ ]:
#@title 5. Plot Deconvoluted Crude Reaction Chromatogram with Role Badges, Hover Structure Cards & Halogen Isotope MS1 Comparison

import subprocess
import sys
import io
import re
import json
import base64
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
from rdkit import Chem
from rdkit.Chem import Draw, rdMolDescriptors

try:
    import plotly.graph_objects as go
    import plotly.io as pio
    from plotly.subplots import make_subplots
except ImportError:
    print("🔧 Installing missing dependencies (plotly, kaleido) via uv...")
    subprocess.run(["uv", "pip", "install", "--system", "--upgrade", "plotly", "kaleido"], capture_output=True)
    import plotly.graph_objects as go
    import plotly.io as pio
    from plotly.subplots import make_subplots
    print("✅ Repair complete.")

try:
    import kaleido
except ImportError:
    print("🔧 Installing/reloading missing kaleido via uv...")
    subprocess.run(["uv", "pip", "install", "--system", "--upgrade", "kaleido"], capture_output=True)
    print("✅ Kaleido installation complete.")

# Ensure Plotly renders natively in Colab
if 'google.colab' in sys.modules:
    pio.renderers.default = 'colab'

if 'to_plotly_color' not in globals():
    def to_plotly_color(color_val):
        rgba = mcolors.to_rgba(color_val)
        return f"rgb({int(rgba[0]*255)}, {int(rgba[1]*255)}, {int(rgba[2]*255)})"

# Role styling matching Section 2 Cell 3's Reaction Predictor
ROLE_STYLES = {
    "Reactant":          {"bg": "#eff6ff", "fg": "#1d4ed8", "border": "#bfdbfe", "short": "SM"},
    "Starting Material": {"bg": "#eff6ff", "fg": "#1d4ed8", "border": "#bfdbfe", "short": "SM"},
    "Reagent":           {"bg": "#fefce8", "fg": "#a16207", "border": "#fde047", "short": "Rgt"},
    "Product":           {"bg": "#f0fdf4", "fg": "#15803d", "border": "#86efac", "short": "Prod"},
    "Target Product":    {"bg": "#f0fdf4", "fg": "#15803d", "border": "#86efac", "short": "Prod"},
    "Byproduct":         {"bg": "#fff1f2", "fg": "#be123c", "border": "#fecdd3", "short": "Byp"},
}

def _extract_role(assign_dict):
    """Infers the synthetic role (Reactant, Reagent, Product, Byproduct) from peak_assignments."""
    if not assign_dict:
        return "Unassigned"
    if "classification" in assign_dict and assign_dict["classification"] in ROLE_STYLES:
        return assign_dict["classification"]
    full_str = str(assign_dict.get("full_assignment", ""))
    for role_key in ROLE_STYLES:
        if f"({role_key})" in full_str or role_key.lower() in full_str.lower():
            return role_key
    return "Product"

def _detect_halogen_tag(formula_str: str) -> str:
    """Flags Br or Cl atoms so chemists immediately see 79Br/81Br or 35Cl/37Cl M+2 signatures."""
    if not formula_str:
        return ""
    br_match = re.search(r"Br(\d*)", formula_str)
    cl_match = re.search(r"Cl(\d*)", formula_str)
    tags = []
    if br_match:
        n_br = int(br_match.group(1) or 1)
        tags.append(f"⁷⁹Br/⁸¹Br ({n_br}×Br, M+2 ≈1:1)")
    if cl_match:
        n_cl = int(cl_match.group(1) or 1)
        tags.append(f"³⁵Cl/³⁷Cl ({n_cl}×Cl, M+2 ≈3:1)")
    return " | ".join(tags)

if 'detected_df' not in globals() or 'chrom' not in globals() or chrom is None:
    print("⚠️ Required data (`detected_df`, `chrom`) not found. Please run Section 2 Cell 4 first.")
else:
    def _peak_comparison_label(p_id):
        """Human-readable label for a peak_id with synthetic role and adduct summary."""
        if 'peak_assignments' in globals() and p_id in peak_assignments:
            a_info = peak_assignments[p_id]
            role = _extract_role(a_info)
            role_short = ROLE_STYLES.get(role, {"short": role})["short"]
            iso_short = " [Isobaric]" if a_info.get("is_isobaric", False) else ""
            adduct_summary = ", ".join([f"{x['adduct']} ({x['obs_mz']:.2f})" for x in a_info['adducts'][:2]])
            return f"Peak {p_id} [{role_short}]: {a_info['display_name']}{iso_short} [{adduct_summary}] (Conf: {a_info['confidence']:.0f}%)"
        if 'detected_df' in globals():
            dom_rows = detected_df.loc[detected_df['peak_id'] == p_id, 'dominant m/z']
            if len(dom_rows):
                dom_str = dom_rows.values[0]
                first_ion = str(dom_str).split(',')[0] if dom_str != "None" else "N/A"
                return f"Peak {p_id}: Unassigned Impurity (Base m/z: {first_ion})"
        return f"Peak {p_id}: Unassigned Impurity"

    # --- 1. Compute Crude Reaction Conversion & Component Breakdown Banner ---
    role_area_totals = {"Product": 0.0, "Reactant": 0.0, "Byproduct": 0.0, "Reagent": 0.0, "Unassigned": 0.0}
    for _, peak_row in detected_df.iterrows():
        p_id = int(peak_row['peak_id'])
        raw_area_str = str(peak_row.get('normalised area', '0%')).replace('%', '').strip()
        try:
            area_pct = float(raw_area_str)
        except ValueError:
            area_pct = 0.0

        if 'peak_assignments' in globals() and p_id in peak_assignments:
            r_key = _extract_role(peak_assignments[p_id])
            if r_key in ("Target Product", "Product"):
                role_area_totals["Product"] += area_pct
            elif r_key in ("Starting Material", "Reactant"):
                role_area_totals["Reactant"] += area_pct
            elif r_key == "Byproduct":
                role_area_totals["Byproduct"] += area_pct
            elif r_key == "Reagent":
                role_area_totals["Reagent"] += area_pct
            else:
                role_area_totals["Product"] += area_pct
        else:
            role_area_totals["Unassigned"] += area_pct

    rxn_denom = role_area_totals["Product"] + role_area_totals["Reactant"] + role_area_totals["Byproduct"]
    est_conversion = ((role_area_totals["Product"] + role_area_totals["Byproduct"]) / rxn_denom * 100.0) if rxn_denom > 0 else 0.0
    est_selectivity = (role_area_totals["Product"] / rxn_denom * 100.0) if rxn_denom > 0 else 0.0

    summary_banner_html = f"""
    <div style="display:flex; flex-wrap:wrap; gap:12px; align-items:center; background:#f8fafc;
                border:1px solid #cbd5e1; border-radius:10px; padding:10px 16px; margin-bottom:10px; font-family:sans-serif;">
        <div style="font-weight:700; font-size:13.5px; color:#0f172a; margin-right:8px;">
            ⚗️ Crude Reaction TIC Breakdown:
        </div>
        <span style="background:#f0fdf4; color:#15803d; border:1px solid #86efac; border-radius:6px; padding:3px 9px; font-size:12px; font-weight:700;">
            🎯 Target Product: {role_area_totals['Product']:.1f}%
        </span>
        <span style="background:#eff6ff; color:#1d4ed8; border:1px solid #bfdbfe; border-radius:6px; padding:3px 9px; font-size:12px; font-weight:700;">
            🧪 Unreacted SM: {role_area_totals['Reactant']:.1f}%
        </span>
        <span style="background:#fff1f2; color:#be123c; border:1px solid #fecdd3; border-radius:6px; padding:3px 9px; font-size:12px; font-weight:700;">
            ⚠️ Byproducts: {role_area_totals['Byproduct']:.1f}%
        </span>
        <span style="background:#fefce8; color:#a16207; border:1px solid #fde047; border-radius:6px; padding:3px 9px; font-size:12px; font-weight:700;">
            🧰 Reagents/Catalyst: {role_area_totals['Reagent']:.1f}%
        </span>
        <span style="background:#f3f4f6; color:#4b5563; border:1px solid #d1d5db; border-radius:6px; padding:3px 9px; font-size:12px; font-weight:600;">
            ❓ Unassigned: {role_area_totals['Unassigned']:.1f}%
        </span>
        <div style="margin-left:auto; font-size:12px; color:#334155;">
            <b>Est. SM Conversion:</b> {est_conversion:.1f}% &nbsp;|&nbsp; <b>LC-MS Assay Yield:</b> {est_selectivity:.1f}%
        </div>
    </div>
    """
    display(HTML(summary_banner_html))

    # --- 2. Build Clean Interactive Plotly Chromatogram with Floating Hover Structure Cards ---
    FIG_WIDTH_PX, FIG_HEIGHT_PX = 1300, 800
    MARGIN_L, MARGIN_R, MARGIN_T, MARGIN_B = 80, 40, 65, 220

    time_arr = chrom.df[chrom.time_col].values
    raw_signal = chrom.df[chrom.int_col].values
    X_RANGE = [max(0.0, float(np.min(time_arr)) - 5.0), float(np.max(time_arr)) + 5.0]

    convolved = np.sum(chrom.unmixed_chromatograms, axis=1) if chrom.peaks is not None else None
    y_max_plot = float(np.max(raw_signal))
    prom_val = globals().get('current_prominence_val', 0.05)

    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

    # Pre-build rich HTML hover cards (including Base64-encoded RDKit 2D structure PNGs) per peak
    hover_cards_by_peak = {}
    qc_struct_images = {}

    for _, peak_row in detected_df.iterrows():
        p_id = int(peak_row['peak_id'])
        rt_apex = float(peak_row['retention_time'])
        norm_area = peak_row.get('normalised area', 'N/A')
        peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])

        if 'peak_assignments' in globals() and p_id in peak_assignments:
            assign = peak_assignments[p_id]
            role = _extract_role(assign)
            r_style = ROLE_STYLES.get(role, {"bg": "#f3f4f6", "fg": "#374151", "border": "#d1d5db", "short": role})

            mol_images = [Draw.MolToImage(m, size=(185, 150)) for m in assign["mols"]]
            if len(mol_images) > 1:
                total_w = sum(im.width for im in mol_images)
                struct_img = Image.new('RGBA', (total_w, mol_images[0].height), (255, 255, 255, 255))
                x_off = 0
                for im in mol_images:
                    struct_img.paste(im, (x_off, 0))
                    x_off += im.width
            else:
                struct_img = mol_images[0]

            qc_struct_images[p_id] = struct_img

            buf = io.BytesIO()
            struct_img.save(buf, format='PNG')
            b64_png = base64.b64encode(buf.getvalue()).decode('utf-8')

            adduct_lines = "<br>".join([
                f"<b>{a['adduct']}</b>: {a['obs_mz']:.4f} m/z ({a['ppm']:+.1f} ppm)"
                for a in assign["adducts"]
            ])
            halogen_info = _detect_halogen_tag(assign.get("formula", ""))
            halogen_badge = (
                f"<div style='color:#7c3aed; font-weight:600; font-size:11px; margin-top:2px;'>"
                f"🔬 Isotope Signature: {halogen_info}</div>"
            ) if halogen_info else ""

            iso_badge = (
                f"<div style='color:#b45309; font-weight:600; font-size:11px; margin-top:2px;'>"
                f"⚠️ Isobaric Regioisomers / Byproducts ({assign['formula']})</div>"
            ) if assign.get("is_isobaric", False) else ""

            hover_cards_by_peak[p_id] = (
                f"<div style='border:2px solid {peak_color}; border-radius:10px; background:#ffffff; "
                f"padding:10px 12px; box-shadow:0 8px 20px rgba(0,0,0,0.18); font-family:sans-serif; "
                f"color:#111827; min-width:235px; max-width:400px; text-align:center;'>"
                f"<div style='margin-bottom:4px;'>"
                f"<span style='background:{r_style['bg']}; color:{r_style['fg']}; border:1px solid {r_style['border']}; "
                f"border-radius:5px; padding:2px 8px; font-size:10.5px; font-weight:700;'>{role}</span>"
                f"</div>"
                f"<div style='font-weight:700; font-size:13px; border-bottom:1px solid #e5e7eb; "
                f"padding-bottom:4px; margin-bottom:4px;'>"
                f"Peak {p_id} ({rt_apex:.1f}s / {rt_apex/60.0:.2f}m): {assign['display_name']}</div>"
                f"{iso_badge}{halogen_badge}"
                f"<div style='font-size:11.5px; color:#374151; line-height:1.45; margin-top:4px; margin-bottom:6px;'>"
                f"<b>Formula:</b> {assign.get('formula', 'N/A')}<br>"
                f"{adduct_lines}<br>"
                f"<b>TIC Area:</b> {norm_area} &nbsp;|&nbsp; "
                f"<b>Confidence:</b> <span style='color:#047857; font-weight:700;'>{assign['confidence']:.1f}%</span>"
                f"</div>"
                f"<img src='data:image/png;base64,{b64_png}' "
                f"style='max-width:100%; height:auto; display:block; margin:0 auto; border-radius:4px;'/>"
                f"</div>"
            )
        else:
            dom_str = str(peak_row.get('dominant m/z', 'None'))
            dom_tokens = [x.strip() for x in dom_str.split(',') if x.strip() and x.strip() != 'None']
            top_ions_str = "<br>".join(dom_tokens[:4]) if dom_tokens else "N/A"

            hover_cards_by_peak[p_id] = (
                f"<div style='border:2px solid {peak_color}; border-radius:8px; background:#f9fafb; "
                f"padding:8px 12px; box-shadow:0 6px 16px rgba(0,0,0,0.15); font-family:sans-serif; "
                f"color:#1f2937; min-width:195px; text-align:left;'>"
                f"<div style='font-weight:700; font-size:12.5px; border-bottom:1px solid #d1d5db; "
                f"padding-bottom:3px; margin-bottom:5px;'>"
                f"Peak {p_id} ({rt_apex:.1f}s) — Unassigned Impurity</div>"
                f"<div style='font-size:11px; color:#4b5563; line-height:1.4;'>"
                f"<b>Norm. Area:</b> {norm_area}<br>"
                f"<b>Dominant m/z:</b><br>{top_ions_str}</div>"
                f"</div>"
            )

    chrom_fig = go.Figure()

    # Raw & convolved traces
    chrom_fig.add_trace(go.Scatter(
        x=time_arr, y=raw_signal, mode='lines', name='raw crude TIC trace',
        line=dict(color='black', width=1.6),
        hovertemplate='<b>Raw Crude TIC</b><br>RT=%{x:.2f}s<br>Signal=%{y:.3e}<extra></extra>'
    ))
    if convolved is not None:
        chrom_fig.add_trace(go.Scatter(
            x=time_arr, y=convolved, mode='lines', name='fitted reaction mixture',
            line=dict(color='red', width=1.4, dash='dash'),
            hovertemplate='<b>Fitted Reaction Mixture</b><br>RT=%{x:.2f}s<br>Mixture=%{y:.3e}<extra></extra>'
        ))

    # Deconvoluted peak traces — tagged with peak_id in `meta` so the floating hover card triggers anywhere on the peak
    if chrom.peaks is not None:
        for g, _ in chrom.peaks.groupby('peak_id'):
            p_id = int(g)
            peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])
            component = chrom.unmixed_chromatograms[:, p_id - 1]
            legend_label = _peak_comparison_label(p_id)
            chrom_fig.add_trace(go.Scatter(
                x=time_arr, y=component, mode='lines', fill='tozeroy',
                line=dict(color=peak_color, width=1.0), opacity=0.55,
                name=legend_label,
                meta=p_id,
                hoverinfo='none'
            ))

    shapes, compact_annotations = [], []

    # Add subtle drop-lines, compact "P1 [Role]" badges, and interactive apex hover markers
    for _, peak_row in detected_df.iterrows():
        p_id = int(peak_row['peak_id'])
        rt_apex = float(peak_row['retention_time'])
        c_idx = int(np.argmin(np.abs(time_arr - rt_apex)))
        apex_y = float(raw_signal[c_idx])
        peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])

        shapes.append(dict(
            type='line', x0=rt_apex, x1=rt_apex, y0=0, y1=apex_y,
            line=dict(color='dodgerblue', width=1.1, dash='dot'), opacity=0.55
        ))

        if 'peak_assignments' in globals() and p_id in peak_assignments:
            role = _extract_role(peak_assignments[p_id])
            r_short = ROLE_STYLES.get(role, {"short": role})["short"]
            badge_label = f"<b>P{p_id} ({r_short})</b>"
        else:
            badge_label = f"<b>P{p_id}</b>"

        compact_annotations.append(dict(
            x=rt_apex, y=apex_y + y_max_plot * 0.035, xref='x', yref='y',
            text=badge_label,
            showarrow=False,
            font=dict(size=10, color='#111827'),
            bgcolor='rgba(255,255,255,0.90)',
            bordercolor=peak_color,
            borderwidth=1.5,
            borderpad=2
        ))

        chrom_fig.add_trace(go.Scatter(
            x=[rt_apex], y=[apex_y],
            mode='markers',
            marker=dict(size=9, color=peak_color, line=dict(width=1.5, color='white')),
            showlegend=False,
            meta=p_id,
            hoverinfo='none'
        ))

    chrom_fig.update_layout(
        shapes=shapes,
        annotations=compact_annotations,
        title=dict(
            text=(
                f"Crude Reaction Deconvolution & Byproduct Screening (Prominence: {prom_val:.4f}) "
                f"— Hover over any peak for 2D Structure, Role & Isotope Details"
            ),
            font=dict(size=14.5, color='#111827')
        ),
        xaxis=dict(
            title='Retention Time (seconds)',
            range=X_RANGE,
            rangeslider=dict(visible=True, thickness=0.10)
        ),
        yaxis=dict(
            title='Total Ion Count (baseline corrected)',
            range=[0, y_max_plot * 1.18]
        ),
        legend=dict(
            orientation='h',
            x=0.5, y=-0.26,
            xanchor='center', yanchor='top',
            font=dict(size=10),
            bgcolor='rgba(255,255,255,0.9)',
            bordercolor='#d1d5db',
            borderwidth=1
        ),
        margin=dict(l=MARGIN_L, r=MARGIN_R, t=MARGIN_T, b=MARGIN_B),
        template='plotly_white',
        width=FIG_WIDTH_PX,
        height=FIG_HEIGHT_PX,
        hovermode='closest'
    )

    # Render interactive HTML with custom floating hover card listener
    plot_div_id = "crude_rxn_interactive_chrom_div"
    fig_html = chrom_fig.to_html(include_plotlyjs='cdn', full_html=False, div_id=plot_div_id)
    cards_json = json.dumps({str(k): v for k, v in hover_cards_by_peak.items()})

    custom_hover_html = f"""
    <div style="position:relative; width:{FIG_WIDTH_PX}px;">
        {fig_html}
        <div id="{plot_div_id}_tooltip" style="display:none; position:fixed; pointer-events:none; z-index:99999; transition: opacity 0.08s ease-in-out;"></div>
    </div>
    <script>
    (function() {{
        var cards = {cards_json};
        var gd = document.getElementById('{plot_div_id}');
        var tooltip = document.getElementById('{plot_div_id}_tooltip');

        function attachHoverEvents() {{
            if (!gd || typeof gd.on !== 'function') {{
                setTimeout(attachHoverEvents, 150);
                return;
            }}
            gd.on('plotly_hover', function(data) {{
                if (!data || !data.points || data.points.length === 0) return;
                var pt = data.points[0];
                var p_id = pt.data.meta;
                if (p_id && cards[String(p_id)]) {{
                    tooltip.innerHTML = cards[String(p_id)];
                    tooltip.style.display = 'block';
                    if (data.event) {{
                        var x = data.event.clientX + 18;
                        var y = data.event.clientY - 30;
                        var rect = tooltip.getBoundingClientRect();
                        if (x + rect.width > window.innerWidth - 15) {{
                            x = data.event.clientX - rect.width - 18;
                        }}
                        if (y + rect.height > window.innerHeight - 15) {{
                            y = window.innerHeight - rect.height - 15;
                        }}
                        if (y < 10) y = 10;
                        tooltip.style.left = x + 'px';
                        tooltip.style.top = y + 'px';
                    }}
                }}
            }});
            gd.on('plotly_unhover', function() {{
                tooltip.style.display = 'none';
            }});
            gd.addEventListener('mousemove', function(e) {{
                if (tooltip.style.display === 'block') {{
                    var x = e.clientX + 18;
                    var y = e.clientY - 30;
                    var rect = tooltip.getBoundingClientRect();
                    if (x + rect.width > window.innerWidth - 15) {{
                        x = e.clientX - rect.width - 18;
                    }}
                    if (y + rect.height > window.innerHeight - 15) {{
                        y = window.innerHeight - rect.height - 15;
                    }}
                    if (y < 10) y = 10;
                    tooltip.style.left = x + 'px';
                    tooltip.style.top = y + 'px';
                }}
            }});
        }}
        attachHoverEvents();
    }})();
    </script>
    """
    display(HTML(custom_hover_html))

    # Cache a static PNG snapshot via kaleido for the Multimodal Vision QC cell
    try:
        qc_fig = go.Figure(chrom_fig)
        qc_images = []
        qc_annotations = list(compact_annotations)
        n_struct = max(len(qc_struct_images), 1)
        banner_xs = np.linspace(X_RANGE[0] + 12.0, X_RANGE[1] - 12.0, n_struct)
        for idx_p, (p_id, struct_img) in enumerate(sorted(qc_struct_images.items())):
            bx = float(banner_xs[idx_p])
            by = y_max_plot * 1.48
            assign = peak_assignments[p_id]
            role = _extract_role(assign)
            r_short = ROLE_STYLES.get(role, {"short": role})["short"]
            peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])
            qc_annotations.append(dict(
                x=bx, y=by + y_max_plot * 0.04, xref='x', yref='y',
                text=f"<b>P{p_id} [{r_short}]: {assign['display_name']}</b> ({assign['confidence']:.0f}%)",
                showarrow=False, font=dict(size=9, color='#111827'),
                bgcolor='white', bordercolor=peak_color, borderwidth=1.5
            ))
            qc_images.append(dict(
                source=struct_img, xref='x', yref='y',
                x=bx, y=by,
                sizex=max(12.0, (X_RANGE[1] - X_RANGE[0]) / (n_struct + 1.5)),
                sizey=y_max_plot * 0.32,
                xanchor='center', yanchor='top', layer='above'
            ))
        qc_fig.update_layout(
            yaxis=dict(title='Total Ion Count (baseline corrected)', range=[0, y_max_plot * 1.62]),
            annotations=qc_annotations,
            images=qc_images
        )
        chromatogram_png_bytes = qc_fig.to_image(format='png', scale=2)
        globals().setdefault('_notebook_figure_registry', {})['deconvoluted_chromatogram_with_structures'] = chromatogram_png_bytes
    except Exception as e:
        print(f"⚠️ Could not rasterize chromatogram for vision QC (kaleido missing?): {e}")

    globals()['_last_chromatogram_figure'] = chrom_fig

    # --- 3. Interactive Multi-Peak MS1 Stick-Spectrum & Halogen Isotope Comparison Widget ---
    ms_peak_selector = widgets.SelectMultiple(
        options=[(_peak_comparison_label(p_id), p_id) for p_id in sorted(peak_ms1_spectra.keys())],
        description='Compare Crude Peaks:',
        rows=min(7, max(4, len(peak_ms1_spectra))),
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='680px')
    )

    ms_output_area = widgets.Output()

    def build_ms_comparison_figure(selected_peak_ids):
        """Builds a stacked, zoom-linked Plotly figure overlaying the MS1 stick spectrum of
        every selected peak, including theoretical [M+H]+ / [M+Na]+ and M+2 halogen isotope markers."""
        mpl_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
        n = max(len(selected_peak_ids), 1)

        fig_ms = make_subplots(
            rows=n, cols=1, shared_xaxes=True,
            vertical_spacing=min(0.12, 0.6 / n) if n > 1 else 0.0,
            subplot_titles=[_peak_comparison_label(p) for p in selected_peak_ids]
        )

        for row_i, p_id in enumerate(selected_peak_ids, start=1):
            plotly_color_val = to_plotly_color(mpl_colors[(p_id - 1) % len(mpl_colors)])
            spectrum = peak_ms1_spectra.get(p_id, {})

            if spectrum:
                mzs = np.array(sorted(spectrum.keys()))
                intens = np.array([spectrum[m] for m in mzs])
                max_int = intens.max()
                rel = (intens / max_int * 100.0) if max_int > 0 else intens

                stick_x = np.repeat(mzs, 3)
                stick_y = np.zeros(stick_x.shape, dtype=float)
                stick_y[1::3] = rel
                stick_y[2::3] = np.nan

                fig_ms.add_trace(
                    go.Scatter(
                        x=stick_x, y=stick_y, mode='lines',
                        line=dict(color=plotly_color_val, width=1.3),
                        hoverinfo='skip', showlegend=False
                    ),
                    row=row_i, col=1
                )
                fig_ms.add_trace(
                    go.Scatter(
                        x=mzs, y=rel, mode='markers',
                        marker=dict(color=plotly_color_val, size=5),
                        customdata=intens, showlegend=False,
                        hovertemplate=(
                            f"Peak {p_id}<br>m/z=%{{x:.4f}}<br>rel. int.=%{{y:.1f}}%"
                            "<br>abs. int.=%{customdata:.3e}<extra></extra>"
                        )
                    ),
                    row=row_i, col=1
                )

                if p_id in peak_assignments:
                    assign_info = peak_assignments[p_id]
                    formula_str = assign_info.get("formula", "")
                    has_halogen = ("Br" in formula_str) or ("Cl" in formula_str)

                    for adduct in assign_info['adducts'][:3]:
                        fig_ms.add_vline(
                            x=adduct['calc_mz'],
                            line=dict(color=plotly_color_val, width=1.1, dash='dot'),
                            annotation_text=f"{adduct['adduct']} (M)",
                            annotation_font_size=8.5,
                            annotation_position="top right",
                            row=row_i, col=1
                        )
                        # Highlight M+2 halogen isotope peak (e.g. 81Br or 37Cl) for [M+H]+
                        if has_halogen and adduct['adduct'] == "[M+H]+":
                            m2_mz = adduct['calc_mz'] + (1.99795 if "Br" in formula_str else 1.99705)
                            hal_label = "⁸¹Br M+2" if "Br" in formula_str else "³⁷Cl M+2"
                            fig_ms.add_vline(
                                x=m2_mz,
                                line=dict(color='#7c3aed', width=1.1, dash='dash'),
                                annotation_text=hal_label,
                                annotation_font_size=8.5,
                                annotation_font_color='#7c3aed',
                                annotation_position="top left",
                                row=row_i, col=1
                            )
            else:
                fig_ms.add_annotation(
                    text="No MS1 signal captured in this peak's elution window",
                    x=0.5, y=0.5, xref=f'x{row_i} domain', yref=f'y{row_i} domain',
                    showarrow=False, font=dict(color='#9ca3af', size=11)
                )

            fig_ms.update_yaxes(title_text="Rel. Int. (%)", range=[0, 122], row=row_i, col=1)

        fig_ms.update_xaxes(matches='x', title_text="m/z", row=n, col=1)
        fig_ms.update_layout(
            width=FIG_WIDTH_PX,
            height=max(320, 260 * n),
            template='plotly_white',
            title_text="Crude Reaction Multi-Peak MS1 Comparison — Adducts, Byproduct Mass Shifts & Halogen (⁷⁹Br/⁸¹Br, ³⁵Cl/³⁷Cl) Isotope Doublets",
            margin=dict(t=90, b=50),
            hovermode='closest'
        )
        return fig_ms

    def render_ms_comparison(change=None):
        with ms_output_area:
            clear_output(wait=True)
            selected = list(ms_peak_selector.value)
            if not selected:
                print("Select one or more crude reaction peaks above to compare their MS1 spectra and isotope envelopes.")
                return
            build_ms_comparison_figure(selected).show()

    ms_peak_selector.observe(render_ms_comparison, names='value')

    display(widgets.HTML(
        "<b>🔬 Crude Reaction Multi-Peak MS1 & Halogen Isotope Comparison</b> — select 2+ peaks below to stack "
        "their MS1 stick spectra on a shared, zoomable m/z axis (pre-selecting Unreacted SM, Target Product, "
        "and primary Byproducts when available)."
    ))
    display(ms_peak_selector)
    display(ms_output_area)

    # Smart pre-selection: prioritize 1 Reactant (e.g. aryl bromide SM), 1 Product, and 1 Byproduct
    smart_preselect = []
    if 'peak_assignments' in globals() and peak_assignments:
        for target_role in ("Reactant", "Product", "Byproduct"):
            for p_id in sorted(peak_ms1_spectra.keys()):
                if p_id in peak_assignments and _extract_role(peak_assignments[p_id]) == target_role and p_id not in smart_preselect:
                    smart_preselect.append(p_id)
                    break
    for _, p_id in ms_peak_selector.options:
        if len(smart_preselect) >= 3:
            break
        if p_id not in smart_preselect:
            smart_preselect.append(p_id)

    ms_peak_selector.value = tuple(smart_preselect[:3])
    if ms_peak_selector.value:
        render_ms_comparison()

# Cell 6 (AI & Tool Dependencies + Core RDKit & MS1 Tools):

Installs google-genai, mcp, and rdkit to support the Gemini agent and chemistry tool-calling layer. Defines local analytical functions for the agent—including candidate/peak retrieval, conversion summaries, raw MS1 scan lookups, in-silico ESI+ fragmentation, halogen $M+2$ checks, and 2D SVG structure card rendering.   

In [ ]:
#@title 6. Install AI/RDKit Dependencies & Load Upgraded Crude Reaction Analytical Tools
!uv pip install --system -q google-genai mcp rdkit

import re
import math
import base64
import numpy as np
import pandas as pd
import scipy.signal
import pyopenms as po
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, Crippen
from rdkit.Chem.Draw import rdMolDraw2D

# 1. Silence noisy C++ stderr logs from RDKit so Colab output stays clean
RDLogger.DisableLog('rdApp.*')

PROTON_MASS = 1.00727647
ELECTRON_MASS = 0.00054858

# Preserve or initialize the global SVG registry so both [[MOL_IMG_X]] and {{SVG_X}} render seamlessly
_rendered_svg_registry = globals().setdefault('_rendered_svg_registry', {})

def _register_mol_svg(mol, label: str = "", width: int = 300, height: int = 220) -> str:
    """
    Renders an RDKit Mol object to a Base64-encoded SVG <img> HTML card,
    stores it in `_rendered_svg_registry` under both `[[MOL_IMG_N]]` and `{{SVG_N}}`,
    and returns the token `[[MOL_IMG_N]]`.
    """
    mol_copy = Chem.Mol(mol)
    AllChem.Compute2DCoords(mol_copy)

    drawer = rdMolDraw2D.MolDraw2DSVG(width, height)
    opts = drawer.drawOptions()
    opts.addStereoAnnotation = True
    opts.padding = 0.12

    exact_mass = Descriptors.ExactMolWt(mol_copy)
    formula = rdMolDescriptors.CalcMolFormula(mol_copy)
    f_charge = Chem.GetFormalCharge(mol_copy)
    mh_mz = exact_mass + PROTON_MASS if f_charge == 0 else exact_mass - (f_charge * ELECTRON_MASS)

    drawer.DrawMolecule(mol_copy)
    drawer.FinishDrawing()
    svg_str = drawer.GetDrawingText()

    b64_svg = base64.b64encode(svg_str.encode('utf-8')).decode('utf-8')
    mol_count = sum(1 for k in _rendered_svg_registry if k.startswith("[[MOL_IMG_")) + 1

    caption_title = f"<div style='font-weight:700; font-size:12px; color:#111827; margin-bottom:4px;'>{label}</div>" if label else ""
    caption_sub = f"<div style='font-size:11px; color:#374151; margin-top:4px;'><b>{formula}</b> | Exact: {exact_mass:.4f} | m/z: {mh_mz:.4f}</div>"

    html_card = (
        f"<div style='display:inline-block; margin:10px 6px; padding:10px 14px; border:2px solid #cbd5e1; "
        f"border-radius:10px; background:#ffffff; text-align:center; box-shadow:0 2px 6px rgba(0,0,0,0.15);'>"
        f"{caption_title}"
        f"<img src='data:image/svg+xml;base64,{b64_svg}' width='{width}' height='{height}' style='display:block; margin:0 auto; background:#ffffff;'/>"
        f"{caption_sub}"
        f"</div>"
    )

    token_primary = f"[[MOL_IMG_{mol_count}]]"
    token_alt = f"{{{{SVG_{mol_count}}}}}"
    _rendered_svg_registry[token_primary] = html_card
    _rendered_svg_registry[token_alt] = html_card
    return token_primary


def safe_mol_from_smiles(smiles_or_str: str):
    """
    Robust SMILES parser that automatically repairs common LLM kekulization mistakes
    (e.g., missing [nH] in pyrroles/indoles/imidazoles/benzimidazoles).
    Returns (mol, canonical_smiles, error_msg).
    """
    s = smiles_or_str.strip()
    if not s:
        return None, None, "Empty input string."

    if re.fullmatch(r'([A-Z][a-z]?\d*)+([+-]\d*)?', s) and not any(c in s for c in ['c', 'n', 'o', 's', '=', '#', '(', ')', '[']):
        if any(char.isdigit() for char in s) and not re.search(r'[1-9].*[1-9]', s[:3]):
            return None, None, (
                f"'{s}' is a molecular formula, not a SMILES string. "
                "Pass a valid SMILES string or use `analyze_structure_and_mass`."
            )

    mol = Chem.MolFromSmiles(s)
    if mol is not None:
        return mol, Chem.CanonSmiles(s), None

    mol_no_san = Chem.MolFromSmiles(s, sanitize=False)
    if mol_no_san is not None:
        try:
            mol_no_san.UpdatePropertyCache(strict=False)
            Chem.GetSymmSSSR(mol_no_san)
            for atom in mol_no_san.GetAtoms():
                if atom.GetAtomicNum() == 7 and atom.GetIsAromatic() and atom.GetDegree() == 2 and atom.GetFormalCharge() == 0 and atom.GetNumExplicitHs() == 0:
                    test_mol = Chem.Mol(mol_no_san)
                    test_mol.GetAtomWithIdx(atom.GetIdx()).SetNumExplicitHs(1)
                    try:
                        Chem.SanitizeMol(test_mol)
                        return test_mol, Chem.MolToSmiles(test_mol), None
                    except Exception:
                        continue
        except Exception:
            pass

    return None, None, (
        f"RDKit could not parse or kekulize '{s}'. "
        "Tip: In aromatic 5-membered N-heterocycles (indoles, pyrroles, imidazoles, triazoles), "
        "explicitly write '[nH]' for the protonated ring nitrogen."
    )


def _get_live_candidates_list() -> list:
    """Internal helper to fetch (name, role, smiles) tuples from Cell 3's table_container or fallback."""
    if 'table_container' in globals() and len(table_container.children) > 1:
        cands = []
        for row in table_container.children[1:]:
            name = row.children[1].value.strip()
            role = row.children[2].value
            smi = row.children[3].value.strip()
            if smi:
                cands.append((name, role, smi))
        if cands:
            return cands
    if 'get_candidates_from_table' in globals():
        return get_candidates_from_table()
    return []


def _normalize_role_label(assign_dict: dict) -> str:
    """Safely extracts synthetic role without substring collisions (e.g. 'Product' inside 'Byproduct')."""
    if not assign_dict:
        return "Unassigned"
    raw = assign_dict.get("role") or assign_dict.get("classification")
    if not raw:
        full_str = str(assign_dict.get("full_assignment", ""))
        for r_candidate in ("Byproduct", "Reactant", "Starting Material", "Reagent", "Target Product", "Product"):
            if f"({r_candidate})" in full_str:
                raw = r_candidate
                break
    if raw in ("Starting Material", "Reactant"):
        return "Reactant"
    if raw in ("Target Product", "Product"):
        return "Product"
    if raw in ("Byproduct", "Reagent"):
        return raw
    return "Unassigned"


def _trapz(y, x) -> float:
    return float(np.trapezoid(y, x) if hasattr(np, "trapezoid") else np.trapz(y, x))


# ==============================================================================
# CORE REACTION & PEAK SUMMARY TOOLS
# ==============================================================================

def get_user_candidate_molecules() -> str:
    """
    Returns the crude reaction table of Reactants, Reagents, Predicted Primary Products,
    and Predicted Byproducts along with SMILES, formulas, exact masses, [M+H]+, [M+Na]+,
    cLogP, TPSA, and expected halogen isotope signatures.
    """
    candidates = _get_live_candidates_list()
    if not candidates:
        return "No candidate reaction components found in table."

    rows = []
    for name, role, smi in candidates:
        mol, clean_smi, _ = safe_mol_from_smiles(smi)
        if mol:
            em = Descriptors.ExactMolWt(mol)
            formula = rdMolDescriptors.CalcMolFormula(mol)
            iso_sig = "79Br/81Br (1:1 M+2)" if "Br" in formula else ("35Cl/37Cl (3:1 M+2)" if "Cl" in formula else ("10B/11B (M-1 ~25%)" if "B" in formula else "Standard"))
            rows.append({
                "Name": name,
                "Role": role,
                "SMILES": clean_smi,
                "Formula": formula,
                "Exact_Mass": round(em, 4),
                "[M+H]+_mz": round(em + PROTON_MASS, 4),
                "[M+Na]+_mz": round(em + 22.989218, 4),
                "cLogP": round(Crippen.MolLogP(mol), 2),
                "TPSA": round(rdMolDescriptors.CalcTPSA(mol), 1),
                "Isotope_Signature": iso_sig
            })
    return pd.DataFrame(rows).to_markdown(index=False)


def get_detected_peaks_summary() -> str:
    """Returns a full tabular summary of all deconvoluted crude reaction HPLC-MS peaks, retention times, areas, roles, dominant m/z ions, and assignments."""
    if 'detected_df' not in globals():
        return "Error: detected_df not found in memory. Run Cell 4 first."

    cols = [
        c for c in [
            'peak_id', 'retention_time', 'peak start time', 'peak end time',
            'normalised area', 'Role', 'dominant m/z', 'Assignment', 'Confidence (%)'
        ]
        if c in detected_df.columns
    ]
    return detected_df[cols].to_markdown(index=False)


def calculate_crude_reaction_conversion_summary() -> dict:
    """
    Computes synthetic conversion, LC-MS TIC assay yield, unreacted starting material percentage,
    and byproduct distribution across all deconvoluted peaks in `detected_df`.
    """
    if 'detected_df' not in globals():
        return {"error": "detected_df not found in memory. Run Cell 4 first."}

    role_totals = {"Product": 0.0, "Reactant": 0.0, "Byproduct": 0.0, "Reagent": 0.0, "Unassigned": 0.0}
    peak_breakdown = []

    for _, row in detected_df.iterrows():
        p_id = int(row['peak_id'])
        rt = float(row['retention_time'])
        area_str = str(row.get('normalised area', '0%')).replace('%', '').strip()
        try:
            area_pct = float(area_str)
        except ValueError:
            area_pct = 0.0

        role = "Unassigned"
        name = "Unassigned Impurity"
        if 'peak_assignments' in globals() and p_id in peak_assignments:
            assign = peak_assignments[p_id]
            name = assign.get("display_name", "Assigned")
            role = _normalize_role_label(assign)
        elif 'Role' in row and str(row['Role']) in role_totals:
            role = str(row['Role'])

        role_totals[role] = round(role_totals.get(role, 0.0) + area_pct, 2)
        peak_breakdown.append({
            "peak_id": p_id,
            "rt_sec": round(rt, 1),
            "role": role,
            "compound": name,
            "tic_area_pct": round(area_pct, 2)
        })

    rxn_mass_balance = role_totals["Product"] + role_totals["Reactant"] + role_totals["Byproduct"]
    conversion_pct = round(((role_totals["Product"] + role_totals["Byproduct"]) / rxn_mass_balance) * 100.0, 2) if rxn_mass_balance > 0 else 0.0
    selectivity_pct = round((role_totals["Product"] / rxn_mass_balance) * 100.0, 2) if rxn_mass_balance > 0 else 0.0

    return {
        "role_area_percentages": role_totals,
        "estimated_sm_conversion_pct": conversion_pct,
        "estimated_product_selectivity_among_reacting_species_pct": selectivity_pct,
        "peak_by_peak_breakdown": peak_breakdown,
        "analytical_note": (
            "TIC area percentages reflect ESI+ ionization efficiency as well as molar abundance "
            "(e.g., triphenylphosphine oxide and basic amines ionize strongly in ESI+)."
        )
    }


def get_peak_details(peak_id: int) -> dict:
    """Retrieves detailed analytical evidence, observed adducts, halogen M+2 checks, and raw MS1 ions for a specific peak ID."""
    result = {}
    if 'detected_df' in globals():
        row = detected_df[detected_df['peak_id'] == peak_id]
        if not row.empty:
            r_dict = row.iloc[0].to_dict()
            result["chromatogram_row"] = {k: str(v) for k, v in r_dict.items()}
            rt_start = r_dict.get('peak start time', np.nan)
            rt_end = r_dict.get('peak end time', np.nan)
            if 'od' in globals() and not pd.isna(rt_start) and not pd.isna(rt_end):
                agg = {}
                for spec in od:
                    if spec.getMSLevel() == 1 and rt_start <= spec.getRT() <= rt_end:
                        mzs, ints = spec.get_peaks()
                        for m, inten in zip(np.round(mzs, 4), ints):
                            if inten > 1e3:
                                agg[m] = agg.get(m, 0.0) + float(inten)
                if agg:
                    top_ions = sorted(agg.items(), key=lambda x: x[1], reverse=True)[:25]
                    max_i = top_ions[0][1]
                    result["raw_ms1_top_25_ions_in_peak_window"] = [
                        {"mz": float(m), "intensity": round(i, 1), "rel_pct": round((i / max_i) * 100, 2)}
                        for m, i in top_ions
                    ]

    if 'peak_assignments' in globals() and peak_id in peak_assignments:
        assign = peak_assignments[peak_id]
        result["assigned_compound"] = {
            "display_name": assign["display_name"],
            "role": _normalize_role_label(assign),
            "full_assignment": assign.get("full_assignment", assign["display_name"]),
            "formula": assign["formula"],
            "confidence": assign["confidence"],
            "evidence": assign["evidence"],
            "adducts_observed": [
                f"{a['adduct']} at {a['obs_mz']:.4f} m/z ({a['ppm']:+.1f} ppm, intensity: {a['intensity']:.1e})"
                for a in assign["adducts"]
            ]
        }
    return result if result else {"error": f"Peak {peak_id} not found."}


def check_mz_in_peak(peak_id: int, target_mz: float, tolerance_da: float = 0.05) -> dict:
    """
    Directly searches all raw MS1 scans across a specific peak's retention time window
    to verify whether a specific m/z (e.g. an adduct, byproduct, or M+2 halogen isotope ion) is present.
    """
    if 'detected_df' not in globals() or 'od' not in globals():
        return {"error": "detected_df or pyOpenMS 'od' experiment not found in memory."}

    row = detected_df[detected_df['peak_id'] == peak_id]
    if row.empty:
        return {"error": f"Peak {peak_id} not found in detected_df."}

    rt_start = float(row.iloc[0]['peak start time'])
    rt_end = float(row.iloc[0]['peak end time'])
    dom_str = str(row.iloc[0]['dominant m/z'])

    matches = []
    for spec in od:
        if spec.getMSLevel() == 1 and rt_start <= spec.getRT() <= rt_end:
            mzs, ints = spec.get_peaks()
            mask = (mzs >= target_mz - tolerance_da) & (mzs <= target_mz + tolerance_da)
            if np.any(mask):
                for m, inten in zip(mzs[mask], ints[mask]):
                    matches.append({
                        "rt": round(float(spec.getRT()), 2),
                        "observed_mz": round(float(m), 4),
                        "error_da": round(float(m - target_mz), 4),
                        "ppm_error": round(float((m - target_mz) / target_mz * 1e6), 1),
                        "intensity": round(float(inten), 1)
                    })

    matches = sorted(matches, key=lambda x: x["intensity"], reverse=True)
    return {
        "peak_id": peak_id,
        "rt_window": f"{rt_start:.1f}s - {rt_end:.1f}s",
        "dominant_mz_summary": dom_str,
        "target_mz_searched": target_mz,
        "tolerance_da": tolerance_da,
        "found_in_raw_scans": len(matches) > 0,
        "max_observed_hit": matches[0] if matches else None,
        "top_5_scan_hits": matches[:5]
    }


# ==============================================================================
# UPGRADED CHROMATOGRAPHIC & MASS-SPECTRAL DIAGNOSTIC TOOLS (6 NEW CAPABILITIES)
# ==============================================================================

def find_mz_across_chromatogram(
    target_mz: float = 0.0,
    smiles_or_formula: str = "",
    adduct: str = "[M+H]+",
    ppm_tolerance: float = 15.0
) -> dict:
    """
    Full-run Extracted Ion Chromatogram (EIC) scanner across ALL MS1 scans in `od`.
    Finds trace starting materials, reagents (e.g. TPPO), dimers, or low-abundance byproducts
    even if they fell below Cell 4's TIC peak-detection threshold.
    Provide either `target_mz` (>0) OR `smiles_or_formula` with an `adduct` ('[M+H]+', '[M+Na]+', '[M+NH4]+', '[M+K]+', '[M]+').
    """
    if 'od' not in globals() or od.size() == 0:
        return {"error": "pyOpenMS experiment 'od' not loaded. Run Cell 2 first."}

    adduct_shifts = {
        "[M+H]+": PROTON_MASS,
        "[M+Na]+": 22.989218,
        "[M+NH4]+": 18.033823,
        "[M+K]+": 38.963158,
        "[M]+": -ELECTRON_MASS,
        "[M-H]-": -PROTON_MASS,
    }
    resolved_label = ""
    if target_mz <= 0.0 and smiles_or_formula.strip():
        mol, clean_smi, _ = safe_mol_from_smiles(smiles_or_formula)
        if mol is not None:
            em = Descriptors.ExactMolWt(mol)
            resolved_label = f"{rdMolDescriptors.CalcMolFormula(mol)} ({clean_smi})"
        else:
            try:
                em = rdMolDescriptors.CalcExactMolWt(smiles_or_formula.strip())
                resolved_label = smiles_or_formula.strip()
            except Exception:
                return {"error": f"Could not parse '{smiles_or_formula}' as SMILES or molecular formula."}
        shift = adduct_shifts.get(adduct.strip(), PROTON_MASS)
        target_mz = em + shift

    if target_mz <= 0.0:
        return {"error": "Provide a positive `target_mz` or a valid `smiles_or_formula`."}

    tol_da = max(0.015, target_mz * ppm_tolerance * 1e-6)
    rts, eic_vals, obs_mzs = [], [], []
    for spec in od:
        if spec.getMSLevel() != 1:
            continue
        rts.append(float(spec.getRT()))
        mzs, ints = spec.get_peaks()
        mask = (mzs >= target_mz - tol_da) & (mzs <= target_mz + tol_da)
        if np.any(mask):
            eic_vals.append(float(np.sum(ints[mask])))
            best_i = int(np.argmax(ints[mask]))
            obs_mzs.append(float(mzs[mask][best_i]))
        else:
            eic_vals.append(0.0)
            obs_mzs.append(np.nan)

    order = np.argsort(rts)
    rts_arr = np.array(rts, dtype=float)[order]
    eic_arr = np.array(eic_vals, dtype=float)[order]
    obs_arr = np.array(obs_mzs, dtype=float)[order]

    max_eic = float(eic_arr.max()) if len(eic_arr) else 0.0
    if max_eic <= 0.0:
        return {
            "target_mz": round(target_mz, 4),
            "query_species": resolved_label or "Custom m/z",
            "found_in_run": False,
            "note": f"No MS1 signal within ±{tol_da:.4f} Da ({ppm_tolerance} ppm) across the chromatogram."
        }

    nonzero = eic_arr[eic_arr > 0]
    baseline_est = float(np.percentile(eic_arr, 20))
    noise_est = float(np.median(np.abs(np.diff(eic_arr)))) + 1.0
    is_persistent_bg = (len(nonzero) / max(len(eic_arr), 1) > 0.70) and (float(np.median(eic_arr)) > 0.25 * max_eic)

    eic_norm = (eic_arr - eic_arr.min()) / (max_eic - eic_arr.min() + 1e-12)
    peak_idxs, _ = scipy.signal.find_peaks(eic_norm, prominence=0.10, distance=3)
    if len(peak_idxs) == 0 and max_eic > 0:
        peak_idxs = np.array([int(np.argmax(eic_arr))])

    eic_hits = []
    for idx in sorted(peak_idxs, key=lambda i: -eic_arr[i])[:6]:
        rt_p = float(rts_arr[idx])
        obs_m = float(obs_arr[idx]) if not np.isnan(obs_arr[idx]) else target_mz
        ppm_err = ((obs_m - target_mz) / target_mz) * 1e6
        snr = (float(eic_arr[idx]) - baseline_est) / noise_est
        lo_i = max(0, idx - 6)
        hi_i = min(len(eic_arr), idx + 7)
        local_area = _trapz(eic_arr[lo_i:hi_i], rts_arr[lo_i:hi_i])

        overlapping_peaks = []
        if 'detected_df' in globals() and len(detected_df) > 0:
            for _, prow in detected_df.iterrows():
                if float(prow['peak start time']) <= rt_p <= float(prow['peak end time']):
                    overlapping_peaks.append({
                        "peak_id": int(prow['peak_id']),
                        "tic_apex_rt_sec": round(float(prow['retention_time']), 2),
                        "current_assignment": str(prow.get('Assignment', 'Unassigned'))
                    })

        eic_hits.append({
            "eic_apex_rt_sec": round(rt_p, 2),
            "eic_apex_rt_min": round(rt_p / 60.0, 3),
            "observed_mz": round(obs_m, 4),
            "ppm_error": round(ppm_err, 2),
            "apex_intensity": round(float(eic_arr[idx]), 1),
            "estimated_snr": round(float(snr), 1),
            "local_eic_area": round(local_area, 1),
            "overlaps_detected_tic_peaks": overlapping_peaks
        })

    return {
        "target_mz": round(target_mz, 4),
        "adduct": adduct,
        "query_species": resolved_label or "Custom m/z",
        "found_in_run": True,
        "appears_persistent_background_ion": is_persistent_bg,
        "max_eic_intensity": round(max_eic, 1),
        "eic_elution_apices": eic_hits
    }


def check_peak_purity_and_coelution(peak_id: int, top_n_ions: int = 8) -> dict:
    """
    Audits peak purity for `peak_id` by splitting its elution window into leading edge (front 30%),
    apex (middle 40%), and trailing edge (back 30%), and computing EIC Pearson correlations & apex RT shifts
    for the top MS1 ions. Detects hidden co-eluting impurities or unresolved isomers.
    """
    if 'detected_df' not in globals() or 'od' not in globals():
        return {"error": "detected_df or pyOpenMS 'od' experiment not found. Run Cells 2 and 4 first."}

    row = detected_df[detected_df['peak_id'] == peak_id]
    if row.empty:
        return {"error": f"Peak {peak_id} not found in detected_df."}

    rt_start = float(row.iloc[0]['peak start time'])
    rt_end = float(row.iloc[0]['peak end time'])
    rt_apex = float(row.iloc[0]['retention_time'])

    scans = []
    for spec in od:
        if spec.getMSLevel() == 1 and rt_start <= spec.getRT() <= rt_end:
            mzs, ints = spec.get_peaks()
            scans.append((float(spec.getRT()), np.asarray(mzs, dtype=float), np.asarray(ints, dtype=float)))
    scans.sort(key=lambda s: s[0])
    if len(scans) < 3:
        return {"error": f"Too few MS1 scans ({len(scans)}) in Peak {peak_id} window ({rt_start:.1f}–{rt_end:.1f} s) for purity analysis."}

    scan_rts = np.array([s[0] for s in scans])
    dt_local = float(np.median(np.diff(scan_rts))) if len(scan_rts) > 1 else 1.0

    # Aggregate ions across whole peak and across Front / Apex / Tail slices
    n_scans = len(scans)
    i_front = max(1, int(round(0.30 * n_scans)))
    i_tail = min(n_scans - 1, int(round(0.70 * n_scans)))
    slices = {
        "leading_edge_front_30pct": scans[:i_front],
        "apex_middle_40pct": scans[i_front:i_tail] if i_tail > i_front else scans[i_front - 1:i_front + 1],
        "trailing_edge_back_30pct": scans[i_tail:]
    }

    def _top_slice_ions(slice_scans, k=5):
        agg = {}
        for _, mzs, ints in slice_scans:
            for m, inten in zip(np.round(mzs, 3), ints):
                if inten > 0:
                    agg[m] = agg.get(m, 0.0) + float(inten)
        if not agg:
            return []
        ordered = sorted(agg.items(), key=lambda x: -x[1])[:k]
        mx = ordered[0][1]
        return [{"mz": float(m), "rel_pct": round(v / mx * 100.0, 1)} for m, v in ordered]

    slice_summary = {k: _top_slice_ions(v, 5) for k, v in slices.items()}

    # Select top_n_ions overall
    total_agg = {}
    for _, mzs, ints in scans:
        for m, inten in zip(np.round(mzs, 3), ints):
            total_agg[m] = total_agg.get(m, 0.0) + float(inten)
    top_ions = sorted(total_agg.items(), key=lambda x: -x[1])[:max(3, min(int(top_n_ions), 12))]
    if not top_ions:
        return {"error": f"No MS1 ions found in Peak {peak_id}."}

    base_mz = float(top_ions[0][0])
    eics = {}
    for m, _ in top_ions:
        tol = max(0.015, m * 20e-6)
        eics[m] = np.array([float(np.sum(ints[np.abs(mzs - m) <= tol])) for _, mzs, ints in scans])

    base_vec = eics[base_mz]
    base_apex_rt = float(scan_rts[int(np.argmax(base_vec))])
    shift_flag_thresh = max(1.5 * dt_local, 1.5)

    ion_profiles = []
    coeluting_flags = []
    for m, tot_i in top_ions:
        vec = eics[m]
        ion_apex_rt = float(scan_rts[int(np.argmax(vec))])
        rt_shift = ion_apex_rt - base_apex_rt
        if np.std(base_vec) > 1e-9 and np.std(vec) > 1e-9:
            corr_r = float(np.corrcoef(base_vec, vec)[0, 1])
        else:
            corr_r = 1.0 if m == base_mz else 0.0

        # Identify obvious isotopologues of the base ion (+1.003, +1.998, -0.996)
        dm = m - base_mz
        is_likely_isotope = any(abs(dm - shift) <= 0.025 for shift in (1.0034, 1.9980, 2.0067, -0.9964))
        is_coeluter = (m != base_mz) and (not is_likely_isotope) and ((corr_r < 0.85) or (abs(rt_shift) >= shift_flag_thresh))

        entry = {
            "mz": round(float(m), 3),
            "rel_total_pct": round(tot_i / top_ions[0][1] * 100.0, 1),
            "eic_apex_rt_sec": round(ion_apex_rt, 2),
            "apex_shift_vs_base_sec": round(rt_shift, 2),
            "pearson_r_vs_base": round(corr_r, 3),
            "likely_base_isotopologue": is_likely_isotope,
            "flagged_coeluting_species": is_coeluter
        }
        ion_profiles.append(entry)
        if is_coeluter:
            side = "leading edge (elutes earlier)" if rt_shift < 0 else "trailing edge (elutes later)"
            coeluting_flags.append(f"m/z {m:.3f} (r={corr_r:.2f}, ΔRT={rt_shift:+.2f} s on {side})")

    purity_verdict = "IMPURE / CO-ELUTING SPECIES DETECTED" if coeluting_flags else "SPECTRA HOMOGENEOUS (PURE PEAK)"
    return {
        "peak_id": peak_id,
        "rt_window_sec": [round(rt_start, 2), round(rt_end, 2)],
        "tic_apex_rt_sec": round(rt_apex, 2),
        "base_ion_mz": round(base_mz, 3),
        "purity_verdict": purity_verdict,
        "coeluting_alerts": coeluting_flags,
        "ion_eic_correlations": ion_profiles,
        "front_apex_tail_spectra": slice_summary
    }


def verify_isotope_envelope_in_peak(
    peak_id: int,
    smiles_or_formula: str,
    adduct: str = "[M+H]+",
    ppm_tolerance: float = 20.0
) -> dict:
    """
    Quantitatively compares the theoretical isotope envelope (M-1 for 10B, M, M+1 for 13C,
    M+2 for 81Br/37Cl/34S, M+4 for dihalogens) generated by pyOpenMS against the observed MS1
    apex spectrum in `peak_id`. Returns cosine similarity, per-peak intensity errors, and ppm errors.
    """
    if 'detected_df' not in globals() or 'od' not in globals():
        return {"error": "detected_df or pyOpenMS 'od' experiment not found."}

    row = detected_df[detected_df['peak_id'] == peak_id]
    if row.empty:
        return {"error": f"Peak {peak_id} not found in detected_df."}

    mol, _, _ = safe_mol_from_smiles(smiles_or_formula)
    formula_str = rdMolDescriptors.CalcMolFormula(mol) if mol is not None else smiles_or_formula.strip()

    adduct_formulas = {
        "[M+H]+": "H",
        "[M+Na]+": "Na",
        "[M+NH4]+": "NH4",
        "[M+K]+": "K",
        "[M]+": "",
    }
    if adduct not in adduct_formulas:
        return {"error": f"Unsupported adduct '{adduct}'. Use one of {list(adduct_formulas.keys())}."}

    try:
        ef = po.EmpiricalFormula(formula_str)
        if adduct_formulas[adduct]:
            ef = ef + po.EmpiricalFormula(adduct_formulas[adduct])
        iso_gen = po.CoarseIsotopePatternGenerator(6)
        iso_dist = ef.getIsotopeDistribution(iso_gen)
    except Exception as e:
        return {"error": f"Could not generate isotope distribution for '{formula_str}' + {adduct}: {e}"}

    raw_theo = [(float(p.getMZ()) - ELECTRON_MASS, float(p.getIntensity())) for p in iso_dist.getContainer()]
    if not raw_theo:
        return {"error": "Empty theoretical isotope distribution."}

    max_theo_int = max(x[1] for x in raw_theo)
    base_theo_mz = max(raw_theo, key=lambda x: x[1])[0]
    theo_peaks = [
        {
            "offset_label": f"M{int(round(mz - base_theo_mz)):+d}" if int(round(mz - base_theo_mz)) != 0 else "M (Base)",
            "theo_mz": mz,
            "theo_rel_pct": (inten / max_theo_int) * 100.0
        }
        for mz, inten in raw_theo
        if (inten / max_theo_int) * 100.0 >= 1.0
    ]

    # Collect apex scans (closest 5 scans around apex within peak window)
    rt_start = float(row.iloc[0]['peak start time'])
    rt_end = float(row.iloc[0]['peak end time'])
    rt_apex = float(row.iloc[0]['retention_time'])
    window_scans = [s for s in od if s.getMSLevel() == 1 and rt_start <= s.getRT() <= rt_end]
    if not window_scans:
        return {"error": f"No MS1 scans found in Peak {peak_id} window."}
    apex_scans = sorted(window_scans, key=lambda s: abs(s.getRT() - rt_apex))[:5]

    obs_ints, obs_mzs = [], []
    for tp in theo_peaks:
        target = tp["theo_mz"]
        tol_da = max(0.018, target * ppm_tolerance * 1e-6)
        total_i = 0.0
        weighted_mz = 0.0
        for s in apex_scans:
            mzs, ints = s.get_peaks()
            mask = (mzs >= target - tol_da) & (mzs <= target + tol_da)
            if np.any(mask):
                w = ints[mask]
                total_i += float(np.sum(w))
                weighted_mz += float(np.sum(mzs[mask] * w))
        obs_ints.append(total_i)
        obs_mzs.append((weighted_mz / total_i) if total_i > 0 else np.nan)

    max_obs = max(obs_ints) if obs_ints else 0.0
    theo_vec = np.array([tp["theo_rel_pct"] for tp in theo_peaks], dtype=float)
    obs_vec = np.array([(v / max_obs * 100.0) if max_obs > 0 else 0.0 for v in obs_ints], dtype=float)

    if max_obs > 0 and np.linalg.norm(theo_vec) > 0 and np.linalg.norm(obs_vec) > 0:
        cosine_sim = float(np.dot(theo_vec, obs_vec) / (np.linalg.norm(theo_vec) * np.linalg.norm(obs_vec)))
    else:
        cosine_sim = 0.0

    comparison_rows = []
    ppm_errors = []
    for tp, o_rel, o_mz, o_abs in zip(theo_peaks, obs_vec, obs_mzs, obs_ints):
        ppm_e = ((o_mz - tp["theo_mz"]) / tp["theo_mz"] * 1e6) if not np.isnan(o_mz) else None
        if ppm_e is not None:
            ppm_errors.append(abs(ppm_e))
        comparison_rows.append({
            "isotope": tp["offset_label"],
            "theo_mz": round(tp["theo_mz"], 4),
            "obs_mz": round(float(o_mz), 4) if not np.isnan(o_mz) else None,
            "ppm_error": round(ppm_e, 2) if ppm_e is not None else None,
            "theo_rel_pct": round(tp["theo_rel_pct"], 1),
            "obs_rel_pct": round(float(o_rel), 1),
            "rel_intensity_diff_pct": round(float(o_rel - tp["theo_rel_pct"]), 1),
            "detected": o_abs > 0
        })

    verdict = (
        "STRONG ISOTOPE MATCH" if cosine_sim >= 0.96 and max_obs > 0
        else ("MODERATE / PARTIAL MATCH" if cosine_sim >= 0.85 else "POOR ISOTOPE MATCH (LIKELY WRONG FORMULA/HALOGEN STATE)")
    )

    return {
        "peak_id": peak_id,
        "formula": formula_str,
        "adduct": adduct,
        "isotope_cosine_similarity": round(cosine_sim, 4),
        "mean_abs_ppm_error": round(float(np.mean(ppm_errors)), 2) if ppm_errors else None,
        "verdict": verdict,
        "envelope_comparison": comparison_rows
    }


def analyze_chromatographic_resolution_and_shape(peak_id_1: int = 0, peak_id_2: int = 0) -> dict:
    """
    Computes quantitative chromatographic quality & purification metrics directly from
    `chrom.unmixed_chromatograms`: FWHM, USP tailing factor (Tf at 5% height), theoretical plates (N),
    and pairwise USP resolution (Rs) between adjacent peaks or between two specified peak IDs.
    """
    if 'detected_df' not in globals() or 'chrom' not in globals() or chrom is None:
        return {"error": "detected_df or chrom not found. Run Cell 4 first."}

    t_arr = chrom.df[chrom.time_col].to_numpy(dtype=float)
    unmixed = getattr(chrom, 'unmixed_chromatograms', None)
    if unmixed is None or unmixed.shape[1] == 0:
        return {"error": "No deconvoluted component curves available in chrom.unmixed_chromatograms."}

    peak_metrics = {}
    for idx_row, row in detected_df.iterrows():
        p_id = int(row['peak_id'])
        if idx_row >= unmixed.shape[1]:
            continue
        comp = unmixed[:, idx_row]
        y_max = float(comp.max())
        if y_max <= 0:
            continue
        apex_idx = int(np.argmax(comp))
        rt_apex = float(t_arr[apex_idx])

        def _crossing_bounds(frac):
            above = np.where(comp >= frac * y_max)[0]
            if len(above) < 2:
                return rt_apex, rt_apex
            i_l, i_r = int(above[0]), int(above[-1])
            # Linear interpolation for sub-scan accuracy
            if i_l > 0 and comp[i_l] != comp[i_l - 1]:
                t_l = t_arr[i_l - 1] + (frac * y_max - comp[i_l - 1]) / (comp[i_l] - comp[i_l - 1]) * (t_arr[i_l] - t_arr[i_l - 1])
            else:
                t_l = t_arr[i_l]
            if i_r < len(t_arr) - 1 and comp[i_r + 1] != comp[i_r]:
                t_r = t_arr[i_r] + (frac * y_max - comp[i_r]) / (comp[i_r + 1] - comp[i_r]) * (t_arr[i_r + 1] - t_arr[i_r])
            else:
                t_r = t_arr[i_r]
            return float(t_l), float(t_r)

        l50, r50 = _crossing_bounds(0.50)
        l05, r05 = _crossing_bounds(0.05)
        fwhm = max(r50 - l50, 1e-4)
        w05 = max(r05 - l05, 1e-4)
        front_05 = max(rt_apex - l05, 1e-4)
        usp_tailing = w05 / (2.0 * front_05)
        plates_n = 5.54 * ((rt_apex / fwhm) ** 2)

        shape_flag = (
            "Symmetric" if 0.85 <= usp_tailing <= 1.25
            else ("Tailing" if usp_tailing > 1.25 else "Fronting")
        )
        assign_name = peak_assignments[p_id]["display_name"] if ('peak_assignments' in globals() and p_id in peak_assignments) else "Unassigned"
        role = _normalize_role_label(peak_assignments.get(p_id)) if 'peak_assignments' in globals() else str(row.get('Role', 'Unassigned'))

        peak_metrics[p_id] = {
            "peak_id": p_id,
            "role": role,
            "compound": assign_name,
            "rt_sec": round(rt_apex, 2),
            "fwhm_sec": round(fwhm, 2),
            "usp_tailing_factor_Tf": round(usp_tailing, 2),
            "shape_assessment": shape_flag,
            "theoretical_plates_N": int(round(plates_n))
        }

    def _pair_res(p1, p2):
        m1, m2 = peak_metrics[p1], peak_metrics[p2]
        rs = 1.18 * abs(m2["rt_sec"] - m1["rt_sec"]) / max(m1["fwhm_sec"] + m2["fwhm_sec"], 1e-4)
        feasibility = (
            "Baseline resolved (Rs >= 1.5) — straightforward flash or prep-HPLC isolation" if rs >= 1.5
            else ("Partial overlap (1.0 <= Rs < 1.5) — use shallow-gradient C18 prep-HPLC" if rs >= 1.0
                  else "Co-eluting / severe overlap (Rs < 1.0) — requires method redevelopment or orthogonal stationary phase")
        )
        return {
            "pair": f"Peak {p1} ({m1['compound']}) vs Peak {p2} ({m2['compound']})",
            "delta_rt_sec": round(abs(m2["rt_sec"] - m1["rt_sec"]), 2),
            "usp_resolution_Rs": round(rs, 2),
            "purification_guidance": feasibility
        }

    ordered_ids = sorted(peak_metrics.keys(), key=lambda pid: peak_metrics[pid]["rt_sec"])
    pairwise_resolutions = []
    if peak_id_1 > 0 and peak_id_2 > 0:
        if peak_id_1 in peak_metrics and peak_id_2 in peak_metrics:
            pairwise_resolutions.append(_pair_res(peak_id_1, peak_id_2))
        else:
            return {"error": f"Requested peak IDs ({peak_id_1}, {peak_id_2}) not found in deconvoluted peaks."}
    else:
        for i in range(len(ordered_ids) - 1):
            pairwise_resolutions.append(_pair_res(ordered_ids[i], ordered_ids[i + 1]))

    return {
        "per_peak_shape_metrics": [peak_metrics[pid] for pid in ordered_ids],
        "adjacent_or_requested_resolutions": pairwise_resolutions
    }


def elucidate_unknown_formula_from_mz(
    observed_mz: float,
    adduct: str = "[M+H]+",
    ppm_tolerance: float = 15.0,
    peak_id: int = 0,
    max_results: int = 10
) -> dict:
    """
    Reaction-constrained exact-mass formula generator for unknown impurities.
    Automatically constrains allowed elements from Cell 3's Reactants & Reagents,
    filters by Nitrogen Rule & integer RDBE, and (if `peak_id > 0`) ranks candidates
    against observed M+1 (13C carbon count) and M+2 (Br/Cl) isotope ratios in that peak.
    """
    adduct_shifts = {
        "[M+H]+": PROTON_MASS,
        "[M+Na]+": 22.989218,
        "[M+NH4]+": 18.033823,
        "[M+K]+": 38.963158,
        "[M]+": -ELECTRON_MASS,
        "[M-H]-": -PROTON_MASS,
    }
    if adduct not in adduct_shifts:
        return {"error": f"Unsupported adduct '{adduct}'. Choose from {list(adduct_shifts.keys())}."}

    neutral_mass = observed_mz - adduct_shifts[adduct]
    if neutral_mass < 25.0 or neutral_mass > 1200.0:
        return {"error": f"Neutral mass {neutral_mass:.4f} Da is outside supported small-molecule range (25–1200 Da)."}

    tol_da = max(0.003, observed_mz * ppm_tolerance * 1e-6)

    # Exact monoisotopic masses & valences for common organic/medchem elements
    elem_mass = {
        "C": 12.000000, "H": 1.00782503, "N": 14.00307400, "O": 15.99491462,
        "F": 18.99840316, "P": 30.97376199, "S": 31.97207117, "Cl": 34.96885268,
        "Br": 78.9183376, "I": 126.9044719, "B": 11.00930536, "Si": 27.97692653
    }

    # Infer reaction-constrained element caps from Cell 3 reactants & reagents
    rxn_max = {"N": 0, "O": 0, "F": 0, "P": 0, "S": 0, "Cl": 0, "Br": 0, "I": 0, "B": 0, "Si": 0}
    for _, _, smi in _get_live_candidates_list():
        mol = Chem.MolFromSmiles(smi)
        if not mol:
            continue
        counts = {}
        for atom in mol.GetAtoms():
            sym = atom.GetSymbol()
            counts[sym] = counts.get(sym, 0) + 1
        for sym in rxn_max:
            rxn_max[sym] = max(rxn_max[sym], counts.get(sym, 0))

    # Always allow O (oxidation/hydrolysis) and N; allow other heteroatoms if present in the reaction
    het_bounds = []
    het_bounds.append(("O", min(10, max(rxn_max["O"] * 2 + 2, 5))))
    het_bounds.append(("N", min(8, max(rxn_max["N"] * 2, 2))))
    for sym, hard_cap in [("Br", 3), ("Cl", 3), ("B", 2), ("P", 2), ("S", 3), ("F", 6), ("I", 2), ("Si", 2)]:
        if rxn_max[sym] > 0:
            het_bounds.append((sym, min(hard_cap, max(rxn_max[sym] * 2, 2))))

    # Optional observed M+1 and M+2 ratios from peak_id to rank formulas
    obs_m1_pct, obs_m2_pct = None, None
    if peak_id > 0 and 'peak_ms1_spectra' in globals() and peak_id in peak_ms1_spectra:
        spec_dict = peak_ms1_spectra[peak_id]
        m0_i = max([v for k, v in spec_dict.items() if abs(k - observed_mz) <= 0.02], default=0.0)
        if m0_i > 0:
            m1_i = max([v for k, v in spec_dict.items() if abs(k - (observed_mz + 1.0034)) <= 0.022], default=0.0)
            m2_i = max([v for k, v in spec_dict.items() if abs(k - (observed_mz + 1.9980)) <= 0.025], default=0.0)
            if m1_i > 0:
                obs_m1_pct = (m1_i / m0_i) * 100.0
            if m2_i > 0:
                obs_m2_pct = (m2_i / m0_i) * 100.0

    hits = []
    steps = [0]
    MAX_STEPS = 40000

    def _recurse(idx, cur_counts, cur_mass):
        if steps[0] >= MAX_STEPS:
            return
        if idx == len(het_bounds):
            steps[0] += 1
            rem = neutral_mass - cur_mass
            if rem < 12.0 - tol_da:
                return
            max_c = int((rem + tol_da) // 12.0)
            min_c = max(1, int((rem - tol_da) // 15.0))
            n_n = cur_counts.get("N", 0)
            n_p = cur_counts.get("P", 0)
            n_b = cur_counts.get("B", 0)
            n_x = cur_counts.get("F", 0) + cur_counts.get("Cl", 0) + cur_counts.get("Br", 0) + cur_counts.get("I", 0)
            n_si = cur_counts.get("Si", 0)

            for n_c in range(min_c, max_c + 1):
                h_mass = rem - n_c * 12.0
                if h_mass < -tol_da:
                    break
                n_h = int(round(h_mass / elem_mass["H"]))
                if n_h < 0 or n_h > (2 * n_c + 2 + n_n + n_p + n_b + 2 * n_si):
                    continue
                calc_neutral = cur_mass + n_c * 12.0 + n_h * elem_mass["H"]
                err_da = calc_neutral - neutral_mass
                if abs(err_da) > tol_da:
                    continue

                # Nitrogen Rule / closed-shell neutral RDBE check
                rdbe_2 = 2 * (n_c + n_si) - (n_h + n_x) + (n_n + n_p + n_b) + 2
                if rdbe_2 % 2 != 0:
                    continue
                rdbe = rdbe_2 // 2
                if rdbe < 0 or rdbe > n_c + 4:
                    continue

                ppm_err = (err_da / observed_mz) * 1e6
                theo_m1 = n_c * 1.07 + n_n * 0.37 + cur_counts.get("S", 0) * 0.76
                theo_m2 = cur_counts.get("Br", 0) * 97.3 + cur_counts.get("Cl", 0) * 32.0 + cur_counts.get("S", 0) * 4.2 + cur_counts.get("O", 0) * 0.2

                # Combined ranking score (lower is better)
                iso_penalty = 0.0
                if obs_m1_pct is not None:
                    iso_penalty += abs(theo_m1 - obs_m1_pct) * 0.6
                if obs_m2_pct is not None:
                    iso_penalty += abs(theo_m2 - obs_m2_pct) * 0.35
                elif cur_counts.get("Br", 0) > 0 and peak_id > 0:
                    iso_penalty += 35.0  # Br expected strong M+2 but none seen

                # Format Hill-system formula
                f_parts = [f"C{n_c if n_c > 1 else ''}"]
                if n_h > 0:
                    f_parts.append(f"H{n_h if n_h > 1 else ''}")
                for sym in sorted(cur_counts.keys()):
                    cnt = cur_counts[sym]
                    if cnt > 0:
                        f_parts.append(f"{sym}{cnt if cnt > 1 else ''}")
                formula_str = "".join(f_parts)

                hits.append({
                    "formula": formula_str,
                    "neutral_exact_mass": round(calc_neutral, 4),
                    "expected_adduct_mz": round(calc_neutral + adduct_shifts[adduct], 4),
                    "ppm_error": round(ppm_err, 2),
                    "RDBE": rdbe,
                    "theo_M1_pct": round(theo_m1, 1),
                    "theo_M2_pct": round(theo_m2, 1),
                    "_rank_score": abs(ppm_err) + iso_penalty
                })
            return

        sym, cap = het_bounds[idx]
        m_sym = elem_mass[sym]
        max_here = min(cap, int((neutral_mass + tol_da - cur_mass) // m_sym))
        for cnt in range(max_here + 1):
            if cnt > 0:
                cur_counts[sym] = cnt
            elif sym in cur_counts:
                del cur_counts[sym]
            _recurse(idx + 1, cur_counts, cur_mass + cnt * m_sym)
        if sym in cur_counts:
            del cur_counts[sym]

    _recurse(0, {}, 0.0)
    hits.sort(key=lambda x: x["_rank_score"])
    top_hits = []
    for h in hits[:max(1, min(int(max_results), 20))]:
        h_clean = {k: v for k, v in h.items() if k != "_rank_score"}
        top_hits.append(h_clean)

    return {
        "observed_mz": round(observed_mz, 4),
        "adduct_assumed": adduct,
        "target_neutral_mass": round(neutral_mass, 4),
        "allowed_reaction_elements": ["C", "H"] + [s for s, _ in het_bounds],
        "observed_isotope_constraints": {
            "peak_id": peak_id if peak_id > 0 else None,
            "obs_M1_pct": round(obs_m1_pct, 1) if obs_m1_pct is not None else None,
            "obs_M2_pct": round(obs_m2_pct, 1) if obs_m2_pct is not None else None,
        },
        "total_valid_formulas_found": len(hits),
        "top_ranked_formulas": top_hits
    }


def diagnose_mass_delta_to_reaction_species(
    peak_id: int = 0,
    observed_mh_mz: float = 0.0,
    tolerance_da: float = 0.02
) -> dict:
    """
    Computes the exact mass difference (Δm) between an unknown peak (or observed [M+H]+ m/z)
    and every Reactant, Reagent, and Product in Cell 3, matching against common synthetic
    transformations (oxidation, hydrolysis, demethylation, dehalogenation, homocoupling, solvent adducts).
    """
    if observed_mh_mz <= 0.0 and peak_id > 0 and 'detected_df' in globals():
        row = detected_df[detected_df['peak_id'] == peak_id]
        if not row.empty:
            dom_str = str(row.iloc[0].get('dominant m/z', 'None'))
            m = re.search(r"([0-9]+\.[0-9]+)", dom_str)
            if m:
                observed_mh_mz = float(m.group(1))

    if observed_mh_mz <= 0.0:
        return {"error": "Provide a valid `peak_id` with MS1 data or a positive `observed_mh_mz`."}

    obs_neutral = observed_mh_mz - PROTON_MASS

    known_deltas = [
        ("Exact mass match (isomer / regioisomer)", 0.0000),
        ("Dehydrogenation / oxidation (-H2)", -2.01565),
        ("Reduction / hydrogenation (+H2)", +2.01565),
        ("Demethylation / methyl ester hydrolysis (-CH2)", -14.01565),
        ("Methylation / MeOH transesterification (+CH2)", +14.01565),
        ("Mono-oxidation / hydroxylation / N-oxide (+O)", +15.99491),
        ("Ammonium adduct shift ([M+NH4]+ vs [M+H]+)", +17.02655),
        ("Dehydration / cyclization (-H2O)", -18.01056),
        ("Hydration / nitrile-or-amide hydrolysis (+H2O)", +18.01056),
        ("Sodiated adduct shift ([M+Na]+ vs [M+H]+)", +21.98194),
        ("Carbonylation / N-formylation (+CO)", +27.99491),
        ("Decarbonylation (-CO)", -27.99491),
        ("Ethyl ester hydrolysis (-C2H4)", -28.03130),
        ("Di-oxidation (+O2)", +31.98983),
        ("MeOH adduct or MeOH addition (+CH3OH)", +32.02621),
        ("MeOH loss / acylium cleavage (-CH3OH)", -32.02621),
        ("Hydrodechlorination (-Cl + H)", -33.96103),
        ("Potassiated adduct shift ([M+K]+ vs [M+H]+)", +37.95588),
        ("ACN solvent adduct ([M+ACN+H]+ vs [M+H]+)", +41.02655),
        ("N-Acetylation (+C2H2O)", +42.01056),
        ("Decarboxylation (-CO2)", -43.98983),
        ("Protodeboronation (-B(OH)2 + H)", -43.99064),
        ("Boronic acid oxidation to phenol (-B(OH)2 + OH)", -27.99573),
        ("t-Bu / isobutene cleavage (-C4H8)", -56.06260),
        ("Aryl bromide hydroxylation (-Br + OH)", -61.91560),
        ("Hydrodebromination (-Br + H)", -77.91051),
        ("Full Boc cleavage (-C5H8O2)", -100.05243),
        ("TFA trifluoroacetamide adduct (+C2F3O - H)", +95.98230),
    ]

    candidates = _get_live_candidates_list()
    diagnostic_matches = []
    raw_deltas = []

    for name, role, smi in candidates:
        if role not in ("Reactant", "Reagent", "Product"):
            continue
        mol = Chem.MolFromSmiles(smi)
        if not mol:
            continue
        em = Descriptors.ExactMolWt(mol)
        dm = obs_neutral - em
        raw_deltas.append({"species": f"{name} ({role})", "exact_mass": round(em, 4), "delta_Da": round(dm, 4)})

        for label, shift in known_deltas:
            if abs(dm - shift) <= tolerance_da:
                diagnostic_matches.append({
                    "reference_species": f"{name} ({role})",
                    "transformation_hypothesis": label,
                    "observed_delta_Da": round(dm, 4),
                    "theoretical_delta_Da": round(shift, 4),
                    "error_Da": round(dm - shift, 4)
                })

        # Check homocoupling / dimerization hypotheses against 2 * em
        for dimer_label, dimer_mass in [
            ("Non-covalent ESI dimer ([2M+H]+)", 2 * em),
            ("Oxidative dimerization (2M - 2H)", 2 * em - 2.01565),
            ("Aryl bromide homocoupling dimer (2M - 2Br)", 2 * em - 2 * 78.91834),
            ("Aryl chloride homocoupling dimer (2M - 2Cl)", 2 * em - 2 * 34.96885),
            ("Boronic acid homocoupling dimer (2M - 2B(OH)2)", 2 * em - 2 * 44.99846),
        ]:
            if abs(obs_neutral - dimer_mass) <= tolerance_da:
                diagnostic_matches.append({
                    "reference_species": f"{name} ({role})",
                    "transformation_hypothesis": dimer_label,
                    "observed_delta_Da": round(obs_neutral - 2 * em, 4),
                    "theoretical_delta_Da": round(dimer_mass - 2 * em, 4),
                    "error_Da": round(obs_neutral - dimer_mass, 4)
                })

    diagnostic_matches.sort(key=lambda x: abs(x["error_Da"]))
    return {
        "peak_id": peak_id if peak_id > 0 else None,
        "observed_mh_mz": round(observed_mh_mz, 4),
        "inferred_neutral_mass": round(obs_neutral, 4),
        "matched_synthetic_transformations": diagnostic_matches,
        "all_mass_deltas_vs_reaction_species": raw_deltas
    }


# ==============================================================================
# IN-SILICO FRAGMENTATION, MASS ANALYSIS & 2D SVG TOOLS
# ==============================================================================

def generate_in_silico_fragments(parent_smiles: str, check_peak_id: int = 0) -> dict:
    """
    Generates chemically valid, RDKit-sanitized ESI+ fragments & synthetic neutral-loss ions
    for organic reaction components (esters, amides, biaryls, aryl halides, boronic acids,
    Boc/Cbz carbamates, sulfonamides). Automatically pre-renders 2D SVG cards (`image_token`)
    for the parent and every fragment, and optionally verifies if each m/z appears in `check_peak_id`.
    """
    mol, clean_parent, err = safe_mol_from_smiles(parent_smiles)
    if not mol:
        return {"error": err}

    parent_em = Descriptors.ExactMolWt(mol)
    parent_formula = rdMolDescriptors.CalcMolFormula(mol)
    parent_mh = round(parent_em + PROTON_MASS, 4)
    parent_img_token = _register_mol_svg(mol, label=f"Parent ({parent_formula})")

    fragments_dict = {}

    def _add_frag(frag_smi, pathway_label, is_intrinsic_cation=False):
        f_mol, f_clean, _ = safe_mol_from_smiles(frag_smi)
        if not f_mol:
            return
        em = Descriptors.ExactMolWt(f_mol)
        if em < 40.0 or em >= parent_em - 1.0:
            return
        form = rdMolDescriptors.CalcMolFormula(f_mol)
        f_charge = Chem.GetFormalCharge(f_mol)
        if is_intrinsic_cation or f_charge > 0:
            mz_val = round(em - ELECTRON_MASS, 4)
            ion_type = "[F]+ (acylium / carbocation / radical cation)"
        else:
            mz_val = round(em + PROTON_MASS, 4)
            ion_type = "[F+H]+"

        key = (form, round(mz_val, 3))
        if key not in fragments_dict:
            entry = {
                "fragment_name": pathway_label,
                "smiles": f_clean,
                "formula": form,
                "neutral_exact_mass": round(em, 4),
                "ion_type": ion_type,
                "expected_esi_pos_mz": mz_val,
                "neutral_loss_from_parent_Da": round(parent_mh - mz_val, 4),
                "_temp_mol": f_mol
            }
            if check_peak_id > 0:
                hit_info = check_mz_in_peak(check_peak_id, mz_val, tolerance_da=0.05)
                entry["present_in_peak"] = hit_info.get("found_in_raw_scans", False)
                entry["peak_observation"] = hit_info.get("max_observed_hit", None)
            fragments_dict[key] = entry

    synthetic_fg_rules = [
        ("[C,c:1][C:2](=[O:3])O[CH3]", "[C,c:1][C+:2]=[O:3]", "Acylium ion via MeOH loss ([M+H-CH3OH]+, -32.03 Da)", True),
        ("[C,c:1][C:2](=[O:3])O[CH2][CH3]", "[C,c:1][C+:2]=[O:3]", "Acylium ion via EtOH loss ([M+H-EtOH]+, -46.04 Da)", True),
        ("[C,c:1][C:2](=[O:3])[OH]", "[C,c:1][C+:2]=[O:3]", "Acylium ion via dehydration ([M+H-H2O]+, -18.01 Da)", True),
        ("[C,c:1]C(=O)[OH]", "[C,c:1][H]", "Decarboxylated core ([M+H-CO2]+, -43.99 Da)", False),
        ("[c,C:1][Br]", "[c,C:1][H]", "Protodebrominated fragment ([M+H-Br+H]+, loss of Br)", False),
        ("[c,C:1][Cl]", "[c,C:1][H]", "Protodechlorinated fragment ([M+H-Cl+H]+, loss of Cl)", False),
        ("[c,C:1]B(O)O", "[c,C:1][H]", "Protodeboronated core ([M+H-B(OH)2+H]+)", False),
        ("[c:1]O[CH3]", "[c:1]O", "Demethylated phenol fragment", False),
        ("[N,O:1]C(=O)OC(C)(C)C", "[N,O:1]C(=O)O", "Carbamic acid ion after isobutene loss ([M+H-C4H8]+, -56.06 Da)", False),
        ("[N,O:1]C(=O)OC(C)(C)C", "[N,O:1][H]", "Deprotected amine after Boc loss ([M+H-Boc]+, -100.05 Da)", False),
        ("[C,c:1][C:2](=[O:3])[N:4]", "[C,c:1][C+:2]=[O:3]", "Acylium ion from amide C-N cleavage", True),
    ]

    for smarts_pat, prod_smarts, label, is_cat in synthetic_fg_rules:
        try:
            rxn = AllChem.ReactionFromSmarts(f"{smarts_pat}>>{prod_smarts}")
            for prod_tuple in rxn.RunReactants((mol,)):
                try:
                    p_mol = prod_tuple[0]
                    Chem.SanitizeMol(p_mol)
                    _add_frag(Chem.MolToSmiles(p_mol, isomericSmiles=True), label, is_intrinsic_cation=is_cat)
                except Exception:
                    continue
        except Exception:
            continue

    cleavable_bonds = [
        b.GetIdx() for b in mol.GetBonds()
        if not b.IsInRing() and b.GetBondType() in (Chem.BondType.SINGLE, Chem.BondType.AROMATIC)
    ]
    for b_idx in cleavable_bonds:
        try:
            frag_mol = Chem.FragmentOnBonds(mol, [b_idx], addDummies=False)
            for f in Chem.GetMolFrags(frag_mol, asMols=True, sanitizeFrags=False):
                _add_frag(Chem.MolToSmiles(f), "Acyclic / Biaryl single-bond cleavage ([F+H]+)")
        except Exception:
            pass

    sorted_frags = sorted(fragments_dict.values(), key=lambda x: x["expected_esi_pos_mz"], reverse=True)
    for f_entry in sorted_frags:
        f_mol = f_entry.pop("_temp_mol")
        f_entry["image_token"] = _register_mol_svg(f_mol, label=f_entry["fragment_name"])

    return {
        "parent_smiles": clean_parent,
        "parent_formula": parent_formula,
        "parent_exact_mass": round(parent_em, 4),
        "parent_[M+H]+_mz": parent_mh,
        "parent_image_token": parent_img_token,
        "generated_fragments": sorted_frags,
        "rendering_instruction": (
            "Each structure above already has its 2D SVG card pre-rendered! "
            "Place the exact `parent_image_token` or `image_token` (e.g. [[MOL_IMG_1]]) "
            "on its own line in your response where you want that molecule displayed."
        )
    }


def analyze_structure_and_mass(smiles_or_formula: str, target_mz: float = 0.0, label: str = "") -> dict:
    """
    Calculates exact monoisotopic mass, molecular formula, ESI m/z values ([M]+, [M+H]+, [M+Na]+),
    halogen M+2 isotope shift (if Br or Cl is present), cLogP, and TPSA, and pre-renders a 2D SVG card (`image_token`).
    """
    mol, clean_smi, err = safe_mol_from_smiles(smiles_or_formula)
    if mol is None:
        try:
            em = rdMolDescriptors.CalcExactMolWt(smiles_or_formula.strip())
            return {
                "input_interpreted_as": "Molecular Formula (no 2D topology)",
                "formula": smiles_or_formula.strip(),
                "exact_monoisotopic_mass": round(em, 5),
                "calculated_mz_species": {
                    "intrinsic_or_radical_[M]+": round(em - ELECTRON_MASS, 4),
                    "protonated_[M+H]+": round(em + PROTON_MASS, 4),
                    "sodiated_[M+Na]+": round(em + 22.989218, 4)
                }
            }
        except Exception:
            return {"error": err}

    exact_mass = Descriptors.ExactMolWt(mol)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    formal_charge = Chem.GetFormalCharge(mol)

    mz_intrinsic = exact_mass - (formal_charge * ELECTRON_MASS) if formal_charge != 0 else exact_mass
    mz_mh_plus = exact_mass + PROTON_MASS if formal_charge == 0 else mz_intrinsic
    img_token = _register_mol_svg(mol, label=label or formula)

    res = {
        "canonical_smiles": clean_smi,
        "formula": formula,
        "formal_charge": formal_charge,
        "exact_monoisotopic_mass": round(exact_mass, 5),
        "rdkit_cLogP": round(Crippen.MolLogP(mol), 3),
        "TPSA_A2": round(rdMolDescriptors.CalcTPSA(mol), 2),
        "calculated_mz_species": {
            "intrinsic_or_radical_[M]+": round(exact_mass - ELECTRON_MASS, 4),
            "protonated_[M+H]+": round(mz_mh_plus, 4),
            "sodiated_[M+Na]+": round(exact_mass + 22.989218, 4) if formal_charge == 0 else "N/A"
        },
        "image_token": img_token,
        "instruction": f"Place {img_token} on its own line in your response to display the 2D structure."
    }
    if "Br" in formula:
        res["halogen_M_plus_2_mz"] = round(mz_mh_plus + 1.99795, 4)
        res["halogen_note"] = "Contains Bromine (79Br/81Br ~1:1 M / M+2 doublet)"
    elif "Cl" in formula:
        res["halogen_M_plus_2_mz"] = round(mz_mh_plus + 1.99705, 4)
        res["halogen_note"] = "Contains Chlorine (35Cl/37Cl ~3:1 M / M+2 doublet)"

    if target_mz > 0:
        diff_da = mz_mh_plus - target_mz
        res["target_mz_comparison"] = {
            "target_mz": target_mz,
            "error_Da": round(diff_da, 4),
            "ppm_error": round((diff_da / target_mz) * 1e6, 1)
        }
    return res


def generate_2d_structure_svg(smiles: str, label: str = "", width: int = 300, height: int = 220) -> dict:
    """
    Uses RDKit to generate a 2D chemical structure SVG from a SMILES string.
    Stores the SVG in `_rendered_svg_registry` under both `[[MOL_IMG_X]]` and `{{SVG_X}}`.
    """
    mol, clean_smi, err = safe_mol_from_smiles(smiles)
    if not mol:
        return {"error": err}

    token_id = _register_mol_svg(mol, label=label, width=width, height=height)
    exact_mass = Descriptors.ExactMolWt(mol)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    mh_mz = exact_mass + PROTON_MASS if Chem.GetFormalCharge(mol) == 0 else exact_mass

    return {
        "status": "success",
        "canonical_smiles_drawn": clean_smi,
        "formula": formula,
        "exact_mass": round(exact_mass, 4),
        "expected_mh_mz": round(mh_mz, 4),
        "image_token": token_id,
        "svg_placeholder_token": token_id,
        "instruction": f"Insert the exact token {token_id} on its own line in your response where you want this 2D structure displayed."
    }


# Auto-exported tool list for Cell 8E
CELL6_TOOLS = [
    get_user_candidate_molecules,
    get_detected_peaks_summary,
    calculate_crude_reaction_conversion_summary,
    get_peak_details,
    check_mz_in_peak,
    find_mz_across_chromatogram,
    check_peak_purity_and_coelution,
    verify_isotope_envelope_in_peak,
    analyze_chromatographic_resolution_and_shape,
    elucidate_unknown_formula_from_mz,
    diagnose_mass_delta_to_reaction_species,
    generate_in_silico_fragments,
    analyze_structure_and_mass,
    generate_2d_structure_svg,
]

print(f"✅ Cell 6 ready: {len(CELL6_TOOLS)} Crude Reaction RDKit & MS1 diagnostic tools loaded!")

In [ ]:
#@title 8A. Imports, Global Registries & RDKit SVG Helpers

import re
import io
import json
import uuid
import base64
import html
import math
import sqlite3
import urllib.request
import urllib.parse
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import pubchempy as pcp
from scipy.integrate import trapezoid
from scipy.signal import find_peaks, peak_widths
from google.colab import userdata
from google import genai
from google.genai import types
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown, HTML
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, Crippen, Draw
from rdkit.Chem.Draw import rdMolDraw2D

# 1. Silence RDKit C++ warnings
RDLogger.DisableLog('rdApp.*')

# Global dictionary mapping placeholder tokens ('[[MOL_IMG_X]]', '{{SVG_X}}', '[[PLOTLY_CHROM_X]]') to HTML blocks

# Preserve existing registries across cell re-runs so you don't lose SVGs/state
_rendered_svg_registry = globals().setdefault('_rendered_svg_registry', {})
SPECTRAL_STATE = globals().setdefault('SPECTRAL_STATE', {})
CHEM_CACHE_PATH = "/tmp/lcms_chem_cache.sqlite"
BENCH_CONTAMINANTS = {
    "TPPO": 279.093, "DIPEA": 130.159, "Et3N": 102.128,
    "DMAP": 123.092, "DCU": 225.196, "DIU": 145.134,
    "dibutyl phthalate": 279.159,
}
NEUTRAL_LOSSES = {
    "H2O": 18.0106, "NH3": 17.0265, "CO/N2": 28.0061,
    "HCl": 35.9767, "HCl (average)": 36.46, "CO2": 43.9898, "isobutene": 56.0626,
    "Boc": 100.0524,
}

if 'to_plotly_color' not in globals():
    def to_plotly_color(color_val):
        rgba = mcolors.to_rgba(color_val)
        return f"rgb({int(rgba[0]*255)}, {int(rgba[1]*255)}, {int(rgba[2]*255)})"


def _register_mol_svg(mol, label: str = "", width: int = 300, height: int = 220) -> str:
    """
    Renders an RDKit Mol object to a Base64-encoded SVG <img> HTML card,
    stores it in `_rendered_svg_registry` under both `[[MOL_IMG_N]]` and `{{SVG_N}}`,
    and returns the token `[[MOL_IMG_N]]`.
    """
    mol_copy = Chem.Mol(mol)
    AllChem.Compute2DCoords(mol_copy)

    drawer = rdMolDraw2D.MolDraw2DSVG(width, height)
    opts = drawer.drawOptions()
    opts.addStereoAnnotation = True
    opts.padding = 0.12

    exact_mass = Descriptors.ExactMolWt(mol_copy)
    formula = rdMolDescriptors.CalcMolFormula(mol_copy)
    f_charge = Chem.GetFormalCharge(mol_copy)
    mh_mz = exact_mass + 1.007276 if f_charge == 0 else exact_mass - (f_charge * 0.00054858)

    drawer.DrawMolecule(mol_copy)
    drawer.FinishDrawing()
    svg_str = drawer.GetDrawingText()

    b64_svg = base64.b64encode(svg_str.encode('utf-8')).decode('utf-8')
    mol_count = sum(1 for k in _rendered_svg_registry if k.startswith("[[MOL_IMG_")) + 1

    caption_title = f"<div style='font-weight:700; font-size:12px; color:#111827; margin-bottom:4px;'>{label}</div>" if label else ""
    caption_sub = f"<div style='font-size:11px; color:#374151; margin-top:4px;'><b>{formula}</b> | Exact: {exact_mass:.4f} | m/z: {mh_mz:.4f}</div>"

    html_card = (
        f"<div style='display:inline-block; margin:10px 6px; padding:10px 14px; border:2px solid #cbd5e1; "
        f"border-radius:10px; background:#ffffff; text-align:center; box-shadow:0 2px 6px rgba(0,0,0,0.15);'>"
        f"{caption_title}"
        f"<img src='data:image/svg+xml;base64,{b64_svg}' width='{width}' height='{height}' style='display:block; margin:0 auto; background:#ffffff;'/>"
        f"{caption_sub}"
        f"</div>"
    )

    token_primary = f"[[MOL_IMG_{mol_count}]]"
    token_alt = f"{{{{SVG_{mol_count}}}}}"
    _rendered_svg_registry[token_primary] = html_card
    _rendered_svg_registry[token_alt] = html_card
    return token_primary


def safe_mol_from_smiles(smiles_or_str: str):
    """Auto-repairs unkekulized pyrrole/purine/imidazole nitrogens (missing [nH])."""
    s = smiles_or_str.strip()
    if not s:
        return None, None, "Empty input string."

    if re.fullmatch(r'([A-Z][a-z]?\d*)+([+-]\d*)?', s) and not any(c in s for c in ['c', 'n', 'o', 's', '=', '#', '(', ')', '[']):
        if any(char.isdigit() for char in s) and not re.search(r'[1-9].*[1-9]', s[:3]):
            return None, None, f"'{s}' is a molecular formula, not a SMILES string."

    mol = Chem.MolFromSmiles(s)
    if mol is not None:
        return mol, Chem.CanonSmiles(s), None

    mol_no_san = Chem.MolFromSmiles(s, sanitize=False)
    if mol_no_san is not None:
        try:
            mol_no_san.UpdatePropertyCache(strict=False)
            Chem.GetSymmSSSR(mol_no_san)
            for atom in mol_no_san.GetAtoms():
                if atom.GetAtomicNum() == 7 and atom.GetIsAromatic() and atom.GetDegree() == 2 and atom.GetFormalCharge() == 0 and atom.GetNumExplicitHs() == 0:
                    test_mol = Chem.Mol(mol_no_san)
                    test_mol.GetAtomWithIdx(atom.GetIdx()).SetNumExplicitHs(1)
                    try:
                        Chem.SanitizeMol(test_mol)
                        return test_mol, Chem.MolToSmiles(test_mol), None
                    except Exception:
                        continue
        except Exception:
            pass

    return None, None, f"RDKit could not parse '{s}'. Ensure aromatic 5-membered ring NH groups are written as '[nH]'."


In [ ]:
#@title 8B. Local Spectral State, Isotope Math & Multi-Panel LCMS Dashboard
# --- LOCAL SPECTRAL STATE & UV/DAD DASHBOARD HELPERS ---

def calculate_expected_ions(smiles: str) -> dict:
    """Monoisotopic ESI ions (singly charged) from a neutral, valid SMILES."""
    mol, clean, error = safe_mol_from_smiles(smiles)
    if mol is None:
        return {"error": error}
    mass = Descriptors.ExactMolWt(mol)
    positive = {
        "[M+H]+": mass + 1.0073, "[M+NH4]+": mass + 18.0338,
        "[M+Na]+": mass + 22.9892, "[M+K]+": mass + 38.9632,
        "[M+CH3CN+H]+": mass + 42.0338,
        "[2M+H]+": 2 * mass + 1.0073, "[2M+Na]+": 2 * mass + 22.9892,
    }
    negative = {
        "[M-H]-": mass - 1.0073, "[M+Cl]-": mass + 34.9694,
        "[M+HCOO]-": mass + 44.9982, "[M+CH3COO]-": mass + 59.0139,
    }
    return {
        "smiles": clean, "formula": rdMolDescriptors.CalcMolFormula(mol),
        "exact_mass": round(mass, 5),
        "cLogP": round(Crippen.MolLogP(mol), 2),
        "TPSA": round(rdMolDescriptors.CalcTPSA(mol), 1),
        "positive": {k: round(v, 4) for k, v in positive.items()},
        "negative": {k: round(v, 4) for k, v in negative.items()},
    }


def parse_reaction_candidates(text: str) -> list[dict]:
    """Accept SMILES, reaction SMILES, or labeled/positional comma-separated SMILES."""
    text = text.strip()
    if not text:
        return []
    if ">" in text:
        parts = text.split(">")
        if len(parts) != 3:
            return []
        entries = (
            [(s, "Starting Material") for s in parts[0].split(".")] +
            [(s, "Reagent") for s in parts[1].split(".")] +
            [(s, "Target Product") for s in parts[2].split(".")]
        )
    elif "," in text:
        pieces = [s.strip() for s in text.split(",") if s.strip()]
        entries = []
        for i, piece in enumerate(pieces):
            name, sep, smi = piece.partition(":")
            if sep:
                role = {"sm": "Starting Material", "starting material": "Starting Material",
                        "reagent": "Reagent", "reagents": "Reagent",
                        "product": "Target Product", "target product": "Target Product"}.get(name.strip().lower())
                entries.append((smi.strip(), role or "Reagent"))
            else:
                entries.append((piece, "Starting Material" if i == 0 else
                                "Target Product" if i == len(pieces) - 1 else "Reagent"))
    else:
        entries = [(text, "Target Product")]
    return [
        {"role": role, **ions}
        for smi, role in entries if (ions := calculate_expected_ions(smi)) and "error" not in ions
    ]


def reaction_mass_hypotheses(candidates: list[dict]) -> list[dict]:
    """Mass shifts are diagnostic hypotheses, not claimed molecular structures."""
    products = [c for c in candidates if c["role"] in ("Target Product", "Product")]
    if not products:
        return []
    product = products[0]
    hypotheses = []
    product_mol = Chem.MolFromSmiles(product["smiles"]) if product.get("smiles") else None
    for candidate in candidates:
        if candidate["role"] in ("Starting Material", "Reactant"):
            hypotheses.append({
                "name": "Unreacted SM", "mz": candidate["positive"]["[M+H]+"],
                "delta": round(candidate["exact_mass"] - product["exact_mass"], 4),
                "cLogP": candidate["cLogP"], "TPSA": candidate["TPSA"],
            })
    transformations = {
        "Hydrolysis": ("[C]#[N]", "C(=O)N"),
        "Des-Br": ("[Br]", None),
        "Des-Cl": ("[Cl]", None),
        "Protodeboronation": ("[B]([O;H1])[O;H1]", None),
        "Boc cleavage": ("[C](=[O])[O][C]([CH3])([CH3])[CH3]", None),
        "In-source isobutene / t-Bu ester": ("[O][C]([CH3])([CH3])[CH3]", "O"),
    }
    for name, shift in (
        ("Hydrolysis", 18.0106), ("Des-Br", -77.9105),
        ("Des-Cl", -33.9610), ("Protodeboronation", -43.9900),
        ("Boc cleavage", -100.0524), ("In-source isobutene / t-Bu ester", -56.0626),
        ("Fmoc cleavage", -222.0681),
    ):
        if name == "Des-Br" and "Br" not in product["formula"]:
            continue
        if name == "Des-Cl" and "Cl" not in product["formula"]:
            continue
        if name == "Protodeboronation" and "B" not in product["formula"]:
            continue
        mz = product["positive"]["[M+H]+"] + shift
        if mz > 0:
            entry = {"name": name, "mz": round(mz, 4), "delta": shift,
                     "cLogP": None, "TPSA": None}
            if product_mol and name in transformations:
                pattern, replacement = transformations[name]
                query = Chem.MolFromSmarts(pattern)
                if query and product_mol.HasSubstructMatch(query):
                    try:
                        if replacement:
                            altered = Chem.ReplaceSubstructs(product_mol, query,
                                Chem.MolFromSmiles(replacement), replaceAll=False)[0]
                        else:
                            altered = Chem.DeleteSubstructs(product_mol, query, onlyFrags=False)
                        Chem.SanitizeMol(altered)
                        if abs(Descriptors.ExactMolWt(altered) -
                               (product["exact_mass"] + shift)) < .05:
                            entry.update(smiles=Chem.MolToSmiles(altered),
                                cLogP=round(Crippen.MolLogP(altered), 2),
                                TPSA=round(rdMolDescriptors.CalcTPSA(altered), 1))
                    except Exception:
                        pass
            hypotheses.append(entry)
    return hypotheses


def theoretical_isotope_envelope(formula_or_smiles: str) -> list[dict]:
    """Natural-abundance nominal M..M+4 envelope with exact centroid mass offsets."""
    mol, _, _ = safe_mol_from_smiles(formula_or_smiles)
    formula = rdMolDescriptors.CalcMolFormula(mol) if mol is not None else formula_or_smiles.strip()
    matches = list(re.finditer(r"([A-Z][a-z]?)(\d*)", formula))
    if not matches or "".join(m.group(0) for m in matches) != formula:
        raise ValueError("Provide a neutral molecular formula or valid SMILES.")
    isotope = {
        "C": [(0, 0, .9893), (1, 1.003355, .0107)],
        "H": [(0, 0, .999885), (1, 1.006277, .000115)],
        "N": [(0, 0, .99636), (1, .997035, .00364)],
        "O": [(0, 0, .99757), (1, 1.004217, .00038), (2, 2.004245, .00205)],
        "S": [(0, 0, .9499), (1, .999388, .0075), (2, 1.995796, .0425), (4, 3.995010, .0001)],
        "Cl": [(0, 0, .7578), (2, 1.997050, .2422)],
        "Br": [(0, 0, .5069), (2, 1.997953, .4931)],
    }
    pt = Chem.GetPeriodicTable()
    mass = 0.0
    distribution = {0: (1.0, 0.0)}
    for match in matches:
        element, count = match.group(1), int(match.group(2) or 1)
        if pt.GetAtomicNumber(element) == 0 or count > 300:
            raise ValueError("Invalid or oversized molecular formula.")
        mass += pt.GetMostCommonIsotopeMass(element) * count
        for _ in range(count):
            next_dist = {}
            for shift, (prob, weighted) in distribution.items():
                for step, delta, abundance in isotope.get(element, [(0, 0.0, 1.0)]):
                    if shift + step > 4:
                        continue
                    p, w = next_dist.get(shift + step, (0.0, 0.0))
                    next_dist[shift + step] = (p + prob * abundance,
                                               w + (weighted + prob * delta) * abundance)
            distribution = next_dist
    max_prob = max(p for p, _ in distribution.values())
    return [
        {"label": f"M+{shift}" if shift else "M",
         "mz": round(mass + 1.007276 + weighted / prob, 5),
         "relative_intensity": round(100 * prob / max_prob, 2)}
        for shift in (0, 1, 2, 4)
        if (prob := distribution.get(shift, (0, 0))[0]) > 0
        for weighted in [distribution[shift][1]]
    ]


def annotate_neutral_losses(ions: list[dict], tolerance_da: float = 0.04) -> list[dict]:
    """Match neutral losses among the ten most intense baseline-subtracted ions."""
    losses = []
    top = sorted(ions, key=lambda ion: ion["intensity"], reverse=True)[:10]
    for parent in top:
        for fragment in top:
            delta = parent["mz"] - fragment["mz"]
            for name, mass in NEUTRAL_LOSSES.items():
                if abs(delta - mass) <= tolerance_da:
                    losses.append({"loss": name, "parent": parent["mz"], "fragment": fragment["mz"]})
    return losses


def _centroid_scans(scans: list[tuple], rt: float, onset: float) -> list[dict]:
    """Average three nearest MS1 scans; subtract a preceding background scan by m/z bin."""
    if not scans:
        return []
    closest = int(np.argmin([abs(s[0] - rt) for s in scans]))
    selected = scans[max(0, closest - 1):closest + 2]
    def bins(spectrum):
        result = {}
        for mz, intensity in zip(spectrum[1], spectrum[2]):
            key = round(float(mz), 2)
            if intensity > 0:
                total, weighted_mz = result.get(key, (0.0, 0.0))
                result[key] = (total + float(intensity),
                               weighted_mz + float(mz * intensity))
        return result
    averaged = {}
    for scan in selected:
        for mz, (intensity, weighted_mz) in bins(scan).items():
            total, weighted = averaged.get(mz, (0.0, 0.0))
            averaged[mz] = (total + intensity / len(selected),
                            weighted + weighted_mz / len(selected))
    preceding_idx = [i for i, scan in enumerate(scans) if scan[0] < onset]
    if preceding_idx:
        best_bg_idx = min(preceding_idx, key=lambda i: abs(scans[i][0] - (onset - 0.05)))
        background = bins(scans[best_bg_idx])
    else:
        background = {}
    corrected = {mz: (max(0.0, intensity - background.get(mz, (0.0, 0.0))[0]),
                      weighted / intensity)
                 for mz, (intensity, weighted) in averaged.items() if intensity > 0}
    if not corrected:
        return []
    threshold = max(intensity for intensity, _ in corrected.values()) * .02
    return [{"mz": round(centroid, 4), "intensity": round(intensity, 1)}
            for mz, (intensity, centroid) in sorted(corrected.items(),
                                                    key=lambda item: -item[1][0])[:10]
            if intensity >= threshold]


def _flag_artifacts(ions: list[dict]) -> list[str]:
    masses = [ion["mz"] for ion in ions]
    flags = [name for name, mz in BENCH_CONTAMINANTS.items()
             if any(abs(observed - mz) <= .03 for observed in masses)]
    if any(abs(a - b - 44.026) <= .03 for a in masses for b in masses if a > b) or any(
        abs(mz - (18.0106 + n * 44.0262 + adduct)) <= .03
        for mz in masses for n in range(6, 13) for adduct in (1.0073, 22.9892)
    ):
        flags.append("PEG-400")
    return flags


def prepare_spectral_state(max_peaks: int = 6, uv_ms_delay_min: float = 0.08) -> dict:
    """Keep raw arrays locally; only small peak metadata may enter the model context."""
    if not 1 <= max_peaks <= 20 or not math.isfinite(uv_ms_delay_min) or uv_ms_delay_min < 0:
        raise ValueError("Use 1-20 peaks and a nonnegative UV/MS delay.")
    scans = []
    if "od" in globals():
        for spec in od:
            if spec.getMSLevel() == 1:
                mzs, intensities = spec.get_peaks()
                scans.append((float(spec.getRT()) / 60.0, np.asarray(mzs), np.asarray(intensities)))
    scans.sort(key=lambda s: s[0])
    times = np.array([s[0] for s in scans])
    tic = np.array([np.sum(s[2]) for s in scans])
    bpc = np.array([np.max(s[2]) if len(s[2]) else 0 for s in scans])
    if not scans and "hplc_df" in globals():
        times = np.asarray(hplc_df["Time"], dtype=float) / 60.0
        tic = np.asarray(hplc_df["Signal"], dtype=float)
        bpc = np.zeros_like(tic)
    signal = np.maximum(0, tic - np.percentile(tic, 10)) if len(tic) else tic
    indices = find_peaks(signal, prominence=max(float(np.max(signal)) * .05, 1e-10),
                         distance=3)[0] if len(signal) >= 3 else []
    indices = sorted(indices, key=lambda i: -signal[i])[:max_peaks]
    candidates = parse_reaction_candidates(reaction_input.value) if "reaction_input" in globals() else []
    if not candidates and "get_candidates_from_table" in globals():
        candidates = [
            {"role": classification, **ions}
            for _, classification, smi in get_candidates_from_table()
            if (ions := calculate_expected_ions(smi)) and "error" not in ions
        ]
    hypotheses = reaction_mass_hypotheses(candidates)
    peaks = []
    for idx in indices:
        width = peak_widths(signal, [idx], rel_height=.8)
        onset = float(times[max(0, int(width[2][0]))])
        ions = _centroid_scans(scans, float(times[idx]), onset)
        matches = [c for c in candidates
                   if any(abs(ion["mz"] - mz) <= max(.03, mz * 15e-6)
                          for mz in c["positive"].values() for ion in ions)]
        peaks.append({
            "rt_min": round(float(times[idx]), 3), "onset_min": onset,
            "ions": ions, "artifacts": _flag_artifacts(ions),
            "neutral_losses": annotate_neutral_losses(ions),
            "candidates": [{"role": c["role"], "formula": c["formula"],
                            "cLogP": c["cLogP"], "TPSA": c["TPSA"]} for c in matches],
        })
    SPECTRAL_STATE.update(scans=scans, times=times, tic=tic, bpc=bpc, peaks=peaks,
                          candidates=candidates, hypotheses=hypotheses,
                          uv_ms_delay_min=uv_ms_delay_min,
                          data_id=id(globals().get("od", globals().get("hplc_df"))),
                          reaction_text=reaction_input.value if "reaction_input" in globals() else "")
    SPECTRAL_STATE.pop("pending_plot_html", None)
    return SPECTRAL_STATE


def register_uv_trace(uv_data: pd.DataFrame, time_col: str = "Time",
                      time_unit: str = "min") -> list[str]:
    """Attach an optional DAD CSV (Time, 254 nm, 214 nm, etc.) without replacing MS data."""
    if time_unit not in ("min", "s") or time_col not in uv_data:
        raise ValueError("Provide a Time column and specify 'min' or 's'.")
    times = np.asarray(uv_data[time_col], dtype=float) / (60.0 if time_unit == "s" else 1.0)
    if not len(times) or not np.all(np.isfinite(times)) or np.any(np.diff(times) <= 0):
        raise ValueError("UV retention times must be finite and strictly increasing.")
    traces = {}
    for column in uv_data.columns:
        if column == time_col:
            continue
        values = np.asarray(uv_data[column], dtype=float)
        if len(values) == len(times) and np.all(np.isfinite(values)):
            traces[str(column)] = values
    SPECTRAL_STATE["uv"] = {"times": times, "traces": traces}
    return list(traces)


def match_uv_apex_to_ms_scan(uv_rt_min: float, uv_ms_delay_min: float = 0.08):
    """Map a UV apex to the closest MS1 scan, compensating for detector dead volume."""
    scans = SPECTRAL_STATE.get("scans", [])
    if not scans:
        return None
    if uv_ms_delay_min < 0:
        raise ValueError("UV/MS delay must be nonnegative.")
    target = uv_rt_min + uv_ms_delay_min
    return min(scans, key=lambda scan: abs(scan[0] - target))[0]


def _xic(scans: list[tuple], target_mz: float, ppm: float) -> np.ndarray:
    return np.array([
        float(np.sum(scan[2][np.abs(scan[1] - target_mz) <= target_mz * ppm * 1e-6]))
        for scan in scans
    ])


def update_lcms_plot(
    extracted_ion_mzs_csv: str = "",
    rt_start_min: float = 0.0,
    rt_end_min: float = 0.0,
    ppm_tolerance: float = 15.0,
    select_scan_rt_min: float = 0.0,
    overlay_formula_isotopes: str = ""
) -> str:
    """
    Updates the local multi-panel LCMS Plotly dashboard (UV/DAD, MS1 TIC/BPC/XIC in minutes,
    and optional MS1 scan with theoretical isotope envelope overlay).
    - extracted_ion_mzs_csv: Comma-separated target m/z values for XIC traces (e.g. '268.1041, 252.1091').
    - rt_start_min, rt_end_min: Optional retention time zoom window in minutes (leave 0.0 for full range).
    - ppm_tolerance: Mass tolerance in ppm (default 15.0).
    - select_scan_rt_min: Retention time in minutes for MS1 spectrum bar plot (0.0 for auto).
    - overlay_formula_isotopes: Optional molecular formula or SMILES to overlay theoretical isotope bars.
    """
    if not SPECTRAL_STATE or "times" not in SPECTRAL_STATE:
        prepare_spectral_state()
    if not 0 < ppm_tolerance <= 1000:
        return "Invalid ppm tolerance."

    extracted_ion_mz = []
    if extracted_ion_mzs_csv.strip():
        try:
            extracted_ion_mz = [float(x.strip()) for x in extracted_ion_mzs_csv.split(",") if x.strip()]
        except ValueError:
            return "Invalid extracted_ion_mzs_csv format; provide comma-separated numbers."

    if len(extracted_ion_mz) > 8 or any(not math.isfinite(mz) or mz <= 0 for mz in extracted_ion_mz):
        return "Provide at most eight positive finite extracted-ion masses."

    rt_window = [rt_start_min, rt_end_min] if (rt_end_min > rt_start_min >= 0.0) else None
    select_scan_rt = select_scan_rt_min if select_scan_rt_min > 0.0 else None
    iso_str = overlay_formula_isotopes.strip()

    if iso_str and not SPECTRAL_STATE.get("scans"):
        return "No MS1 scan is available for an isotope overlay."
    try:
        envelope = theoretical_isotope_envelope(iso_str) if iso_str else []
    except (ValueError, RuntimeError) as exc:
        return f"Invalid isotope formula: {exc}"

    scans = SPECTRAL_STATE.get("scans", [])
    times = SPECTRAL_STATE["times"]
    tic = SPECTRAL_STATE["tic"]
    bpc = SPECTRAL_STATE["bpc"]
    uv = SPECTRAL_STATE.get("uv")
    show_ms = bool(scans) and (select_scan_rt is not None or bool(envelope))
    fig = make_subplots(rows=3 if show_ms else 2, cols=1, shared_xaxes=False,
                        row_heights=[.32, .36, .32] if show_ms else [.44, .56],
                        vertical_spacing=.09,
                        subplot_titles=("UV/DAD (if supplied)", "MS1 chromatogram (minutes)") +
                                       (("MS1 scan (m/z)",) if show_ms else ()))
    if uv:
        uv_times = uv["times"]
        for wavelength, values in uv["traces"].items():
            fig.add_trace(go.Scatter(x=uv_times, y=values, name=f"UV {wavelength}",
                                     mode="lines"), row=1, col=1)
            baseline = float(np.percentile(values, 10))
            corrected = np.maximum(0, values - baseline)
            apices = find_peaks(corrected, prominence=max(float(np.max(corrected)) * .05, 1e-10),
                                distance=3)[0] if len(values) >= 3 else []
            if len(apices):
                widths = peak_widths(corrected, apices, rel_height=.8)
                bounds = [(max(0, int(left)), min(len(values), int(right) + 1))
                          for left, right in zip(widths[2], widths[3])]
                areas = [float(trapezoid(corrected[left:right], uv_times[left:right]))
                         for left, right in bounds]
                total = sum(areas)
                for idx, area in zip(apices, areas):
                    if total:
                        fig.add_annotation(x=float(uv_times[idx]), y=float(values[idx]),
                                           text=f"{area / total * 100:.1f}%", showarrow=True,
                                           arrowhead=2, row=1, col=1)
    else:
        fig.add_annotation(x=.5, y=.5, xref="x domain", yref="y domain",
                           text="UV/DAD not supplied — upload a Time + wavelength CSV",
                           showarrow=False, row=1, col=1)
    fig.add_trace(go.Scatter(x=times, y=tic, mode="lines", name="TIC"), row=2, col=1)
    fig.add_trace(go.Scatter(x=times, y=bpc, mode="lines", name="BPC",
                             visible="legendonly" if len(bpc) and len(tic) and
                             np.max(bpc) < np.max(tic) * .05 else True), row=2, col=1)
    for mz in extracted_ion_mz:
        if scans:
            fig.add_trace(go.Scatter(x=times, y=_xic(scans, mz, ppm_tolerance),
                                     mode="lines", name=f"XIC {mz:.4f}"), row=2, col=1)
    for peak in SPECTRAL_STATE.get("peaks", []):
        fig.add_vline(x=peak["rt_min"], line_dash="dot", line_color="#64748b", row=2, col=1)
    for h in SPECTRAL_STATE.get("hypotheses", []):
        if not scans:
            break
        intensities = _xic(scans, h["mz"], ppm_tolerance)
        indices = find_peaks(intensities, prominence=max(float(np.max(intensities)) * .05, 1e-10))[0] if len(intensities) >= 3 else []
        if len(indices):
            apex = float(times[max(indices, key=lambda idx: intensities[idx])])
            fig.add_vline(x=apex, line_dash="dash",
                          line_color="#e11d48" if "SM" in h["name"] else "#7c3aed",
                          annotation_text=html.escape(h["name"]), row=2, col=1)
    if rt_window:
        fig.update_xaxes(range=rt_window, row=2, col=1)
        fig.update_xaxes(range=rt_window, row=1, col=1)
    if show_ms:
        if select_scan_rt is None and uv and uv["traces"]:
            first = next(iter(uv["traces"].values()))
            select_scan_rt = match_uv_apex_to_ms_scan(float(uv["times"][int(np.argmax(first))]),
                                                       SPECTRAL_STATE.get("uv_ms_delay_min", 0.08))
        if select_scan_rt is None:
            select_scan_rt = SPECTRAL_STATE["peaks"][0]["rt_min"] if SPECTRAL_STATE.get("peaks") else scans[0][0]
        scan = min(scans, key=lambda s: abs(s[0] - select_scan_rt))
        fig.add_trace(go.Bar(x=scan[1], y=scan[2], name=f"Scan at {scan[0]:.3f} min",
                             marker_color="#2563eb"), row=3, col=1)
        if envelope:
            observed = float(_xic([scan], envelope[0]["mz"], ppm_tolerance)[0])
            scale = max(observed, float(np.max(scan[2])) * .1 if len(scan[2]) else 0, 1)
            fig.add_trace(go.Bar(
                x=[entry["mz"] for entry in envelope],
                y=[entry["relative_intensity"] * scale / 100.0 for entry in envelope],
                width=.08, opacity=.45, marker=dict(color="#e11d48",
                                                      line=dict(color="#e11d48", width=2)),
                name=f"Theoretical {iso_str}",
                text=[entry["label"] for entry in envelope], hovertemplate="%{text}: %{x:.4f}<extra></extra>"
            ), row=3, col=1)
        for peak in SPECTRAL_STATE.get("peaks", []):
            if abs(peak["rt_min"] - scan[0]) < .06:
                for loss in peak["neutral_losses"][:4]:
                    mask = np.abs(scan[1] - loss["fragment"]) < .03
                    if np.any(mask):
                        fig.add_annotation(x=loss["fragment"], y=float(np.max(scan[2][mask])),
                                           text=f"-{loss['loss']}", showarrow=True,
                                           ay=-50, row=3, col=1)
        fig.update_xaxes(title_text="m/z", row=3, col=1)
    fig.update_xaxes(title_text="Retention time (min)", row=2, col=1)
    fig.update_yaxes(title_text="UV absorbance", row=1, col=1)
    fig.update_yaxes(title_text="MS intensity", row=2, col=1)
    fig.update_layout(title="LCMS reaction monitoring", template="plotly_white",
                      barmode="overlay", height=850 if show_ms else 660)
    SPECTRAL_STATE["figure"] = fig
    SPECTRAL_STATE["pending_plot_html"] = fig.to_html(include_plotlyjs="cdn", full_html=False,
                                                       div_id=f"lcms_{uuid.uuid4().hex[:8]}")
    return "LCMS dashboard updated locally; the plot will be displayed below your response."

In [ ]:
#@title 8C. Interactive Crude Reaction Plotly Chromatogram & Role/Halogen Hover Cards

# --- SELF-CONTAINED INTERACTIVE PLOTLY CHROMATOGRAM BUILDER (CRUDE REACTION MODE) ---

ROLE_STYLES_8C = {
    "Byproduct":         {"bg": "#fff1f2", "fg": "#be123c", "border": "#fecdd3", "short": "Byp"},
    "Reactant":          {"bg": "#eff6ff", "fg": "#1d4ed8", "border": "#bfdbfe", "short": "SM"},
    "Starting Material": {"bg": "#eff6ff", "fg": "#1d4ed8", "border": "#bfdbfe", "short": "SM"},
    "Reagent":           {"bg": "#fefce8", "fg": "#a16207", "border": "#fde047", "short": "Rgt"},
    "Product":           {"bg": "#f0fdf4", "fg": "#15803d", "border": "#86efac", "short": "Prod"},
    "Target Product":    {"bg": "#f0fdf4", "fg": "#15803d", "border": "#86efac", "short": "Prod"},
}

def _extract_role_8c(assign_dict: dict) -> str:
    """Reads exact role/classification first so 'Byproduct' is never misread as 'Product'."""
    if not assign_dict:
        return "Unassigned"
    for key in ("role", "classification"):
        val = assign_dict.get(key)
        if val in ROLE_STYLES_8C:
            return val
    full_str = str(assign_dict.get("full_assignment", ""))
    for r_key in ("Byproduct", "Reactant", "Starting Material", "Reagent", "Target Product", "Product"):
        if f"({r_key})" in full_str:
            return r_key
    return "Unassigned"

def _detect_halogen_tag_8c(formula_str: str) -> str:
    """Flags Br or Cl atoms so chemists immediately see 79Br/81Br or 35Cl/37Cl M+2 signatures."""
    if not formula_str:
        return ""
    tags = []
    br_m = re.search(r"Br(\d*)", formula_str)
    cl_m = re.search(r"Cl(\d*)", formula_str)
    if br_m:
        tags.append(f"⁷⁹Br/⁸¹Br ({int(br_m.group(1) or 1)}×Br, M+2 ≈1:1)")
    if cl_m:
        tags.append(f"³⁵Cl/³⁷Cl ({int(cl_m.group(1) or 1)}×Cl, M+2 ≈3:1)")
    return " | ".join(tags)

def _peak_label_for_legend(p_id: int) -> str:
    if 'peak_assignments' in globals() and p_id in peak_assignments:
        a_info = peak_assignments[p_id]
        role = _extract_role_8c(a_info)
        r_short = ROLE_STYLES_8C.get(role, {"short": role})["short"]
        iso_short = " [Isobaric]" if a_info.get("is_isobaric", False) else ""
        adduct_summary = ", ".join([f"{x['adduct']} ({x['obs_mz']:.2f})" for x in a_info['adducts'][:2]])
        return f"Peak {p_id} [{r_short}]: {a_info['display_name']}{iso_short} [{adduct_summary}] (Conf: {a_info['confidence']:.0f}%)"
    if 'detected_df' in globals():
        dom_rows = detected_df.loc[detected_df['peak_id'] == p_id, 'dominant m/z']
        if len(dom_rows):
            dom_str = dom_rows.values[0]
            first_ion = str(dom_str).split(',')[0] if dom_str != "None" else "N/A"
            return f"Peak {p_id}: Unassigned Impurity (Base m/z: {first_ion})"
    return f"Peak {p_id}: Unassigned Impurity"


def build_interactive_chromatogram_html(div_id: str = "crude_rxn_replot_div") -> str:
    """
    Rebuilds the interactive Crude Reaction Plotly chromatogram HTML with adaptive RT scaling,
    raw Cell 2 TIC preservation, accurate role badges, halogen isotope alerts, and floating
    RDKit 2D structure hover cards from the live `chrom`, `detected_df`, and `peak_assignments`.
    """
    if 'detected_df' not in globals() or 'chrom' not in globals() or chrom is None:
        return "<div style='color:#b91c1c; font-family:sans-serif;'>⚠️ Required data (`detected_df`, `chrom`) not found. Run Cell 4 first.</div>"

    FIG_WIDTH_PX, FIG_HEIGHT_PX = 1220, 760
    MARGIN_L, MARGIN_R, MARGIN_T, MARGIN_B = 80, 40, 65, 220

    time_arr = chrom.df[chrom.time_col].to_numpy(dtype=float)
    fitted_base_signal = chrom.df[chrom.int_col].to_numpy(dtype=float)
    raw_tic_signal = chrom.df['Raw_TIC'].to_numpy(dtype=float) if 'Raw_TIC' in chrom.df.columns else fitted_base_signal

    t_span = max(float(time_arr.max() - time_arr.min()), 1.0)
    x_pad = 0.02 * t_span
    X_RANGE = [max(0.0, float(time_arr.min()) - x_pad), float(time_arr.max()) + x_pad]

    unmixed = getattr(chrom, 'unmixed_chromatograms', None)
    has_unmixed = (unmixed is not None) and (unmixed.ndim == 2) and (unmixed.shape[1] > 0)
    convolved = np.sum(unmixed, axis=1) if has_unmixed else None

    y_max_plot = float(max(
        np.max(raw_tic_signal) if len(raw_tic_signal) else 0.0,
        np.max(fitted_base_signal) if len(fitted_base_signal) else 0.0,
        np.max(convolved) if convolved is not None else 0.0,
        1.0
    ))
    prom_val = globals().get('current_prominence_val', 0.01)
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

    hover_cards_by_peak = {}
    qc_struct_images = {}

    for _, peak_row in detected_df.iterrows():
        p_id = int(peak_row['peak_id'])
        rt_apex = float(peak_row['retention_time'])
        norm_area = peak_row.get('normalised area', 'N/A')
        peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])

        if 'peak_assignments' in globals() and p_id in peak_assignments:
            assign = peak_assignments[p_id]
            role = _extract_role_8c(assign)
            r_style = ROLE_STYLES_8C.get(role, {"bg": "#f3f4f6", "fg": "#374151", "border": "#d1d5db", "short": role})

            mol_images = [Draw.MolToImage(m, size=(175, 145)) for m in assign["mols"]]
            if len(mol_images) > 1:
                total_w = sum(im.width for im in mol_images)
                struct_img = Image.new('RGBA', (total_w, mol_images[0].height), (255, 255, 255, 255))
                x_off = 0
                for im in mol_images:
                    struct_img.paste(im, (x_off, 0))
                    x_off += im.width
            else:
                struct_img = mol_images[0]

            qc_struct_images[p_id] = struct_img
            buf = io.BytesIO()
            struct_img.save(buf, format='PNG')
            b64_png = base64.b64encode(buf.getvalue()).decode('utf-8')

            adduct_lines = "<br>".join([
                f"<b>{a['adduct']}</b>: {a['obs_mz']:.4f} m/z ({a['ppm']:+.1f} ppm)"
                for a in assign["adducts"]
            ])
            hal_info = _detect_halogen_tag_8c(assign.get("formula", ""))
            hal_badge = (
                f"<div style='color:#7c3aed; font-weight:600; font-size:11px; margin-top:2px;'>"
                f"🔬 Isotope Signature: {hal_info}</div>"
            ) if hal_info else ""
            iso_badge = (
                f"<div style='color:#b45309; font-weight:600; font-size:11px; margin-top:2px;'>"
                f"⚠️ Isobaric Regioisomers / Byproducts ({assign['formula']})</div>"
            ) if assign.get("is_isobaric", False) else ""

            hover_cards_by_peak[p_id] = (
                f"<div style='border:2px solid {peak_color}; border-radius:10px; background:#ffffff; "
                f"padding:10px 12px; box-shadow:0 8px 20px rgba(0,0,0,0.18); font-family:sans-serif; "
                f"color:#111827; min-width:230px; max-width:395px; text-align:center;'>"
                f"<div style='margin-bottom:4px;'>"
                f"<span style='background:{r_style['bg']}; color:{r_style['fg']}; border:1px solid {r_style['border']}; "
                f"border-radius:5px; padding:2px 8px; font-size:10.5px; font-weight:700;'>{role}</span></div>"
                f"<div style='font-weight:700; font-size:13px; border-bottom:1px solid #e5e7eb; "
                f"padding-bottom:4px; margin-bottom:4px;'>"
                f"Peak {p_id} ({rt_apex:.1f}s / {rt_apex/60.0:.2f}m): {assign['display_name']}</div>"
                f"{iso_badge}{hal_badge}"
                f"<div style='font-size:11.5px; color:#374151; line-height:1.45; margin-top:4px; margin-bottom:6px;'>"
                f"<b>Formula:</b> {assign.get('formula', 'N/A')}<br>"
                f"{adduct_lines}<br>"
                f"<b>TIC Area:</b> {norm_area} &nbsp;|&nbsp; "
                f"<b>Confidence:</b> <span style='color:#047857; font-weight:700;'>{assign['confidence']:.1f}%</span>"
                f"</div>"
                f"<img src='data:image/png;base64,{b64_png}' "
                f"style='max-width:100%; height:auto; display:block; margin:0 auto; border-radius:4px;'/>"
                f"</div>"
            )
        else:
            dom_str = str(peak_row.get('dominant m/z', 'None'))
            dom_tokens = [x.strip() for x in dom_str.split(',') if x.strip() and x.strip() != 'None']
            top_ions_str = "<br>".join(dom_tokens[:4]) if dom_tokens else "N/A"

            hover_cards_by_peak[p_id] = (
                f"<div style='border:2px solid {peak_color}; border-radius:8px; background:#f9fafb; "
                f"padding:8px 12px; box-shadow:0 6px 16px rgba(0,0,0,0.15); font-family:sans-serif; "
                f"color:#1f2937; min-width:190px; text-align:left;'>"
                f"<div style='font-weight:700; font-size:12.5px; border-bottom:1px solid #d1d5db; "
                f"padding-bottom:3px; margin-bottom:5px;'>"
                f"Peak {p_id} ({rt_apex:.1f}s) — Unassigned Impurity</div>"
                f"<div style='font-size:11px; color:#4b5563; line-height:1.4;'>"
                f"<b>Norm. Area:</b> {norm_area}<br>"
                f"<b>Dominant m/z:</b><br>{top_ions_str}</div>"
                f"</div>"
            )

    chrom_fig = go.Figure()

    # Show subtle Cell 2 raw TIC overlay if SNIP shifted the trace noticeably
    if np.max(np.abs(raw_tic_signal - fitted_base_signal)) > 0.05 * y_max_plot:
        chrom_fig.add_trace(go.Scatter(
            x=time_arr, y=raw_tic_signal, mode='lines', name='raw TIC (Cell 2)',
            line=dict(color='rgba(100,116,139,0.55)', width=1.4, dash='dot'),
            hovertemplate='<b>Raw TIC (Cell 2)</b><br>RT=%{x:.2f}s<br>Signal=%{y:.3e}<extra></extra>'
        ))

    chrom_fig.add_trace(go.Scatter(
        x=time_arr, y=fitted_base_signal, mode='lines', name='crude TIC trace',
        line=dict(color='black', width=1.6),
        hovertemplate='<b>Crude TIC</b><br>RT=%{x:.2f}s<br>Signal=%{y:.3e}<extra></extra>'
    ))
    if convolved is not None:
        chrom_fig.add_trace(go.Scatter(
            x=time_arr, y=convolved, mode='lines', name='fitted reaction mixture',
            line=dict(color='red', width=1.4, dash='dash'),
            hovertemplate='<b>Fitted Reaction Mixture</b><br>RT=%{x:.2f}s<br>Mixture=%{y:.3e}<extra></extra>'
        ))

    if has_unmixed:
        for col_idx, (_, peak_row) in enumerate(detected_df.iterrows()):
            if col_idx >= unmixed.shape[1]:
                continue
            p_id = int(peak_row['peak_id'])
            peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])
            component = unmixed[:, col_idx]
            chrom_fig.add_trace(go.Scatter(
                x=time_arr, y=component, mode='lines', fill='tozeroy',
                line=dict(color=peak_color, width=1.0), opacity=0.55,
                name=_peak_label_for_legend(p_id),
                meta=p_id,
                hoverinfo='none'
            ))

    shapes, compact_annotations = [], []
    for col_idx, (_, peak_row) in enumerate(detected_df.iterrows()):
        p_id = int(peak_row['peak_id'])
        rt_apex = float(peak_row['retention_time'])
        c_idx = int(np.argmin(np.abs(time_arr - rt_apex)))
        comp_y = float(unmixed[c_idx, col_idx]) if (has_unmixed and col_idx < unmixed.shape[1]) else 0.0
        apex_y = max(float(fitted_base_signal[c_idx]), comp_y)
        peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])

        shapes.append(dict(
            type='line', x0=rt_apex, x1=rt_apex, y0=0, y1=apex_y,
            line=dict(color='dodgerblue', width=1.1, dash='dot'), opacity=0.55
        ))

        if 'peak_assignments' in globals() and p_id in peak_assignments:
            role = _extract_role_8c(peak_assignments[p_id])
            r_short = ROLE_STYLES_8C.get(role, {"short": role})["short"]
            badge_text = f"<b>P{p_id} ({r_short})</b>"
        else:
            badge_text = f"<b>P{p_id}</b>"

        compact_annotations.append(dict(
            x=rt_apex, y=apex_y + y_max_plot * 0.035, xref='x', yref='y',
            text=badge_text,
            showarrow=False,
            font=dict(size=10, color='#111827'),
            bgcolor='rgba(255,255,255,0.90)',
            bordercolor=peak_color,
            borderwidth=1.5,
            borderpad=2
        ))

        chrom_fig.add_trace(go.Scatter(
            x=[rt_apex], y=[apex_y],
            mode='markers',
            marker=dict(size=9, color=peak_color, line=dict(width=1.5, color='white')),
            showlegend=False,
            meta=p_id,
            hoverinfo='none'
        ))

    chrom_fig.update_layout(
        shapes=shapes,
        annotations=compact_annotations,
        title=dict(
            text=(
                f"Updated Crude Reaction Chromatogram & Peak Assignments (Prominence: {prom_val:.4f}) "
                f"— Hover over any peak for 2D Structure, Role & Isotope Details"
            ),
            font=dict(size=14.5, color='#111827')
        ),
        xaxis=dict(
            title='Retention Time (seconds)',
            range=X_RANGE,
            rangeslider=dict(visible=True, thickness=0.10)
        ),
        yaxis=dict(
            title='Total Ion Current (Signal)',
            range=[0, y_max_plot * 1.18]
        ),
        legend=dict(
            orientation='h',
            x=0.5, y=-0.26,
            xanchor='center', yanchor='top',
            font=dict(size=10),
            bgcolor='rgba(255,255,255,0.9)',
            bordercolor='#d1d5db',
            borderwidth=1
        ),
        margin=dict(l=MARGIN_L, r=MARGIN_R, t=MARGIN_T, b=MARGIN_B),
        template='plotly_white',
        width=FIG_WIDTH_PX,
        height=FIG_HEIGHT_PX,
        hovermode='closest'
    )

    globals()['_last_chromatogram_figure'] = chrom_fig

    # Refresh static PNG cache for Cell 10 Multimodal Vision QC with proportional banner spacing
    try:
        qc_fig = go.Figure(chrom_fig)
        qc_images = []
        qc_annotations = list(compact_annotations)
        n_struct = len(qc_struct_images)
        x_span = X_RANGE[1] - X_RANGE[0]
        if n_struct == 1:
            banner_xs = [(X_RANGE[0] + X_RANGE[1]) / 2.0]
        elif n_struct > 1:
            banner_xs = np.linspace(X_RANGE[0] + 0.08 * x_span, X_RANGE[1] - 0.08 * x_span, n_struct)
        else:
            banner_xs = []

        for idx_p, (p_id, struct_img) in enumerate(sorted(qc_struct_images.items())):
            bx = float(banner_xs[idx_p])
            by = y_max_plot * 1.48
            assign = peak_assignments[p_id]
            role = _extract_role_8c(assign)
            r_short = ROLE_STYLES_8C.get(role, {"short": role})["short"]
            peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])
            qc_annotations.append(dict(
                x=bx, y=by + y_max_plot * 0.04, xref='x', yref='y',
                text=f"<b>P{p_id} [{r_short}]: {assign['display_name'][:26]}</b> ({assign['confidence']:.0f}%)",
                showarrow=False, font=dict(size=8.5, color='#111827'),
                bgcolor='white', bordercolor=peak_color, borderwidth=1.5
            ))
            qc_images.append(dict(
                source=struct_img, xref='x', yref='y',
                x=bx, y=by,
                sizex=x_span / (max(n_struct, 1) + 1.2),
                sizey=y_max_plot * 0.32,
                xanchor='center', yanchor='top', layer='above'
            ))
        qc_fig.update_layout(
            yaxis=dict(title='Total Ion Current (Signal)', range=[0, y_max_plot * 1.62]),
            annotations=qc_annotations,
            images=qc_images
        )
        globals().setdefault('_notebook_figure_registry', {})['deconvoluted_chromatogram_with_structures'] = qc_fig.to_image(format='png', scale=2)
    except Exception:
        pass

    fig_html = chrom_fig.to_html(include_plotlyjs='cdn', full_html=False, div_id=div_id)
    cards_json = json.dumps({str(k): v for k, v in hover_cards_by_peak.items()})

    return f"""
    <div style="position:relative; width:{FIG_WIDTH_PX}px; margin: 12px 0;">
        {fig_html}
        <div id="{div_id}_tooltip" style="display:none; position:fixed; pointer-events:none; z-index:99999; transition: opacity 0.08s ease-in-out;"></div>
    </div>
    <script>
    (function() {{
        var cards = {cards_json};
        var gd = document.getElementById('{div_id}');
        var tooltip = document.getElementById('{div_id}_tooltip');

        function posTip(e) {{
            var x = e.clientX + 18;
            var y = e.clientY - 30;
            var rect = tooltip.getBoundingClientRect();
            if (x + rect.width > window.innerWidth - 15) x = e.clientX - rect.width - 18;
            if (y + rect.height > window.innerHeight - 15) y = window.innerHeight - rect.height - 15;
            if (y < 10) y = 10;
            tooltip.style.left = x + 'px';
            tooltip.style.top = y + 'px';
        }}

        function attachHoverEvents() {{
            if (!gd || typeof gd.on !== 'function') {{
                setTimeout(attachHoverEvents, 150);
                return;
            }}
            gd.on('plotly_hover', function(data) {{
                if (!data || !data.points || data.points.length === 0) return;
                var p_id = data.points[0].data.meta;
                if (p_id && cards[String(p_id)]) {{
                    tooltip.innerHTML = cards[String(p_id)];
                    tooltip.style.display = 'block';
                    if (data.event) posTip(data.event);
                }}
            }});
            gd.on('plotly_unhover', function() {{
                tooltip.style.display = 'none';
            }});
            gd.addEventListener('mousemove', function(e) {{
                if (tooltip.style.display === 'block') posTip(e);
            }});
        }}
        attachHoverEvents();
    }})();
    </script>
    """

In [ ]:
#@title 8D. Crude Reaction Gemini Agent Tools (Synthetic RDKit Fragmenter, Conversion, PubChem, ChEMBL, MassBank & Replotting)

def get_distilled_peak_details(peak_number: int) -> dict:
    """Retrieves background-subtracted ions, neutral losses, and contaminant flags (TPPO, DIPEA, Et3N, PEG) for a 1-based intensity-ranked peak."""
    if not SPECTRAL_STATE or "peaks" not in SPECTRAL_STATE:
        prepare_spectral_state()
    peaks = SPECTRAL_STATE.get("peaks", [])
    if not 1 <= peak_number <= len(peaks):
        return {"error": f"Distilled peak {peak_number} not found (valid range: 1..{len(peaks)})."}
    peak = peaks[peak_number - 1]
    return {
        "peak_number": peak_number,
        "rt_min": peak["rt_min"],
        "rt_sec": round(peak["rt_min"] * 60.0, 1),
        "ions": peak["ions"],
        "artifacts": peak["artifacts"],
        "neutral_losses": peak["neutral_losses"],
        "candidates": peak["candidates"]
    }


def search_pubchem_by_mass_or_formula(
    observed_mh_mz: float = 0.0,
    molecular_formula: str = "",
    compound_name: str = "",
    mass_window_da: float = 0.015,
    max_results: int = 8
) -> dict:
    """
    Searches PubChem for synthetic intermediates, byproducts, or reagents by compound_name,
    molecular_formula, OR observed [M+H]+ m/z, and pre-renders 2D structure SVGs (`image_token`).
    """
    try:
        max_results = max(1, min(int(max_results), 8))
        compounds = []
        if compound_name.strip():
            compounds = pcp.get_compounds(compound_name.strip(), 'name')[:max_results]
        elif molecular_formula.strip():
            compounds = pcp.get_compounds(molecular_formula.strip(), 'formula')[:max_results]
        elif observed_mh_mz > 0:
            neutral_mass = observed_mh_mz - 1.007276
            low_m = round(neutral_mass - mass_window_da, 4)
            high_m = round(neutral_mass + mass_window_da, 4)
            pug_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/exact_mass/range/{low_m}/{high_m}/cids/JSON"
            req = urllib.request.Request(pug_url, headers={"User-Agent": "Colab-LCMS-Agent/1.0"})
            with urllib.request.urlopen(req, timeout=10) as resp:
                cid_data = json.loads(resp.read().decode('utf-8'))
            cids = cid_data.get("IdentifierList", {}).get("CID", [])[:max_results]
            if cids:
                compounds = pcp.get_compounds(cids, 'cid')

        hits = []
        for c in compounds[:max_results]:
            smi = getattr(c, 'isomeric_smiles', None) or getattr(c, 'canonical_smiles', None)
            if not smi:
                continue
            mol, clean_smi, _ = safe_mol_from_smiles(smi)
            if not mol:
                continue
            em = Descriptors.ExactMolWt(mol)
            mh_calc = em + 1.007276
            ppm_err = ((observed_mh_mz - mh_calc) / mh_calc * 1e6) if observed_mh_mz > 0 else 0.0
            synonyms = (getattr(c, 'synonyms', None) or [])[:5]
            pref_label = synonyms[0] if synonyms else (c.iupac_name or f"CID {c.cid}")
            img_token = _register_mol_svg(mol, label=f"{pref_label} (CID {c.cid})")

            hits.append({
                "cid": c.cid,
                "preferred_name": pref_label,
                "synonyms": synonyms,
                "iupac_name": c.iupac_name,
                "formula": c.molecular_formula,
                "smiles": clean_smi,
                "exact_mass": round(em, 4),
                "calculated_[M+H]+_mz": round(mh_calc, 4),
                "ppm_error_vs_observed": round(ppm_err, 2),
                "rdkit_cLogP": round(Crippen.MolLogP(mol), 3),
                "TPSA_A2": round(rdMolDescriptors.CalcTPSA(mol), 2),
                "image_token": img_token
            })
        return {
            "query_mh_mz": observed_mh_mz,
            "query_formula": molecular_formula,
            "query_name": compound_name,
            "pubchem_hits": hits
        }
    except Exception as e:
        return {"error": f"PubChem search failed: {e}"}


def search_chembl_metabolites_and_isomers(
    molecular_formula: str = "",
    smiles: str = "",
    max_results: int = 8
) -> dict:
    """
    Queries the EBI ChEMBL REST API by molecular formula or canonical SMILES
    to retrieve synthetic scaffolds, regioisomers, ALogP, PSA, RDKit cLogP, and 2D SVG `image_token`s.
    """
    try:
        max_results = max(1, min(int(max_results), 8))
        hits = []
        if smiles.strip():
            mol, clean_smi, err = safe_mol_from_smiles(smiles)
            if not mol:
                return {"error": err}
            inchikey = Chem.MolToInchiKey(mol)
            url = f"https://www.ebi.ac.uk/chembl/api/data/molecule.json?molecule_structures__standard_inchi_key={inchikey}"
        elif molecular_formula.strip():
            url = f"https://www.ebi.ac.uk/chembl/api/data/molecule.json?molecule_properties__full_molformula={urllib.parse.quote(molecular_formula.strip())}&limit={max_results}"
        else:
            return {"error": "Provide either `molecular_formula` or `smiles`."}

        req = urllib.request.Request(url, headers={"Accept": "application/json", "User-Agent": "Colab-LCMS-Agent/1.0"})
        with urllib.request.urlopen(req, timeout=10) as resp:
            data = json.loads(resp.read().decode('utf-8'))

        for mol_entry in data.get("molecules", [])[:max_results]:
            struct = mol_entry.get("molecule_structures") or {}
            props = mol_entry.get("molecule_properties") or {}
            can_smi = struct.get("canonical_smiles")
            if not can_smi:
                continue

            r_mol, clean_s, _ = safe_mol_from_smiles(can_smi)
            if not r_mol:
                continue
            rdkit_clogp = round(Crippen.MolLogP(r_mol), 3)
            em = round(Descriptors.ExactMolWt(r_mol), 4)
            pref_name = mol_entry.get("pref_name") or mol_entry.get("molecule_chembl_id")
            img_token = _register_mol_svg(r_mol, label=f"{pref_name} ({mol_entry.get('molecule_chembl_id')})")

            hits.append({
                "chembl_id": mol_entry.get("molecule_chembl_id"),
                "pref_name": pref_name,
                "formula": props.get("full_molformula"),
                "exact_mass": em,
                "expected_[M+H]+_mz": round(em + 1.007276, 4),
                "chembl_alogp": props.get("alogp"),
                "rdkit_clogp": rdkit_clogp,
                "psa": props.get("psa"),
                "canonical_smiles": clean_s,
                "image_token": img_token
            })

        return {
            "query_formula": molecular_formula,
            "query_smiles": smiles,
            "chembl_candidates": hits,
            "retention_note": "In Reverse-Phase (C18) HPLC, more polar compounds (lower cLogP / higher TPSA, e.g. carboxylic acid hydrolysis byproducts) elute earlier than less polar esters or homocoupling dimers."
        }
    except Exception as e:
        return {"error": f"ChEMBL query failed: {e}"}


def query_massbank_reference_spectrum(
    compound_name_or_smiles: str = "",
    exact_mass: float = 0.0,
    max_records: int = 5
) -> dict:
    """Queries MassBank Europe REST API by InChIKey, compound name, or exact mass for reference ESI+ MS2 spectra."""
    try:
        max_records = max(1, min(int(max_records), 8))
        params = {"ion_mode": "POSITIVE", "ms_type": "MS2"}
        mol, clean_smi, _ = safe_mol_from_smiles(compound_name_or_smiles)
        if mol:
            params["inchi_key"] = Chem.MolToInchiKey(mol)
        elif compound_name_or_smiles.strip():
            params["compound_name"] = compound_name_or_smiles.strip()
        elif exact_mass > 0:
            params["exact_mass"] = f"{exact_mass:.4f}"
            params["mass_tolerance"] = "0.02"
        else:
            return {"error": "Provide a SMILES string, compound name, or exact_mass."}

        query_str = urllib.parse.urlencode(params)
        search_url = f"https://api.massbank.eu/records/search?{query_str}"
        req = urllib.request.Request(search_url, headers={"Accept": "application/json", "User-Agent": "Colab-LCMS-Agent/1.0"})

        records_summary = []
        with urllib.request.urlopen(req, timeout=10) as resp:
            search_res = json.loads(resp.read().decode('utf-8'))

        items = search_res if isinstance(search_res, list) else search_res.get("results", search_res.get("data", []))
        for item in items[:max_records]:
            accession = item if isinstance(item, str) else item.get("accession")
            if not accession:
                continue
            rec_url = f"https://api.massbank.eu/records/{urllib.parse.quote(str(accession), safe='')}"
            req_rec = urllib.request.Request(rec_url, headers={"Accept": "application/json", "User-Agent": "Colab-LCMS-Agent/1.0"})
            with urllib.request.urlopen(req_rec, timeout=8) as r_resp:
                rec_data = json.loads(r_resp.read().decode('utf-8'))

            peaks_raw = rec_data.get("peak", {}).get("peak", {}).get("values", [])
            sorted_peaks = sorted(peaks_raw, key=lambda x: x.get("rel", x.get("intensity", 0)), reverse=True)[:10]
            records_summary.append({
                "accession": accession,
                "title": rec_data.get("record_title"),
                "precursor_mz": rec_data.get("ms_focused_ion", {}).get("precursor_m/z"),
                "top_ms2_fragments": [
                    {"mz": round(float(p.get("mz", 0)), 4), "rel_intensity": p.get("rel", p.get("intensity", 0))}
                    for p in sorted_peaks
                ]
            })

        return {
            "query": compound_name_or_smiles or exact_mass,
            "massbank_records_found": len(records_summary),
            "reference_spectra": records_summary
        }
    except Exception as e:
        return {
            "warning": f"Live MassBank REST lookup returned: {e}. Use `disambiguate_isobaric_peak_isomers` for deterministic in-source MS1 fragment & cLogP matching."
        }


def search_chemical_databases(
    molecular_formula: str = "",
    smiles: str = "",
    observed_mh_mz: float = 0.0
) -> dict:
    """Runs PubChem, ChEMBL, and MassBank searches in one call and registers 2D structure SVG tokens."""
    pubchem_res = search_pubchem_by_mass_or_formula(observed_mh_mz=observed_mh_mz, molecular_formula=molecular_formula)
    chembl_res = (
        search_chembl_metabolites_and_isomers(molecular_formula=molecular_formula, smiles=smiles)
        if (molecular_formula.strip() or smiles.strip())
        else {"note": "Provide molecular_formula or smiles to query ChEMBL."}
    )
    neutral_mass = (observed_mh_mz - 1.007276) if observed_mh_mz > 1.0 else 0.0
    massbank_res = (
        query_massbank_reference_spectrum(compound_name_or_smiles=smiles, exact_mass=neutral_mass)
        if (smiles.strip() or neutral_mass > 0)
        else {"note": "Provide smiles or observed_mh_mz to query MassBank."}
    )
    return {"pubchem": pubchem_res, "chembl": chembl_res, "massbank": massbank_res}


def disambiguate_isobaric_peak_isomers(
    peak_id: int,
    candidate_names: list[str],
    candidate_smiles: list[str]
) -> dict:
    """
    Resolves isobaric synthetic regioisomers or byproducts for a given `peak_id`
    by checking raw MS1 scans for functional-group diagnostic neutral losses
    (MeOH loss -32.026 Da from methyl esters, H2O loss -18.011 Da & CO2 loss -43.990 Da
    from carboxylic acids, isobutene loss -56.063 Da from t-Bu/Boc groups, and 79Br/81Br or 35Cl/37Cl
    M+2 isotope doublets) alongside RDKit Crippen cLogP & TPSA for reverse-phase C18 elution order.
    """
    if len(candidate_names) != len(candidate_smiles):
        return {"error": "candidate_names and candidate_smiles must have the same length."}

    isomer_reports = []
    for name, smi in zip(candidate_names, candidate_smiles):
        mol, clean_smi, err = safe_mol_from_smiles(smi)
        if not mol:
            isomer_reports.append({"name": name, "error": err})
            continue

        em = Descriptors.ExactMolWt(mol)
        formula = rdMolDescriptors.CalcMolFormula(mol)
        mh_mz = round(em + 1.007276, 4)
        clogp = round(Crippen.MolLogP(mol), 3)
        tpsa = round(rdMolDescriptors.CalcTPSA(mol), 2)
        img_token = _register_mol_svg(mol, label=f"Candidate: {name}")

        frag_res = generate_in_silico_fragments(clean_smi, check_peak_id=peak_id)
        observed_diagnostic_frags = [
            {
                "fragment_name": f["fragment_name"],
                "expected_mz": f["expected_esi_pos_mz"],
                "neutral_loss_Da": f["neutral_loss_from_parent_Da"],
                "observed_in_peak": f["peak_observation"],
                "fragment_image_token": f["image_token"]
            }
            for f in frag_res.get("generated_fragments", [])
            if f.get("present_in_peak")
        ]

        synthetic_checks = []
        for loss_label, loss_da in [
            ("Methyl ester MeOH loss (-32.0262 Da -> acylium [M+H-CH3OH]+)", 32.026215),
            ("Carboxylic acid dehydration (-18.0106 Da -> acylium [M+H-H2O]+)", 18.010565),
            ("Carboxylic acid decarboxylation (-43.9898 Da -> [M+H-CO2]+)", 43.989829),
            ("t-Bu / Boc isobutene loss (-56.0626 Da -> [M+H-C4H8]+)", 56.062600),
            ("Boc group loss (-100.0524 Da -> [M+H-Boc]+)", 100.052429),
        ]:
            target_frag_mz = round(mh_mz - loss_da, 4)
            if target_frag_mz > 60.0:
                scan_hit = check_mz_in_peak(peak_id, target_frag_mz, tolerance_da=0.03)
                synthetic_checks.append({
                    "pathway": loss_label,
                    "target_fragment_mz": target_frag_mz,
                    "detected_in_raw_ms1": scan_hit.get("found_in_raw_scans", False),
                    "max_hit": scan_hit.get("max_observed_hit")
                })

        isomer_reports.append({
            "candidate_name": name,
            "canonical_smiles": clean_smi,
            "formula": formula,
            "parent_[M+H]+_mz": mh_mz,
            "rdkit_cLogP": clogp,
            "TPSA_A2": tpsa,
            "structure_image_token": img_token,
            "observed_in_silico_fragments": observed_diagnostic_frags,
            "diagnostic_synthetic_neutral_loss_checks": synthetic_checks
        })

    return {
        "peak_id": peak_id,
        "isobaric_comparison": isomer_reports,
        "interpretation_guide": (
            "1. Inspect `diagnostic_synthetic_neutral_loss_checks` and `observed_in_silico_fragments` for functional-group-specific cleavage hits. "
            "2. In Reverse-Phase C18 HPLC, more polar regioisomers/byproducts (lower `rdkit_cLogP`, higher `TPSA_A2`, free -OH or -COOH groups) "
            "elute EARLIER than less polar isomers (higher `rdkit_cLogP`, lower `TPSA_A2`, esters/ethers/biaryl dimers)."
        )
    }


def assign_peaks_and_replot_chromatogram(
    peak_ids: list[int],
    compound_names: list[str],
    smiles_list: list[str],
    classifications: list[str] = None
) -> dict:
    """
    Assigns one or more crude reaction peaks to specific structures & synthetic roles
    ('Reactant', 'Reagent', 'Product', 'Byproduct'), updates `detected_df` and `peak_assignments`
    in place, and generates an updated interactive Plotly chromatogram with role badges and 2D hover cards.
    Returns a `plot_token` (e.g. `[[PLOTLY_CHROM_1]]`) that you MUST place on its own line in your response.
    """
    global detected_df, peak_assignments

    if 'detected_df' not in globals() or 'chrom' not in globals() or chrom is None:
        return {"error": "detected_df or chrom not initialized. Run Section 2 Cell 4 first."}

    if not (len(peak_ids) == len(compound_names) == len(smiles_list)):
        return {"error": "peak_ids, compound_names, and smiles_list must have the exact same length."}

    if not classifications or len(classifications) != len(peak_ids):
        classifications = ["Byproduct"] * len(peak_ids)

    adduct_defs = globals().get('esi_pos_adducts', [
        ("[M+H]+",   1, 1.007276,  1),
        ("[M+NH4]+", 1, 18.033823, 1),
        ("[M+Na]+",  1, 22.989218, 1),
        ("[M+K]+",   1, 38.963158, 1),
        ("[2M+H]+",  2, 1.007276,  1),
    ])

    updated_summary = []
    for p_id, name, smi, cls_label in zip(peak_ids, compound_names, smiles_list, classifications):
        p_id = int(p_id)
        mol, clean_smi, err = safe_mol_from_smiles(smi)
        if not mol:
            updated_summary.append({"peak_id": p_id, "status": "failed", "error": err})
            continue

        em = round(Descriptors.ExactMolWt(mol), 4)
        formula = rdMolDescriptors.CalcMolFormula(mol)

        observed_adducts = []
        for adduct_name, mult, shift, z in adduct_defs:
            calc_mz = (em * mult + shift) / z
            hit = check_mz_in_peak(p_id, calc_mz, tolerance_da=0.03)
            if hit.get("found_in_raw_scans") and hit.get("max_observed_hit"):
                best_hit = hit["max_observed_hit"]
                observed_adducts.append({
                    "adduct": adduct_name,
                    "obs_mz": best_hit["observed_mz"],
                    "calc_mz": calc_mz,
                    "ppm": best_hit["ppm_error"],
                    "intensity": best_hit["intensity"]
                })

        if not observed_adducts:
            calc_mh = em + 1.007276
            observed_adducts.append({
                "adduct": "[M+H]+",
                "obs_mz": round(calc_mh, 4),
                "calc_mz": calc_mh,
                "ppm": 0.0,
                "intensity": 1e6
            })
        else:
            observed_adducts = sorted(observed_adducts, key=lambda x: x["intensity"], reverse=True)

        row_mask = detected_df['peak_id'] == p_id
        dom_str = detected_df.loc[row_mask, 'dominant m/z'].values[0] if row_mask.any() else "None"
        dom_parsed = parse_dominant_mz_string(dom_str) if 'parse_dominant_mz_string' in globals() else []

        best_group = {
            "exact_mass": em,
            "formula": formula,
            "mols": [mol],
            "adducts": observed_adducts
        }
        if 'compute_confidence_from_dominant_mz' in globals():
            conf_score, evidence_str = compute_confidence_from_dominant_mz(
                best_group, dom_parsed, globals().get('peak_ms1_spectra', {}).get(p_id, {})
            )
        else:
            conf_score, evidence_str = 90.0, f"Assigned ({observed_adducts[0]['ppm']:+.1f} ppm)"

        peak_assignments[p_id] = {
            "display_name": name,
            "classification": cls_label,
            "full_assignment": f"{name} ({cls_label})",
            "is_isobaric": False,
            "formula": formula,
            "mols": [mol],
            "adducts": observed_adducts,
            "confidence": conf_score,
            "evidence": evidence_str
        }

        top_a = observed_adducts[0]
        if row_mask.any():
            detected_df.loc[row_mask, 'Assignment'] = (
                f"{name} ({cls_label}) [{top_a['adduct']}: {top_a['obs_mz']:.4f} m/z ({top_a['ppm']:+.1f} ppm)]"
            )
            detected_df.loc[row_mask, 'Confidence (%)'] = f"{conf_score:.1f}% ({evidence_str})"

        struct_token = _register_mol_svg(mol, label=f"Peak {p_id} [{cls_label}]: {name}")
        updated_summary.append({
            "peak_id": p_id,
            "assigned_name": name,
            "classification": cls_label,
            "formula": formula,
            "primary_adduct": top_a["adduct"],
            "observed_mz": top_a["obs_mz"],
            "ppm_error": top_a["ppm"],
            "confidence_pct": conf_score,
            "evidence": evidence_str,
            "structure_image_token": struct_token
        })

    unique_div_id = f"crude_rxn_replot_{uuid.uuid4().hex[:6]}"
    interactive_html = build_interactive_chromatogram_html(div_id=unique_div_id)

    plot_idx = sum(1 for k in _rendered_svg_registry if k.startswith("[[PLOTLY_CHROM_")) + 1
    plot_token = f"[[PLOTLY_CHROM_{plot_idx}]]"
    _rendered_svg_registry[plot_token] = interactive_html

    return {
        "status": "success",
        "updated_peaks": updated_summary,
        "plot_token": plot_token,
        "rendering_instruction": (
            f"CRITICAL: Place the exact token {plot_token} on its own line in your response "
            "to render the updated interactive Crude Reaction Plotly chromatogram!"
        )
    }

In [ ]:
#@title 11. Unified Interactive Chromatogram + Gemini LC-MS Copilot Workbench
import io
import re
import json
import uuid
import html
import math
import base64
import sys
import functools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, Markdown
from rdkit.Chem import Draw
from google.colab import userdata
from google import genai
from google.genai import types

if 'detected_df' not in globals() or 'chrom' not in globals() or chrom is None:
    raise RuntimeError("⚠️ Required data (`detected_df`, `chrom`) not found. Run Cells 2, 3, and 4 first.")

if 'to_plotly_color' not in globals():
    def to_plotly_color(color_val):
        rgba = mcolors.to_rgba(color_val)
        return f"rgb({int(rgba[0]*255)}, {int(rgba[1]*255)}, {int(rgba[2]*255)})"

_rendered_svg_registry = globals().setdefault('_rendered_svg_registry', {})
SPECTRAL_STATE = globals().setdefault('SPECTRAL_STATE', {})

# --- 1. Universal JSON Sanitizer for Gemini Tool Outputs (Fixes np.bool_ / np.floating Errors) ---
def _sanitize_for_json(obj):
    if obj is None or isinstance(obj, str):
        return obj
    if isinstance(obj, (bool, np.bool_)):
        return bool(obj)
    if isinstance(obj, (int, np.integer)):
        return int(obj)
    if isinstance(obj, (float, np.floating)):
        val = float(obj)
        return None if (math.isnan(val) or math.isinf(val)) else val
    if isinstance(obj, np.ndarray):
        return [_sanitize_for_json(x) for x in obj.tolist()]
    if isinstance(obj, dict):
        return {str(k): _sanitize_for_json(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [_sanitize_for_json(x) for x in obj]
    return str(obj)

def _wrap_json_safe_tool(fn):
    @functools.wraps(fn)
    def _wrapped(*args, **kwargs):
        return _sanitize_for_json(fn(*args, **kwargs))
    return _wrapped

# --- 2. Role Styling & Safe Helpers ---
WB_ROLE_STYLES = {
    "Byproduct":         {"bg": "rgba(244, 63, 94, 0.16)",  "fg": "#fda4af", "border": "rgba(244, 63, 94, 0.45)",  "short": "Byp"},
    "Reactant":          {"bg": "rgba(56, 189, 248, 0.16)", "fg": "#7dd3fc", "border": "rgba(56, 189, 248, 0.45)", "short": "SM"},
    "Starting Material": {"bg": "rgba(56, 189, 248, 0.16)", "fg": "#7dd3fc", "border": "rgba(56, 189, 248, 0.45)", "short": "SM"},
    "Reagent":           {"bg": "rgba(250, 204, 21, 0.16)", "fg": "#fde047", "border": "rgba(250, 204, 21, 0.45)", "short": "Rgt"},
    "Product":           {"bg": "rgba(34, 197, 94, 0.18)",  "fg": "#86efac", "border": "rgba(34, 197, 94, 0.48)",  "short": "Prod"},
    "Target Product":    {"bg": "rgba(34, 197, 94, 0.18)",  "fg": "#86efac", "border": "rgba(34, 197, 94, 0.48)",  "short": "Prod"},
}

def _wb_extract_role(assign_dict):
    if not assign_dict:
        return "Unassigned"
    for key in ("role", "classification"):
        val = assign_dict.get(key)
        if val in WB_ROLE_STYLES:
            return val
    full_str = str(assign_dict.get("full_assignment", ""))
    for role_key in ("Byproduct", "Reactant", "Starting Material", "Reagent", "Target Product", "Product"):
        if f"({role_key})" in full_str:
            return role_key
    return "Unassigned"

def _wb_halogen_tag(formula_str: str) -> str:
    if not formula_str:
        return ""
    tags = []
    br_m = re.search(r"Br(\d*)", formula_str)
    cl_m = re.search(r"Cl(\d*)", formula_str)
    if br_m:
        tags.append(f"⁷⁹Br/⁸¹Br ({int(br_m.group(1) or 1)}×Br, M+2 ≈1:1)")
    if cl_m:
        tags.append(f"³⁵Cl/³⁷Cl ({int(cl_m.group(1) or 1)}×Cl, M+2 ≈3:1)")
    return " | ".join(tags)

def _wb_peak_label(p_id: int) -> str:
    if 'peak_assignments' in globals() and p_id in peak_assignments:
        a_info = peak_assignments[p_id]
        role = _wb_extract_role(a_info)
        r_short = WB_ROLE_STYLES.get(role, {"short": role})["short"]
        iso_short = " [Isobaric]" if a_info.get("is_isobaric", False) else ""
        adduct_summary = ", ".join([f"{x['adduct']} ({x['obs_mz']:.2f})" for x in a_info['adducts'][:2]])
        return f"Peak {p_id} [{r_short}]: {a_info['display_name']}{iso_short} [{adduct_summary}] ({a_info['confidence']:.0f}%)"
    dom_rows = detected_df.loc[detected_df['peak_id'] == p_id, 'dominant m/z']
    if len(dom_rows):
        dom_str = dom_rows.values[0]
        first_ion = str(dom_str).split(',')[0] if dom_str != "None" else "N/A"
        return f"Peak {p_id}: Unassigned (Base m/z: {first_ion})"
    return f"Peak {p_id}: Unassigned"

def _wrap_in_iframe(body_html: str, height_px: int = 600, bg_color: str = "#070d1a", scroll_y: str = "hidden") -> str:
    """Wraps Plotly + JS inside a self-contained iframe so Colab ALWAYS executes scripts cleanly."""
    full_doc = (
        "<!DOCTYPE html><html><head><meta charset='utf-8'>"
        f"<style>html,body{{margin:0;padding:0;width:100%;height:100%;background:{bg_color};"
        f"overflow-x:hidden;overflow-y:{scroll_y};font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;}}</style>"
        f"</head><body>{body_html}</body></html>"
    )
    escaped = html.escape(full_doc, quote=True)
    return (
        f"<iframe srcdoc=\"{escaped}\" "
        f"style=\"width:100%; height:{height_px}px; border:1px solid rgba(148,163,184,0.20); "
        f"border-radius:10px; background:{bg_color}; display:block;\"></iframe>"
    )

# --- 3. Assemble JSON-Safe Tools & Gemini Client ---
_wb_client = globals().get('client') or genai.Client(api_key=userdata.get("KDD_GEMINI_API_KEY"))

def _collect_workbench_tools():
    candidate_names = [
        'get_user_candidate_molecules', 'get_detected_peaks_summary',
        'calculate_crude_reaction_conversion_summary', 'get_distilled_peak_details',
        'get_peak_details', 'check_mz_in_peak', 'find_mz_across_chromatogram',
        'check_peak_purity_and_coelution', 'verify_isotope_envelope_in_peak',
        'analyze_chromatographic_resolution_and_shape', 'elucidate_unknown_formula_from_mz',
        'diagnose_mass_delta_to_reaction_species', 'generate_in_silico_fragments',
        'analyze_structure_and_mass', 'generate_2d_structure_svg',
        'search_pubchem_by_mass_or_formula', 'search_chembl_metabolites_and_isomers',
        'query_massbank_reference_spectrum', 'search_chemical_databases',
        'update_lcms_plot', 'disambiguate_isobaric_peak_isomers',
        'assign_peaks_and_replot_chromatogram',
    ]
    tools, seen = [], set()
    for fn in globals().get('CELL6_TOOLS', []):
        if callable(fn) and fn.__name__ not in seen:
            tools.append(_wrap_json_safe_tool(fn))
            seen.add(fn.__name__)
    for name in candidate_names:
        fn = globals().get(name)
        if callable(fn) and name not in seen:
            tools.append(_wrap_json_safe_tool(fn))
            seen.add(name)
    return tools

wb_tools = _collect_workbench_tools()

def _wb_system_instruction() -> str:
    cand_md = get_user_candidate_molecules() if 'get_user_candidate_molecules' in globals() else "Unavailable"
    peaks_md = get_detected_peaks_summary() if 'get_detected_peaks_summary' in globals() else detected_df.to_markdown(index=False)
    conv = calculate_crude_reaction_conversion_summary() if 'calculate_crude_reaction_conversion_summary' in globals() else {}
    role_json = json.dumps(_sanitize_for_json(conv.get("role_area_percentages", {})))
    sm_conv = conv.get("estimated_sm_conversion_pct", 0)
    sel_pct = conv.get("estimated_product_selectivity_among_reacting_species_pct", 0)
    return (
        "You are an expert medicinal and process chemistry LC-MS analytical copilot embedded directly "
        "beside the user's live interactive crude reaction chromatogram.\n\n"
        f"### Live Reaction Candidates:\n{cand_md}\n\n"
        f"### Live Deconvoluted Peaks (`detected_df`):\n{peaks_md}\n\n"
        f"### Live TIC Breakdown:\n{role_json} | Est. SM Conversion: {sm_conv}% | Est. Selectivity: {sel_pct}%\n\n"
        "### Operating Guidelines:\n"
        "1. Answer concisely and quantitatively with exact m/z values, ppm errors, RTs, and LCAP% areas.\n"
        "2. Whenever a tool returns `image_token`, `parent_image_token`, or `structure_image_token` (e.g. `[[MOL_IMG_1]]`), "
        "place that exact token on its own line so the 2D SVG card renders inline in the chat.\n"
        "3. When asked to show 2D structures of byproducts or reaction components, call `generate_2d_structure_svg(smiles, label=...)` "
        "or `analyze_structure_and_mass(...)` and embed each returned `[[MOL_IMG_N]]` token on its own line.\n"
        "4. When asked to reassign a peak or update roles, call `assign_peaks_and_replot_chromatogram` — the main "
        "chromatogram panel on the left refreshes automatically.\n"
        "5. Use `check_peak_purity_and_coelution`, `verify_isotope_envelope_in_peak`, `find_mz_across_chromatogram`, "
        "`analyze_chromatographic_resolution_and_shape`, `diagnose_mass_delta_to_reaction_species`, and "
        "`elucidate_unknown_formula_from_mz` whenever relevant.\n"
        "6. If the user asks for an Extracted Ion Chromatogram (XIC) or theoretical isotope bar overlay, call `update_lcms_plot`."
    )

# --- 4. Glassmorphism CSS ---
wb_css = widgets.HTML("""
<style>
.lcms-wb-root {
    width: 100% !important;
    background: linear-gradient(135deg, #0b1120 0%, #111827 55%, #161b33 100%);
    border: 1px solid rgba(56, 189, 248, 0.25);
    border-radius: 14px;
    padding: 14px 16px;
    box-sizing: border-box;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
    color: #e2e8f0;
}
.lcms-wb-panel {
    background: rgba(15, 23, 42, 0.72) !important;
    backdrop-filter: blur(12px);
    border: 1px solid rgba(148, 163, 184, 0.20) !important;
    border-radius: 12px !important;
    padding: 10px 12px !important;
    box-sizing: border-box !important;
}
.lcms-wb-chat-scroll {
    background: rgba(2, 6, 23, 0.55) !important;
    border: 1px solid rgba(56, 189, 248, 0.22) !important;
    border-radius: 10px !important;
    padding: 12px !important;
    color: #e2e8f0 !important;
    line-height: 1.55 !important;
    font-size: 13.5px !important;
}
.lcms-wb-chat-scroll p, .lcms-wb-chat-scroll li, .lcms-wb-chat-scroll h1,
.lcms-wb-chat-scroll h2, .lcms-wb-chat-scroll h3, .lcms-wb-chat-scroll h4,
.lcms-wb-chat-scroll span, .lcms-wb-chat-scroll strong {
    color: #e2e8f0 !important;
}
.lcms-wb-chat-scroll table {
    border-collapse: collapse;
    display: block;
    overflow-x: auto;
    margin: 8px 0;
    font-size: 12px;
}
.lcms-wb-chat-scroll th, .lcms-wb-chat-scroll td {
    border: 1px solid rgba(148, 163, 184, 0.30);
    padding: 5px 8px;
    color: #f1f5f9 !important;
}
.lcms-wb-chat-scroll th { background: rgba(30, 41, 59, 0.9); }
.lcms-wb-chat-scroll code {
    background: rgba(30, 41, 59, 0.9);
    color: #67e8f9 !important;
    padding: 1px 5px;
    border-radius: 4px;
}
.lcms-wb-chat-scroll pre {
    background: rgba(15, 23, 42, 0.95);
    padding: 10px;
    border-radius: 6px;
    overflow-x: auto;
}
.lcms-wb-root .widget-label { color: #cbd5e1 !important; font-weight: 600 !important; }
</style>
""")

# --- 5. Plot Builders (All Wrapped in Self-Contained Iframes) ---
def _build_banner_html() -> str:
    role_totals = {"Product": 0.0, "Reactant": 0.0, "Byproduct": 0.0, "Reagent": 0.0, "Unassigned": 0.0}
    for _, peak_row in detected_df.iterrows():
        p_id = int(peak_row['peak_id'])
        try:
            area_pct = float(str(peak_row.get('normalised area', '0%')).replace('%', '').strip())
        except ValueError:
            area_pct = 0.0
        r_key = _wb_extract_role(peak_assignments.get(p_id)) if 'peak_assignments' in globals() else "Unassigned"
        if r_key in ("Target Product", "Product"):
            role_totals["Product"] += area_pct
        elif r_key in ("Starting Material", "Reactant"):
            role_totals["Reactant"] += area_pct
        elif r_key == "Byproduct":
            role_totals["Byproduct"] += area_pct
        elif r_key == "Reagent":
            role_totals["Reagent"] += area_pct
        else:
            role_totals["Unassigned"] += area_pct

    rxn_denom = role_totals["Product"] + role_totals["Reactant"] + role_totals["Byproduct"]
    est_conv = ((role_totals["Product"] + role_totals["Byproduct"]) / rxn_denom * 100.0) if rxn_denom > 0 else 0.0
    est_sel = (role_totals["Product"] / rxn_denom * 100.0) if rxn_denom > 0 else 0.0

    return (
        "<div style='display:flex; flex-wrap:wrap; gap:8px; align-items:center; "
        "background:rgba(15,23,42,0.85); border:1px solid rgba(56,189,248,0.28); "
        "border-radius:10px; padding:8px 12px; margin-bottom:8px; font-family:sans-serif;'>"
        "<span style='font-weight:700; font-size:13px; color:#f8fafc; margin-right:4px;'>⚗️ Live TIC:</span>"
        f"<span style='background:rgba(34,197,94,0.18); color:#86efac; border:1px solid rgba(34,197,94,0.45); border-radius:6px; padding:2px 8px; font-size:11.5px; font-weight:700;'>🎯 Prod: {role_totals['Product']:.1f}%</span>"
        f"<span style='background:rgba(56,189,248,0.18); color:#7dd3fc; border:1px solid rgba(56,189,248,0.45); border-radius:6px; padding:2px 8px; font-size:11.5px; font-weight:700;'>🧪 SM: {role_totals['Reactant']:.1f}%</span>"
        f"<span style='background:rgba(244,63,94,0.18); color:#fda4af; border:1px solid rgba(244,63,94,0.45); border-radius:6px; padding:2px 8px; font-size:11.5px; font-weight:700;'>⚠️ Byp: {role_totals['Byproduct']:.1f}%</span>"
        f"<span style='background:rgba(250,204,21,0.16); color:#fde047; border:1px solid rgba(250,204,21,0.45); border-radius:6px; padding:2px 8px; font-size:11.5px; font-weight:700;'>🧰 Rgt: {role_totals['Reagent']:.1f}%</span>"
        f"<span style='background:rgba(148,163,184,0.18); color:#cbd5e1; border:1px solid rgba(148,163,184,0.4); border-radius:6px; padding:2px 8px; font-size:11.5px;'>❓ Unass: {role_totals['Unassigned']:.1f}%</span>"
        f"<span style='margin-left:auto; font-size:11.5px; color:#cbd5e1;'><b>Conv:</b> {est_conv:.1f}% &nbsp;|&nbsp; <b>Yield:</b> {est_sel:.1f}%</span>"
        "</div>"
    )

def _build_chromatogram_iframe_html(height_px: int = 605) -> str:
    time_arr = chrom.df[chrom.time_col].to_numpy(dtype=float)
    fitted_sig = chrom.df[chrom.int_col].to_numpy(dtype=float)
    raw_tic = chrom.df['Raw_TIC'].to_numpy(dtype=float) if 'Raw_TIC' in chrom.df.columns else fitted_sig
    t_span = max(float(time_arr.max() - time_arr.min()), 1.0)
    x_pad = 0.02 * t_span
    x_range = [max(0.0, float(time_arr.min()) - x_pad), float(time_arr.max()) + x_pad]

    unmixed = getattr(chrom, 'unmixed_chromatograms', None)
    has_unmixed = (unmixed is not None) and (unmixed.ndim == 2) and (unmixed.shape[1] > 0)
    convolved = np.sum(unmixed, axis=1) if has_unmixed else None
    y_max = float(max(np.max(raw_tic), np.max(fitted_sig), np.max(convolved) if convolved is not None else 0.0, 1.0))

    palette = ['#38bdf8', '#a78bfa', '#34d399', '#fbbf24', '#fb7185', '#22d3ee', '#f472b6', '#c084fc', '#4ade80', '#f97316']
    hover_cards = {}

    for _, peak_row in detected_df.iterrows():
        p_id = int(peak_row['peak_id'])
        rt_apex = float(peak_row['retention_time'])
        norm_area = peak_row.get('normalised area', 'N/A')
        p_col = palette[(p_id - 1) % len(palette)]

        if 'peak_assignments' in globals() and p_id in peak_assignments:
            assign = peak_assignments[p_id]
            role = _wb_extract_role(assign)
            r_style = WB_ROLE_STYLES.get(role, {"bg": "#334155", "fg": "#f1f5f9", "border": "#64748b", "short": role})

            mol_imgs = [Draw.MolToImage(m, size=(165, 130)) for m in assign["mols"]]
            if len(mol_imgs) > 1:
                struct_img = Image.new('RGBA', (sum(im.width for im in mol_imgs), mol_imgs[0].height), (255, 255, 255, 255))
                x_off = 0
                for im in mol_imgs:
                    struct_img.paste(im, (x_off, 0))
                    x_off += im.width
            else:
                struct_img = mol_imgs[0]

            buf = io.BytesIO()
            struct_img.save(buf, format='PNG')
            b64_png = base64.b64encode(buf.getvalue()).decode('utf-8')
            adduct_lines = "<br>".join([f"<b>{a['adduct']}</b>: {a['obs_mz']:.4f} m/z ({a['ppm']:+.1f} ppm)" for a in assign["adducts"]])
            hal_info = _wb_halogen_tag(assign.get("formula", ""))
            hal_badge = f"<div style='color:#c4b5fd; font-weight:600; font-size:11px; margin-top:2px;'>🔬 {hal_info}</div>" if hal_info else ""

            hover_cards[p_id] = (
                f"<div style='border:2px solid {p_col}; border-radius:10px; background:rgba(15,23,42,0.96); "
                f"padding:9px 11px; box-shadow:0 10px 26px rgba(0,0,0,0.55); font-family:sans-serif; "
                f"color:#f1f5f9; min-width:225px; max-width:340px; text-align:center;'>"
                f"<div style='margin-bottom:4px;'><span style='background:{r_style['bg']}; color:{r_style['fg']}; "
                f"border:1px solid {r_style['border']}; border-radius:5px; padding:2px 8px; font-size:10.5px; font-weight:700;'>{role}</span></div>"
                f"<div style='font-weight:700; font-size:12px; border-bottom:1px solid rgba(148,163,184,0.25); padding-bottom:4px; margin-bottom:4px;'>"
                f"Peak {p_id} ({rt_apex:.1f}s / {rt_apex/60.0:.2f}m): {html.escape(assign['display_name'])}</div>"
                f"{hal_badge}"
                f"<div style='font-size:11px; color:#cbd5e1; line-height:1.4; margin:4px 0 6px 0;'>"
                f"<b>Formula:</b> {assign.get('formula', 'N/A')}<br>{adduct_lines}<br>"
                f"<b>TIC Area:</b> {norm_area} &nbsp;|&nbsp; <b>Conf:</b> <span style='color:#4ade80; font-weight:700;'>{assign['confidence']:.1f}%</span></div>"
                f"<div style='background:#ffffff; border-radius:6px; padding:4px; display:inline-block;'>"
                f"<img src='data:image/png;base64,{b64_png}' style='max-width:100%; height:auto; display:block;'/></div></div>"
            )
        else:
            dom_tokens = [x.strip() for x in str(peak_row.get('dominant m/z', 'None')).split(',') if x.strip() and x.strip() != 'None']
            top_ions = "<br>".join(dom_tokens[:4]) if dom_tokens else "N/A"
            hover_cards[p_id] = (
                f"<div style='border:2px solid {p_col}; border-radius:8px; background:rgba(15,23,42,0.96); "
                f"padding:8px 12px; box-shadow:0 8px 20px rgba(0,0,0,0.45); font-family:sans-serif; color:#e2e8f0; min-width:190px;'>"
                f"<div style='font-weight:700; font-size:12px; border-bottom:1px solid #334155; padding-bottom:3px; margin-bottom:5px;'>"
                f"Peak {p_id} ({rt_apex:.1f}s) — Unassigned</div>"
                f"<div style='font-size:11px; color:#cbd5e1;'><b>Norm. Area:</b> {norm_area}<br><b>Top m/z:</b><br>{top_ions}</div></div>"
            )

    fig = go.Figure()
    if np.max(np.abs(raw_tic - fitted_sig)) > 0.05 * y_max:
        fig.add_trace(go.Scatter(
            x=time_arr, y=raw_tic, mode='lines', name='raw TIC (Cell 2)',
            line=dict(color='rgba(148,163,184,0.55)', width=1.4, dash='dot'),
            hovertemplate='<b>Raw TIC</b><br>RT=%{x:.2f}s<br>Signal=%{y:.3e}<extra></extra>'
        ))
    fig.add_trace(go.Scatter(
        x=time_arr, y=fitted_sig, mode='lines', name='crude TIC trace',
        line=dict(color='#f8fafc', width=1.7),
        hovertemplate='<b>Crude TIC</b><br>RT=%{x:.2f}s<br>Signal=%{y:.3e}<extra></extra>'
    ))
    if convolved is not None:
        fig.add_trace(go.Scatter(
            x=time_arr, y=convolved, mode='lines', name='fitted mixture',
            line=dict(color='#fb7185', width=1.4, dash='dash'),
            hovertemplate='<b>Fitted Mixture</b><br>RT=%{x:.2f}s<br>Signal=%{y:.3e}<extra></extra>'
        ))

    if has_unmixed:
        for col_idx, (_, peak_row) in enumerate(detected_df.iterrows()):
            if col_idx >= unmixed.shape[1]:
                continue
            p_id = int(peak_row['peak_id'])
            p_col = palette[(p_id - 1) % len(palette)]
            fig.add_trace(go.Scatter(
                x=time_arr, y=unmixed[:, col_idx], mode='lines', fill='tozeroy',
                line=dict(color=p_col, width=1.2), opacity=0.45,
                name=_wb_peak_label(p_id), meta=p_id, hoverinfo='none'
            ))

    shapes, annots = [], []
    for col_idx, (_, peak_row) in enumerate(detected_df.iterrows()):
        p_id = int(peak_row['peak_id'])
        rt_apex = float(peak_row['retention_time'])
        c_idx = int(np.argmin(np.abs(time_arr - rt_apex)))
        comp_y = float(unmixed[c_idx, col_idx]) if (has_unmixed and col_idx < unmixed.shape[1]) else 0.0
        apex_y = max(float(fitted_sig[c_idx]), comp_y)
        p_col = palette[(p_id - 1) % len(palette)]

        shapes.append(dict(type='line', x0=rt_apex, x1=rt_apex, y0=0, y1=apex_y, line=dict(color=p_col, width=1.1, dash='dot'), opacity=0.55))
        if 'peak_assignments' in globals() and p_id in peak_assignments:
            r_short = WB_ROLE_STYLES.get(_wb_extract_role(peak_assignments[p_id]), {"short": "?"})["short"]
            b_lbl = f"<b>P{p_id} ({r_short})</b>"
        else:
            b_lbl = f"<b>P{p_id}</b>"
        annots.append(dict(
            x=rt_apex, y=apex_y + y_max * 0.04, xref='x', yref='y',
            text=b_lbl, showarrow=False, font=dict(size=10, color='#f8fafc'),
            bgcolor='rgba(15,23,42,0.88)', bordercolor=p_col, borderwidth=1.4, borderpad=2
        ))
        fig.add_trace(go.Scatter(
            x=[rt_apex], y=[apex_y], mode='markers',
            marker=dict(size=9, color=p_col, line=dict(width=1.5, color='#0f172a')),
            showlegend=False, meta=p_id, hoverinfo='none'
        ))

    fig.update_layout(
        shapes=shapes, annotations=annots,
        title=dict(text="Crude Reaction Deconvolution — Hover Any Peak for 2D Structure & Isotopes", font=dict(size=13.5, color='#f8fafc')),
        xaxis=dict(title='Retention Time (seconds)', range=x_range, gridcolor='rgba(148,163,184,0.14)', rangeslider=dict(visible=True, thickness=0.09)),
        yaxis=dict(title='Total Ion Current', range=[0, y_max * 1.20], gridcolor='rgba(148,163,184,0.14)'),
        legend=dict(orientation='h', x=0.5, y=-0.30, xanchor='center', yanchor='top', font=dict(size=9.5, color='#cbd5e1'), bgcolor='rgba(15,23,42,0.75)', bordercolor='rgba(148,163,184,0.25)', borderwidth=1),
        margin=dict(l=60, r=18, t=44, b=175),
        paper_bgcolor='#070d1a', plot_bgcolor='#0b1324',
        font=dict(color='#e2e8f0'), autosize=True, height=height_px - 10, hovermode='closest'
    )

    globals()['_last_chromatogram_figure'] = fig
    try:
        qc_fig = go.Figure(fig)
        qc_fig.update_layout(width=1150, height=620)
        globals().setdefault('_notebook_figure_registry', {})['deconvoluted_chromatogram_with_structures'] = qc_fig.to_image(format='png', scale=2)
    except Exception:
        pass

    div_id = f"wb_chrom_{uuid.uuid4().hex[:8]}"
    fig_html = fig.to_html(include_plotlyjs='cdn', full_html=False, div_id=div_id, config={'responsive': True})
    cards_json = json.dumps({str(k): v for k, v in hover_cards.items()})

    body_html = (
        f"<div style='position:relative; width:100%; height:100%;'>{fig_html}"
        f"<div id='{div_id}_tooltip' style='display:none; position:fixed; pointer-events:none; z-index:99999;'></div></div>"
        "<script>(function(){"
        f"var cards={cards_json};"
        f"var gd=document.getElementById('{div_id}');"
        f"var tip=document.getElementById('{div_id}_tooltip');"
        "function posTip(e){"
        "  var x=e.clientX+14, y=e.clientY-20, r=tip.getBoundingClientRect();"
        "  if(x+r.width>window.innerWidth-10) x=Math.max(8, e.clientX-r.width-14);"
        "  if(y+r.height>window.innerHeight-10) y=Math.max(8, window.innerHeight-r.height-10);"
        "  if(y<8) y=8;"
        "  tip.style.left=x+'px'; tip.style.top=y+'px';"
        "}"
        "function init(){"
        "  if(!gd || typeof gd.on!=='function'){ setTimeout(init, 120); return; }"
        "  gd.on('plotly_hover', function(d){"
        "    if(!d || !d.points || !d.points.length) return;"
        "    var pid=d.points[0].data.meta;"
        "    if(pid && cards[String(pid)]){ tip.innerHTML=cards[String(pid)]; tip.style.display='block'; if(d.event) posTip(d.event); }"
        "  });"
        "  gd.on('plotly_unhover', function(){ tip.style.display='none'; });"
        "  gd.addEventListener('mousemove', function(e){ if(tip.style.display==='block') posTip(e); });"
        "}"
        "init();"
        "})();</script>"
    )
    return _wrap_in_iframe(body_html, height_px=height_px, bg_color="#070d1a", scroll_y="hidden")

def _build_ms1_iframe_html(selected_pids, height_px: int = 515) -> str:
    if not selected_pids:
        return _wrap_in_iframe("<div style='color:#94a3b8;padding:18px;font-size:13px;'>Select one or more peaks above to compare MS1 stick spectra and halogen M+2 doublets.</div>", height_px=height_px)
    palette = ['#38bdf8', '#a78bfa', '#34d399', '#fbbf24', '#fb7185', '#22d3ee', '#f472b6']
    n = len(selected_pids)
    fig_ms = make_subplots(
        rows=n, cols=1, shared_xaxes=True,
        vertical_spacing=min(0.14, 0.45 / max(n, 1)) if n > 1 else 0.0,
        subplot_titles=[_wb_peak_label(p) for p in selected_pids]
    )
    for row_i, p_id in enumerate(selected_pids, start=1):
        col = palette[(p_id - 1) % len(palette)]
        spec = globals().get('peak_ms1_spectra', {}).get(p_id, {})
        if spec:
            mzs = np.array(sorted(spec.keys()))
            ints = np.array([spec[m] for m in mzs])
            rel = (ints / ints.max() * 100.0) if ints.max() > 0 else ints
            sx = np.repeat(mzs, 3)
            sy = np.zeros(sx.shape, dtype=float)
            sy[1::3] = rel
            sy[2::3] = np.nan
            fig_ms.add_trace(go.Scatter(x=sx, y=sy, mode='lines', line=dict(color=col, width=1.4), hoverinfo='skip', showlegend=False), row=row_i, col=1)
            fig_ms.add_trace(go.Scatter(x=mzs, y=rel, mode='markers', marker=dict(color=col, size=5), customdata=ints, showlegend=False, hovertemplate=f"Peak {p_id}<br>m/z=%{{x:.4f}}<br>rel=%{{y:.1f}}%<br>int=%{{customdata:.2e}}<extra></extra>"), row=row_i, col=1)
            if 'peak_assignments' in globals() and p_id in peak_assignments:
                a_info = peak_assignments[p_id]
                f_str = a_info.get("formula", "")
                for add in a_info['adducts'][:3]:
                    fig_ms.add_vline(x=add['calc_mz'], line=dict(color=col, width=1.1, dash='dot'), annotation_text=add['adduct'], annotation_font_size=8.5, row=row_i, col=1)
                    if ("Br" in f_str or "Cl" in f_str) and add['adduct'] == "[M+H]+":
                        m2 = add['calc_mz'] + (1.99795 if "Br" in f_str else 1.99705)
                        lbl = "⁸¹Br M+2" if "Br" in f_str else "³⁷Cl M+2"
                        fig_ms.add_vline(x=m2, line=dict(color='#c084fc', width=1.2, dash='dash'), annotation_text=lbl, annotation_font_size=8.5, annotation_font_color='#c084fc', annotation_position="top left", row=row_i, col=1)
        fig_ms.update_yaxes(title_text="Rel. %", range=[0, 122], gridcolor='rgba(148,163,184,0.14)', row=row_i, col=1)
    fig_ms.update_xaxes(matches='x', title_text="m/z", gridcolor='rgba(148,163,184,0.14)', row=n, col=1)
    fig_ms.update_layout(
        height=max(320, 200 * n), autosize=True,
        paper_bgcolor='#070d1a', plot_bgcolor='#0b1324',
        font=dict(color='#e2e8f0'), margin=dict(l=55, r=18, t=48, b=42), hovermode='closest'
    )
    inner_html = fig_ms.to_html(include_plotlyjs='cdn', full_html=False, config={'responsive': True})
    return _wrap_in_iframe(inner_html, height_px=height_px, bg_color="#070d1a", scroll_y="auto")

# --- 6. Left Pane (Banner + ToggleButtons + Reliable Iframe Plot Host) ---
banner_html_widget = widgets.HTML(value=_build_banner_html())
view_toggle = widgets.ToggleButtons(
    options=['📈 Chromatogram', '🔬 MS1 & Halogens', '📊 XIC / Dashboard'],
    value='📈 Chromatogram',
    style={'button_width': 'auto'},
    layout=widgets.Layout(margin='0 0 8px 0')
)

ms_selector = widgets.SelectMultiple(
    options=[(_wb_peak_label(int(p)), int(p)) for p in sorted(detected_df['peak_id'].astype(int).tolist())],
    rows=min(4, max(3, len(detected_df))),
    layout=widgets.Layout(width='100%')
)
ms_controls_box = widgets.VBox([
    widgets.HTML("<div style='font-size:11.5px; color:#cbd5e1; margin-bottom:2px;'>Ctrl/Cmd-click peaks to stack MS1 stick spectra & halogen M+2 markers:</div>"),
    ms_selector
], layout=widgets.Layout(width='100%', display='none', margin='0 0 6px 0'))

plot_host = widgets.HTML(value="", layout=widgets.Layout(width='100%'))
_cached_dash_html = [""]

def render_active_left_view():
    mode = view_toggle.value
    if mode == '📈 Chromatogram':
        ms_controls_box.layout.display = 'none'
        plot_host.value = _build_chromatogram_iframe_html(height_px=605)
    elif mode == '🔬 MS1 & Halogens':
        ms_controls_box.layout.display = 'flex'
        plot_host.value = _build_ms1_iframe_html(list(ms_selector.value), height_px=505)
    else:
        ms_controls_box.layout.display = 'none'
        if _cached_dash_html[0]:
            plot_host.value = _wrap_in_iframe(_cached_dash_html[0], height_px=605, bg_color="#ffffff", scroll_y="auto")
        else:
            plot_host.value = _wrap_in_iframe(
                "<div style='color:#94a3b8;padding:24px;font-size:13.5px;'>Ask the copilot on the right to plot an <b>XIC</b> or <b>isotope overlay</b> (via <code>update_lcms_plot</code>), and the multi-panel dashboard will appear here automatically.</div>",
                height_px=605
            )

def refresh_live_plots():
    banner_html_widget.value = _build_banner_html()
    ms_selector.options = [(_wb_peak_label(int(p)), int(p)) for p in sorted(detected_df['peak_id'].astype(int).tolist())]
    render_active_left_view()

view_toggle.observe(lambda ch: render_active_left_view(), names='value')
ms_selector.observe(lambda ch: render_active_left_view() if view_toggle.value == '🔬 MS1 & Halogens' else None, names='value')

left_pane = widgets.VBox([banner_html_widget, view_toggle, ms_controls_box, plot_host], layout=widgets.Layout(width='63%', min_width='520px'))
left_pane.add_class('lcms-wb-panel')

# --- 7. Right Pane (Gemini Chat Copilot + Quick Actions + Vision QC) ---
model_options = ['gemini-3.8-flash', 'gemini-3.6-flash', 'gemini-3.5-flash', 'gemini-2.5-pro', 'gemini-2.5-flash']
if 'curated_stable_models' in globals():
    for m_id in curated_stable_models:
        if m_id not in model_options:
            model_options.append(m_id)

wb_model_dd = widgets.Dropdown(options=model_options, value=model_options[0], layout=widgets.Layout(width='170px'))
wb_vision_btn = widgets.Button(description='Vision QC', icon='eye', button_style='info', layout=widgets.Layout(width='105px'))
wb_reset_btn = widgets.Button(description='Reset', icon='refresh', button_style='warning', layout=widgets.Layout(width='80px'))

quick_prompts = [
    ("Conversion", "Summarize reaction conversion, LCAP% yield, unreacted SM, and main byproducts."),
    ("Peak Purity", "Check peak purity and co-elution across the major peaks, especially the Target Product."),
    ("Purification (Rs)", "Evaluate chromatographic resolution (Rs), tailing factors (Tf), and prep-HPLC/flash purification strategy."),
    ("Verify Halogens", "Verify which peak(s) show 79Br/81Br or 35Cl/37Cl isotope envelopes quantitatively."),
    ("Check Unassigned", "Identify any unassigned or low-confidence peaks using mass-delta diagnostics and formula elucidation, then update the chromatogram if confident."),
]
quick_btns = []
for lbl, prompt_str in quick_prompts:
    qb = widgets.Button(description=lbl, layout=widgets.Layout(width='auto', height='28px', padding='0 8px'))
    qb._prompt_str = prompt_str
    quick_btns.append(qb)

wb_chat_out = widgets.Output(layout=widgets.Layout(width='100%', height='475px', overflow_y='auto'))
wb_chat_out.add_class('lcms-wb-chat-scroll')

wb_chat_input = widgets.Textarea(
    placeholder='Discuss your chromatogram (e.g. "Is Peak 6 pure? Show 2D structures of the byproducts and check if TPPO is present anywhere in the run.")...',
    layout=widgets.Layout(width='100%', height='72px')
)
wb_send_btn = widgets.Button(description='Ask Copilot', button_style='primary', icon='paper-plane', layout=widgets.Layout(width='125px', height='34px'))
wb_refresh_btn = widgets.Button(description='Sync Plot', icon='sync', layout=widgets.Layout(width='100px', height='34px'))
wb_layout_btn = widgets.Button(description='Stacked', icon='arrows-v', layout=widgets.Layout(width='95px', height='34px'))

wb_chat_session = None
wb_active_model = None
wb_history = []

def _init_wb_chat(force_reset=False):
    global wb_chat_session, wb_active_model
    if force_reset or wb_chat_session is None or wb_active_model != wb_model_dd.value:
        wb_active_model = wb_model_dd.value
        wb_chat_session = _wb_client.chats.create(
            model=wb_active_model,
            config=types.GenerateContentConfig(
                system_instruction=_wb_system_instruction(),
                tools=wb_tools,
                temperature=0.2,
                automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=False, maximum_remote_calls=25)
            )
        )
        if force_reset:
            wb_history.clear()
            _render_wb_history("🔄 Chat reset and synced with live chromatogram.")

def _render_markdown_with_inline_svgs(md_text: str):
    cleaned = re.sub(
        r'\[\[PLOTLY_CHROM_\d+\]\]',
        '\n> 📈 *Live chromatogram on the left has been updated with the new peak assignment(s).*\n',
        md_text
    )
    parts = re.split(r'(\[\[MOL_IMG_\d+\]\]|\{\{SVG_\d+\}\})', cleaned)
    for part in parts:
        p_s = part.strip()
        if p_s in _rendered_svg_registry:
            display(HTML(_rendered_svg_registry[p_s]))
        elif p_s:
            display(Markdown(part))

def _render_wb_history(status_banner: str = ""):
    with wb_chat_out:
        clear_output(wait=True)
        if not wb_history:
            display(HTML(
                "<div style='color:#94a3b8; font-size:13px; padding:8px;'>"
                "<b style='color:#67e8f9;'>⚗️ LC-MS Copilot Ready</b><br>"
                f"Connected to <b>{len(wb_tools)}</b> JSON-safe RDKit & MS1 tools. "
                "Hover peaks on the left, click a quick prompt above, or ask a question below."
                "</div>"
            ))
        for role, content in wb_history:
            if role == "user":
                display(HTML(
                    "<div style='background:rgba(56,189,248,0.14); border:1px solid rgba(56,189,248,0.35); "
                    f"border-radius:8px; padding:8px 10px; margin:6px 0; color:#f8fafc;'><b>👤 You:</b> {html.escape(content)}</div>"
                ))
            else:
                display(HTML("<div style='font-size:11.5px; font-weight:700; color:#a78bfa; margin-top:8px;'>🤖 Gemini Copilot</div>"))
                _render_markdown_with_inline_svgs(content)
                display(HTML("<hr style='border:0; border-top:1px solid rgba(148,163,184,0.2); margin:10px 0;'>"))
        if status_banner:
            display(HTML(f"<div style='color:#67e8f9; font-size:12px; padding:6px 0;'>{status_banner}</div>"))

def _snapshot_assignments():
    if 'peak_assignments' not in globals():
        return ""
    return json.dumps({str(k): (v.get('display_name'), v.get('role'), v.get('classification')) for k, v in peak_assignments.items()}, sort_keys=True)

def _send_prompt_to_copilot(user_text: str):
    user_text = user_text.strip()
    if not user_text:
        return
    wb_chat_input.disabled = True
    wb_send_btn.disabled = True
    try:
        _init_wb_chat()
        before_snap = _snapshot_assignments()
        wb_history.append(("user", user_text))
        _render_wb_history(f"⏳ {wb_model_dd.value} is inspecting MS1 scans & RDKit tools...")

        response = wb_chat_session.send_message(user_text)
        chunks = [p.text for p in response.candidates[0].content.parts if getattr(p, 'text', None)] if response.candidates else []
        reply_text = "".join(chunks) if chunks else "*(No text returned.)*"
        wb_history.append(("model", reply_text))

        after_snap = _snapshot_assignments()
        if after_snap != before_snap or "[[PLOTLY_CHROM_" in reply_text:
            refresh_live_plots()

        pending_dash = SPECTRAL_STATE.pop("pending_plot_html", None)
        if pending_dash:
            _cached_dash_html[0] = pending_dash
            view_toggle.value = '📊 XIC / Dashboard'
            render_active_left_view()

        _render_wb_history()
    except Exception as e:
        wb_history.append(("model", f"❌ **Error:** `{e}`"))
        _render_wb_history()
    finally:
        wb_chat_input.value = ""
        wb_chat_input.disabled = False
        wb_send_btn.disabled = False

def _on_vision_qc_clicked(b):
    wb_vision_btn.disabled = True
    try:
        png_bytes = globals().get('_notebook_figure_registry', {}).get('deconvoluted_chromatogram_with_structures')
        if not png_bytes:
            wb_history.append(("model", "⚠️ No rasterized chromatogram cached (`kaleido` may be unavailable). Try running `analyze_chromatographic_resolution_and_shape` via chat instead."))
            _render_wb_history()
            return
        wb_history.append(("user", "👁️ [Vision QC] Visually inspect the current chromatogram for conversion, peak shape/tailing, baseline noise, and purification feasibility."))
        _render_wb_history(f"⏳ {wb_model_dd.value} is visually auditing the rendered chromatogram...")
        cols = [c for c in ['peak_id', 'retention_time', 'normalised area', 'Role', 'Assignment', 'Confidence (%)'] if c in detected_df.columns]
        v_prompt = (
            "Visually inspect this crude reaction LC-MS chromatogram alongside the peak table below. "
            "Report briefly on: (1) visual conversion & dominant product, (2) baseline quality, "
            "(3) peak shape/tailing/fronting, and (4) co-elution & prep-HPLC/flash purification feasibility.\n\n"
            + detected_df[cols].to_markdown(index=False)
        )
        resp = _wb_client.models.generate_content(
            model=wb_model_dd.value,
            contents=[types.Content(role='user', parts=[
                types.Part.from_text(text=v_prompt),
                types.Part.from_bytes(data=png_bytes, mime_type='image/png')
            ])]
        )
        wb_history.append(("model", getattr(resp, 'text', None) or "*(No visual QC text returned.)*"))
        _render_wb_history()
    except Exception as e:
        wb_history.append(("model", f"❌ **Vision QC Error:** `{e}`"))
        _render_wb_history()
    finally:
        wb_vision_btn.disabled = False

wb_send_btn.on_click(lambda b: _send_prompt_to_copilot(wb_chat_input.value))
wb_reset_btn.on_click(lambda b: _init_wb_chat(force_reset=True))
wb_refresh_btn.on_click(lambda b: refresh_live_plots())
wb_vision_btn.on_click(_on_vision_qc_clicked)
for qb in quick_btns:
    qb.on_click(lambda b: _send_prompt_to_copilot(b._prompt_str))

right_pane = widgets.VBox([
    widgets.HTML("<div style='font-weight:700; font-size:14px; background:linear-gradient(90deg,#22d3ee,#a78bfa); -webkit-background-clip:text; -webkit-text-fill-color:transparent; margin-bottom:4px;'>🤖 Gemini LC-MS Analytical Copilot</div>"),
    widgets.HBox([wb_model_dd, wb_vision_btn, wb_reset_btn], layout=widgets.Layout(gap='6px', flex_flow='row wrap')),
    widgets.HBox(quick_btns, layout=widgets.Layout(gap='5px', flex_flow='row wrap', margin='4px 0')),
    wb_chat_out,
    wb_chat_input,
    widgets.HBox([wb_send_btn, wb_refresh_btn, wb_layout_btn], layout=widgets.Layout(gap='8px', margin='4px 0 0 0'))
], layout=widgets.Layout(width='36%', min_width='340px'))
right_pane.add_class('lcms-wb-panel')

# --- 8. Split / Stacked Toggle & Master Assembly ---
workspace_box = widgets.Box(
    [left_pane, right_pane],
    layout=widgets.Layout(display='flex', flex_flow='row nowrap', width='100%', gap='12px', align_items='stretch')
)

_is_split = [True]
def _on_toggle_layout(b):
    _is_split[0] = not _is_split[0]
    if _is_split[0]:
        workspace_box.layout.flex_flow = 'row nowrap'
        left_pane.layout.width = '63%'
        right_pane.layout.width = '36%'
        wb_layout_btn.description = 'Stacked'
    else:
        workspace_box.layout.flex_flow = 'column nowrap'
        left_pane.layout.width = '100%'
        right_pane.layout.width = '100%'
        wb_layout_btn.description = 'Split View'
    render_active_left_view()

wb_layout_btn.on_click(_on_toggle_layout)

header_bar = widgets.HTML(
    "<div style='display:flex; justify-content:space-between; align-items:center; margin-bottom:10px;'>"
    "<div style='font-size:16px; font-weight:700; color:#f8fafc;'>"
    "⚗️ Interactive Crude Reaction LC-MS Workbench <span style='font-size:12px; color:#67e8f9; font-weight:500;'>(Live Plotly + Gemini Copilot)</span></div>"
    f"<div style='font-size:11.5px; color:#94a3b8;'>Dataset: <code>{html.escape(str(globals().get('mzml_filename', 'active mzML')))}</code> &nbsp;|&nbsp; Peaks: <b>{len(detected_df)}</b></div>"
    "</div>"
)

root_ui = widgets.VBox([wb_css, header_bar, workspace_box], layout=widgets.Layout(width='100%'))
root_ui.add_class('lcms-wb-root')
display(root_ui)

# Smart preselect for MS1 comparator & initial render
smart_pre = []
for role_t in ("Reactant", "Product", "Byproduct"):
    for pid in sorted(detected_df['peak_id'].astype(int).tolist()):
        if 'peak_assignments' in globals() and pid in peak_assignments and _wb_extract_role(peak_assignments[pid]) == role_t and pid not in smart_pre:
            smart_pre.append(pid)
            break
for _, pid in ms_selector.options:
    if len(smart_pre) >= 3:
        break
    if pid not in smart_pre:
        smart_pre.append(pid)
ms_selector.value = tuple(smart_pre[:3])

refresh_live_plots()
_render_wb_history()

---
### 9. Multimodal Vision QC — Let Gemini *See* the Crude Reaction Chromatogram

Until now, Gemini has reasoned over numerical peak tables (`detected_df`) and mass spectra. The cell below feeds the **actual rendered chromatogram images** — the raw MS1 TIC trace from Cell 2 and the deconvoluted crude reaction chromatogram with embedded RDKit 2D structure cards and role badges (`SM`, `Prod`, `Byp`, `Rgt`) from Cell 5 — directly into Gemini's multimodal vision context. This allows the model to visually evaluate:
* **Visual conversion & main product dominance** relative to unreacted starting materials and side products.
* **Peak tailing or fronting** (e.g., from polar carboxylic acid hydrolysis impurities or basic amines overloading the C18 stationary phase).
* **Co-elution & prep-HPLC / flash purification feasibility** (baseline separation between the Target Product and closely eluting impurities).
* **2D chemical structure card verification** (checking that each drawn structure matches the expected synthetic transformation and reverse-phase C18 elution order).


In [ ]:
#@title 10. Multimodal Vision QC — Feed Crude Reaction Chromatogram & 2D Structure Cards to Gemini
import io
from google import genai
from google.genai import types
from google.colab import userdata
from IPython.display import display, Markdown


def _figure_to_png_bytes(fig_or_bytes, dpi: int = 160) -> bytes:
    """Rasterizes a chromatogram image to PNG bytes. Accepts either a matplotlib `Figure`
    (from Cell 2) or raw PNG bytes exported from Cell 5 / Cell 8C's interactive Plotly
    crude reaction chromatogram via `qc_fig.to_image()`/kaleido."""
    if isinstance(fig_or_bytes, (bytes, bytearray)):
        return bytes(fig_or_bytes)
    buf = io.BytesIO()
    fig_or_bytes.savefig(buf, format='png', dpi=dpi, bbox_inches='tight')
    buf.seek(0)
    return buf.getvalue()


def collect_vision_qc_images() -> dict:
    """Gathers the raw HPLC trace and annotated deconvoluted crude reaction chromatogram
    cached in `_notebook_figure_registry` by Section 2 Cells 2, 5, and 8C."""
    registry = globals().get('_notebook_figure_registry', {})
    images = {}
    for name, fig in registry.items():
        try:
            images[name] = _figure_to_png_bytes(fig)
        except Exception as e:
            print(f"⚠️ Could not rasterize '{name}': {e}")
    return images


_vision_client = globals().get('client') or genai.Client(api_key=userdata.get("KDD_GEMINI_API_KEY"))
_vision_model_name = globals()['model_dropdown'].value if 'model_dropdown' in globals() else 'gemini-3.8-flash'

CRUDE_REACTION_VISION_QC_PROMPT = """You are an expert medicinal and process chemist visually inspecting these Crude Reaction LC-MS chromatogram images alongside their embedded 2D chemical structure cards. Examine every image provided and report:

1. **Visual Reaction Profile & Conversion** — Identify the dominant Target Product peak (`Prod`), any remaining Starting Material peaks (`SM`), and Byproduct/Impurity peaks (`Byp`). Does the visual peak height/area profile indicate a clean, high-yielding transformation?
2. **Baseline Quality & Signal-to-Noise** — Check for baseline drift, chemical noise, solvent front spikes, or background humps (e.g., catalyst/ligand bleed such as TPPO or PEG).
3. **Peak Shape, Tailing & Fronting** — Flag any asymmetric peaks (e.g., tailing caused by free carboxylic acid hydrolysis byproducts or basic amines interacting with silanols, or column overloading on the main product peak).
4. **Co-elution & Purification Feasibility (Prep-HPLC / Flash)** — Assess the chromatographic resolution between the Target Product peak and its nearest eluting impurities. Will the Target Product be easy to isolate cleanly by C18 prep-HPLC or flash chromatography, or are there shoulder/co-eluting peaks?
5. **2D Structure Card & Elution Order Audit** — Inspect the 2D molecular structures drawn in the top banner above the chromatogram:
   - Confirm whether the drawn structures chemically match their assigned labels (e.g., intact aryl halide in `SM`, coupled biaryl core in `Prod`, cleaved functional groups in `Byp`).
   - Verify whether their visual elution order makes sense on a Reverse-Phase C18 column (more polar phenols/carboxylic acids/boronic acids eluting earlier; less polar esters, aryl bromides, and symmetrical homocoupling dimers eluting later).

Present your findings as a clear, actionable markdown report for a synthetic chemist."""


def run_multimodal_chromatogram_vision_qc(user_question: str = None) -> str:
    """Sends the cached crude reaction chromatogram/structure images (plus live peak table context)
    into Gemini's multimodal vision context and displays its synthetic QC report."""
    images = collect_vision_qc_images()
    if not images:
        msg = "⚠️ No chromatogram figures found in `_notebook_figure_registry`. Run Section 2 Cells 2 and 5 first."
        display(Markdown(msg))
        return msg

    # Include compact tabular context so Gemini can cross-check the visual badges (P1, P2...) against exact RTs/masses
    context_str = ""
    if 'detected_df' in globals():
        cols = [c for c in ['peak_id', 'retention_time', 'normalised area', 'Assignment', 'Confidence (%)'] if c in detected_df.columns]
        context_str = "\n\n### Reference Numerical Peak Table (`detected_df`):\n" + detected_df[cols].to_markdown(index=False)

    prompt_text = (user_question or CRUDE_REACTION_VISION_QC_PROMPT) + context_str
    parts = [types.Part.from_text(text=prompt_text)]
    for name, png_bytes in images.items():
        parts.append(types.Part.from_text(text=f"--- Image: {name} ---"))
        parts.append(types.Part.from_bytes(data=png_bytes, mime_type='image/png'))

    response = _vision_client.models.generate_content(
        model=_vision_model_name,
        contents=[types.Content(role='user', parts=parts)],
    )

    report = getattr(response, 'text', None) or "*(No text returned.)*"
    display(Markdown(f"### ⚗️ Gemini Crude Reaction Multimodal Vision QC Report ({len(images)} image(s) analyzed)\n\n{report}"))
    return report


# Run once automatically with the default Crude Reaction QC prompt.
# Re-run any time with a custom question, e.g.:
# run_multimodal_chromatogram_vision_qc("Is the Target Biaryl Product (P6) well-resolved from the unreacted aryl bromide SM (P5) and homocoupling dimer (P7) for prep-HPLC?")
_ = run_multimodal_chromatogram_vision_qc()